<a href="https://colab.research.google.com/github/moja7/Dissertation/blob/main/LNG_Dissertation_Analysis_RXZK4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/moja7/Dissertation/blob/main/Diss_Analy_18_7_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Fully detach and clean the mountpoint
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception as e:
    print("unmount note:", e)

# Remove any leftover mount directory that's blocking the remount
import shutil, os
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)
print("mountpoint cleared:", not os.path.exists("/content/drive"))

Drive not mounted, so nothing to flush and unmount.
mountpoint cleared: True


In [4]:
from google.colab import drive
drive.mount("/content/drive")   # choose the CORRECT account in the popup

Mounted at /content/drive


In [5]:
import os
print(os.listdir("/content/drive/"))          # expect ['MyDrive'] or ['MyDrive','Shareddrives']
hits = []
for root, _, files in os.walk("/content/drive/MyDrive/Dissertation Analysis"):
    hits += [os.path.join(root, f) for f in files]
print(f"{len(hits)} files found under project folder")
for h in hits[:10]: print(" ", h)

['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']
246 files found under project folder
  /content/drive/MyDrive/Dissertation Analysis/Data/Nat Gas Pricing - EU full.xls
  /content/drive/MyDrive/Dissertation Analysis/Data/CEGH Hist Data.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c1-btu-dollar-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c1-btu-dollar-weekly-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/SNL HH Day Ahead Daily.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c3-btu-dollar-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c4-btu-dollar-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c5-btu-dollar-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c6-btu-dollar-1990.xlsx
  /content/drive/MyDrive/Dissertation Analysis/Data/HH-c7-btu-dollar-1990.xlsx


## Setup — environment, paths, data load (A1–A3)
Mounts Drive, installs/loads libraries, sets `BASE`/`OUT_DIR`, loads `master_panel.parquet`, and builds the 2016+ modelling frame `p` with causal NaN handling. Everything downstream reads `p` and writes to `OUT_DIR`.

In [ ]:
# %% Cell A1: Install + imports + paths
!pip install -q arch ruptures shap xgboost statsmodels scikit-learn \
    torch pytorch-forecasting lightning

import os, glob, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid"); plt.rcParams["figure.figsize"] = (14, 5)

# --- Confirmed directory layout ---
BASE     = "/content/drive/MyDrive/Dissertation Analysis"
DATA_DIR = f"{BASE}/data"
OUT_DIR  = f"{BASE}/output"
os.makedirs(OUT_DIR, exist_ok=True)

# Verify the folders and the parquet are actually visible this session
print("BASE exists :", os.path.exists(BASE))
print("DATA exists :", os.path.exists(DATA_DIR))
print("OUT  exists :", os.path.exists(OUT_DIR))
print("\nFiles in output/:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", f)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 444.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 20.9 MB/s eta 0:00:00


In [ ]:
# %% Cell A2: Load the master panel
panel_path = f"{OUT_DIR}/master_panel.parquet"

if not os.path.exists(panel_path):
    # last-resort search anywhere under the project
    hits = glob.glob(f"{BASE}/**/master_panel.parquet", recursive=True)
    if hits:
        panel_path = hits[0]
    else:
        raise FileNotFoundError(f"Not found under {BASE}. Re-save from the pipeline notebook.")

print("Loading:", panel_path)
panel = pd.read_parquet(panel_path).sort_index()
print("panel:", panel.shape, "|", panel.index.min().date(), "->", panel.index.max().date())

# Event windows (must match the pipeline)
EVENTS = {"ukraine": ("2022-02-24", "2023-10-31"),
          "redsea":  ("2023-11-19", "2025-05-06"),
          "iran":    ("2026-02-28", None)}

HUBS = [h for h in ["ttf", "jkm", "hh", "nbp", "the"] if f"{h}_c1_usd" in panel.columns]
print("hubs available:", HUBS)

In [ ]:
# %% Cell A3: Modelling window + causal NaN handling (fixes the purge/leakage bug)
# Strategy: start 2016 (LNG era, covers all events); forward-fill FUNDAMENTALS
# only (they persist between releases); never fill prices; drop warm-up rows.
MODEL_START = "2016-01-01"
p = panel.loc[MODEL_START:].copy()

fundamentals = [c for c in ["storage_pct", "storage_anom", "storage_chg",
                            "gpr", "gpr_act", "gpr_threat",
                            "brent_c1_lag1", "wti_c1_lag1", "eua_c1_lag1",
                            "chk_hormuz_ma7", "chk_suez_ma7", "chk_bab_ma7",
                            "chk_cape_ma7"] if c in p.columns]
p[fundamentals] = p[fundamentals].ffill(limit=5)     # short causal fill only

core_prices = [f"{h}_c1_usd" for h in HUBS]
before = len(p)
p = p.dropna(subset=core_prices)                     # a missing core price = real gap
print(f"Rows: {before} -> {len(p)} after requiring core prices present")
print("Modelling span:", p.index.min().date(), "->", p.index.max().date())



In [ ]:
# %% Cell: EIA release-date calendar (gas storage + oil inventory) via EIA API v2
# Purpose: build leakage-safe "scheduled release day" features.
# Release dates are KNOWN IN ADVANCE, so using them to predict that day is NOT leakage.
import requests, pandas as pd, numpy as np

EIA_API_KEY = "YYmJfLrfim5AEye3FU5OUqisAennhSU1YXn3cqE0"   # <-- your key
BASE = "https://api.eia.gov/v2"

def eia_weekly(route, params, label):
    """Pull a weekly EIA series; return DataFrame of period dates + values."""
    url = f"{BASE}/{route}/data/"
    q = {
        "api_key": EIA_API_KEY,
        "frequency": "weekly",
        "data[0]": "value",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": 0,
        "length": 5000,
    }
    q.update(params)
    r = requests.get(url, params=q, timeout=30)
    r.raise_for_status()
    rows = r.json()["response"]["data"]
    df = pd.DataFrame(rows)
    df["period"] = pd.to_datetime(df["period"])
    df = df[["period", "value"]].rename(columns={"value": label}).sort_values("period")
    print(f"{label}: {len(df)} weekly obs, {df.period.min().date()} -> {df.period.max().date()}")
    return df

# ---- 1. NATURAL GAS STORAGE (Weekly NG Storage Report) ----
# Route: natural-gas/stor/wkly ; total lower-48 working gas
gas = eia_weekly(
    "natural-gas/stor/wkly",
    {"facets[series][]": "NW2_EPG0_SWO_R48_BCF"},   # Lower-48 working gas, Bcf
    "gas_storage_bcf",
)
# Gas report: period is week-ending Friday; PUBLISHED the following Thursday (+6 days)
gas["release_date"] = gas["period"] + pd.Timedelta(days=6)

# ---- 2. CRUDE OIL INVENTORY (Weekly Petroleum Status Report) ----
# Route: petroleum/stoc/wstk ; weekly ending stocks of crude oil (excl. SPR)
oil = eia_weekly(
    "petroleum/stoc/wstk",
    {"facets[series][]": "WCESTUS1"},   # US commercial crude stocks, thousand barrels
    "crude_stocks_kbbl",
)
# Oil report: period is week-ending Friday; PUBLISHED the following Wednesday (+5 days)
oil["release_date"] = oil["period"] + pd.Timedelta(days=5)

# ---- 3. Save the raw release-date series (this is what you couldn't download elsewhere) ----
gas[["period", "release_date", "gas_storage_bcf"]].to_csv(f"{OUT_DIR}/eia_gas_releases.csv", index=False)
oil[["period", "release_date", "crude_stocks_kbbl"]].to_csv(f"{OUT_DIR}/eia_oil_releases.csv", index=False)
print("\nSaved release-date series to eia_gas_releases.csv and eia_oil_releases.csv")

# ---- 4. Build leakage-safe daily features aligned to your panel ----
gas_rel = set(gas["release_date"])
oil_rel = set(oil["release_date"])

# use your panel's index (p must exist from Cell A3)
cal = pd.DataFrame(index=p.index.copy())
cal["eia_gas_release"]  = cal.index.isin(gas_rel).astype(int)
cal["eia_oil_release"]  = cal.index.isin(oil_rel).astype(int)
cal["eia_any_release"]  = ((cal["eia_gas_release"] + cal["eia_oil_release"]) > 0).astype(int)

# days-to-next-release (known in advance -> leakage-safe). Countdown to next gas report.
def days_to_next(index, release_set):
    rel = np.array(sorted(release_set), dtype="datetime64[D]")
    idx = np.array(index.values, dtype="datetime64[D]")
    out = np.full(len(idx), np.nan)
    for i, d in enumerate(idx):
        future = rel[rel >= d]
        if len(future):
            out[i] = (future[0] - d) / np.timedelta64(1, "D")   # -> float number of days
    return out

cal["days_to_gas_release"] = days_to_next(cal.index, gas_rel)

print("\nRelease-day counts in your modelling window:")
print("  gas report days:", int(cal["eia_gas_release"].sum()))
print("  oil report days:", int(cal["eia_oil_release"].sum()))
print("  any release days:", int(cal["eia_any_release"].sum()))
print(cal[cal["eia_gas_release"] == 1].head())

# merge into panel
p = p.join(cal)
print("\nMerged. New columns:", list(cal.columns))

In [ ]:
# %% VERIFY POINT 2 — dropped days + trading-day lag indexing

import pandas as pd
import numpy as np

# ---------- 1. How many days were dropped? ----------
# The raw span vs the rows actually kept in the modelling frame.
# 'p' is your cleaned/modelling dataframe (rows = trading days kept).

idx = p.index
start, end = idx.min(), idx.max()

# (a) Calendar days in the span
calendar_days = (end - start).days + 1

# (b) Business days (Mon–Fri) in the span — the natural "expected" trading days
#     before holidays/drops. This is the fair denominator for a daily price series.
business_days = pd.bdate_range(start, end)
n_business = len(business_days)

# (c) Rows actually in your modelling frame
n_kept = len(idx)

# (d) Business days NOT present in your frame = holidays + dropped bad-print days
missing_from_business = business_days.difference(idx)
n_missing = len(missing_from_business)

print("=== Coverage ===")
print(f"Span: {start.date()} to {end.date()}")
print(f"Calendar days in span:            {calendar_days}")
print(f"Business days (Mon-Fri) in span:  {n_business}")
print(f"Rows kept in modelling frame:     {n_kept}")
print(f"Business days absent (holidays+drops): {n_missing}  "
      f"({100*n_missing/n_business:.2f}% of business days)")

# If you have the RAW (pre-drop) frame saved, compare directly for the true drop count.
# Replace 'raw' below with your raw dataframe variable if you have one:
try:
    n_raw = len(raw)            # <-- your raw, pre-cleaning frame if it exists
    dropped = n_raw - n_kept
    print(f"\nRaw rows: {n_raw} | Kept: {n_kept} | Dropped: {dropped} "
          f"({100*dropped/n_raw:.2f}% of raw)")
except NameError:
    print("\n(No 'raw' frame in memory — use the business-day gap above as the estimate,")
    print(" or point this at your pre-drop dataframe to get the exact drop count.)")

# ---------- 2. Are the gaps isolated or consecutive? ----------
# Consecutive drops distort lags more than isolated ones. Check gap lengths.
if len(idx) > 1:
    gaps = idx.to_series().diff().dt.days.dropna()
    # On a trading-day series, normal gap = 1 (next day) or 3 (Fri->Mon).
    big_gaps = gaps[gaps > 4]   # >4 calendar days = more than a normal weekend
    print("\n=== Gap structure ===")
    print(f"Median gap between rows: {gaps.median():.0f} day(s)")
    print(f"Rows preceded by a gap >4 calendar days: {len(big_gaps)}")
    if len(big_gaps):
        print("Largest gaps (calendar days):")
        print(big_gaps.sort_values(ascending=False).head(8).to_string())

# ---------- 3. Confirm lags are ROW-based (trading-day), not calendar-based ----------
# This proves 'shift(1)' = previous TRADING day, not previous calendar day.
demo = p[[f"{HUBS[0]}_ret"]].copy() if f"{HUBS[0]}_ret" in p.columns else p.iloc[:, [0]].copy()
demo.columns = ["value"]
demo["lag1_rowbased"] = demo["value"].shift(1)         # what your pipeline does
demo["prev_index"] = demo.index.to_series().shift(1)   # the date that lag1 actually came from
demo["gap_to_lag_days"] = (demo.index.to_series() - demo["prev_index"]).dt.days

print("\n=== Lag indexing check (first rows around any gap) ===")
print("If 'gap_to_lag_days' is mostly 1 (and 3 across weekends), your lags are")
print("trading-day lags by construction. Values >1 show where a lag spans a gap.\n")
# show a few rows where the lag spans more than one calendar day
spanning = demo[demo["gap_to_lag_days"] > 1].head(10)
print(spanning[["value","lag1_rowbased","gap_to_lag_days"]].to_string())

print(f"\nShare of lags that span exactly 1 calendar day: "
      f"{100*(demo['gap_to_lag_days']==1).mean():.1f}%")
print(f"Share spanning >1 day (weekends/holidays/drops): "
      f"{100*(demo['gap_to_lag_days']>1).mean():.1f}%")

## EIA calendar — release-day volatility (Task 1b)
Tests whether absolute returns are systematically higher on scheduled EIA storage/inventory release days. This is the highest-value use of the calendar: it isolates a *scheduled* volatility driver so the geopolitical signal can be read cleanly (an SQ1 control). Extends the interactive TTF test to all hubs. Saves `eia_release_volatility.csv`.

In [ ]:
# %% Task 1b: Do EIA release days carry higher volatility? (all hubs)
import statsmodels.api as sm
from scipy import stats
import numpy as np, pandas as pd

if "eia_gas_release" not in p.columns:
    print("eia_gas_release not in p — run the EIA calendar cell first; skipping.")
else:
    gas_day = p["eia_gas_release"].fillna(0)
    rows = []
    for h in HUBS:                                   # all available hubs
        col = f"{h}_c1_usd"
        if col not in p.columns: continue
        absret = np.log(p[col]).diff().abs()
        d = pd.DataFrame({"absret": absret, "release": gas_day}).dropna()
        rel = d.loc[d.release == 1, "absret"]; non = d.loc[d.release == 0, "absret"]
        t, pt = stats.ttest_ind(rel, non, equal_var=False)          # Welch
        reg = sm.OLS(d["absret"], sm.add_constant(d["release"])).fit()
        rows.append({"hub": h.upper(),
                     "mean|ret| release": round(rel.mean(), 5),
                     "mean|ret| non": round(non.mean(), 5),
                     "ratio": round(rel.mean()/non.mean(), 2),
                     "welch_t": round(t, 3), "welch_p": round(pt, 4),
                     "reg_coef": round(reg.params["release"], 5),
                     "reg_p": round(reg.pvalues["release"], 4)})
    eia_vol = pd.DataFrame(rows)
    try: display(eia_vol)
    except NameError: print(eia_vol.to_string(index=False))
    eia_vol.to_csv(f"{OUT_DIR}/eia_release_volatility.csv", index=False)
    print("Saved eia_release_volatility.csv")
    print("Framing: gas volatility is higher on EIA storage-report days for hubs where ratio>1 & p<0.05; "
          "controlling for these scheduled releases isolates the geopolitical component (SQ1 control).")


## EDA & stationarity diagnostics (A4)
Descriptives, ADF/KPSS unit-root tests and correlation structure. Confirms prices are I(1) and returns I(0) — the prerequisite for the cointegration/ECM backbone below.

In [ ]:
# %% Cell A4: SECTION 3 - EDA & diagnostics (each test -> a decision)
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# (1) STL on a representative hub -> justifies seasonal features
decomp = seasonal_decompose(p["ttf_c1_usd"].dropna(), model="additive", period=252)
decomp.plot(); plt.suptitle("STL decomposition - TTF (motivates seasonal features)", y=1.01)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda_stl.png", dpi=150); plt.show()

# (2) ACF/PACF on returns -> justifies lag features
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(p["ttf_ret"].dropna(), lags=30, ax=ax[0]); ax[0].set_title("ACF TTF returns")
plot_pacf(p["ttf_ret"].dropna(), lags=30, ax=ax[1]); ax[1].set_title("PACF TTF returns")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda_acf.png", dpi=150); plt.show()

# (3) Full-panel stationarity + kurtosis table -> levels I(1), returns I(0)
rows = []
for h in HUBS:
    lvl = p[f"{h}_c1_usd"].dropna(); ret = p[f"{h}_ret"].dropna()
    rows.append({"Hub": h.upper(),
                 "ADF level p": round(adfuller(lvl)[1], 3),
                 "ADF return p": round(adfuller(ret)[1], 3),
                 "KPSS return p": round(kpss(ret, regression="c", nlags="auto")[1], 3),
                 "Excess kurtosis": round(ret.kurt(), 2)})
diag = pd.DataFrame(rows); print("\n--- Stationarity & distribution ---"); print(diag.to_string(index=False))
diag.to_csv(f"{OUT_DIR}/diag_stationarity.csv", index=False)

# (4) VIF on the TTF curve -> justifies compressing to spread/slope/curvature
curve = [c for c in ["ttf_c1_usd", "ttf_c2_usd", "ttf_c3_usd", "ttf_c4_usd"] if c in p.columns]
sub = p[curve].dropna()
vif = pd.DataFrame({"feature": curve,
                    "VIF": [variance_inflation_factor(sub.values, i) for i in range(len(curve))]})
print("\n--- VIF (TTF curve; high VIF motivates curve compression) ---"); print(vif.to_string(index=False))



# SQ1 Mechanism — Arbitrage Structure (Task A)
**Cointegration &rarr; Error-Correction &rarr; Regime speeds &rarr; Arbitrage threshold**, on the LSEG `panel`.

 Regime windows use the panel's own `evt_*` dummies (dissertation windows). Uses `p` (2016+ modelling frame) and `OUT_DIR` defined above; all variables are namespaced `arb_*` so nothing here overwrites the forecasting/attribution cells.

Model: `JKM = θ0 + θ1·TTF + θ2·Brent + μ`; ECM `ΔJKM = β1ΔTTF + β2ΔBrent + λ·μ̂_{t-1}`; threshold `λ(|S|)=λ0+δ|S|`, `τ=−λ0/δ`.

In [ ]:
# %% Task A.1 - Cointegration (Engle-Granger + Johansen)
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
from statsmodels.api import OLS, add_constant
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# UCL figure theme
ARB_PURPLE, ARB_LILAC, ARB_GREY = "#500778", "#8E7CC3", "#6b6b6b"
ARB_UKR, ARB_RED, ARB_IRN = "#c0392b", "#e67e22", "#2c7fb8"
def arb_style(ax, title, subtitle=None):
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    ax.grid(True, color="#e6e6e6", lw=0.8); ax.set_axisbelow(True)
    ax.text(0, 1.10, title, transform=ax.transAxes, fontsize=13, fontweight="bold", color="#1a1a1a")
    if subtitle: ax.text(0, 1.03, subtitle, transform=ax.transAxes, fontsize=10, color=ARB_GREY)

# Build arbitrage frame from the modelling frame p (falls back to panel)
_base = p if "p" in dir() else panel.loc["2016-01-01":]
arb = pd.DataFrame({
    "JKM": _base["jkm_c1_usd"], "TTF": _base["ttf_c1_usd"], "Brent": _base["brent_c1"],
    "GPR": _base["gpr"],
    "D_Ukraine": _base["evt_ukraine"], "D_RedSea": _base["evt_redsea"], "D_Iran": _base["evt_iran"],
})
arb_price = arb[["JKM","TTF","Brent"]].dropna()
print(f"Arbitrage frame: {arb_price.shape[0]} complete price rows, "
      f"{arb_price.index.min().date()} -> {arb_price.index.max().date()}")

# Engle-Granger cointegrating regression
arb_coint = OLS(arb_price["JKM"], add_constant(arb_price[["TTF","Brent"]])).fit()
arb_th0, arb_th1, arb_th2 = arb_coint.params["const"], arb_coint.params["TTF"], arb_coint.params["Brent"]
print(f"JKM = {arb_th0:.3f} + {arb_th1:.4f}*TTF + {arb_th2:.4f}*Brent   R2 = {arb_coint.rsquared:.4f}")
arb["ecm_resid"] = arb_coint.resid
arb_eg = adfuller(arb_coint.resid, autolag="AIC")
print(f"Engle-Granger ADF on residual: {arb_eg[0]:.4f} (p={arb_eg[1]:.4f}) -> "
      + ("COINTEGRATED (< -3.37 EG CV)" if arb_eg[0] < -3.37 else "check vs EG CV"))

# Johansen robustness
arb_jo = coint_johansen(arb_price, det_order=0, k_ar_diff=3)
arb_rank = 0
print("\nJohansen trace test:")
for i in range(3):
    tr, cv5, cv1 = arb_jo.lr1[i], arb_jo.cvt[i,1], arb_jo.cvt[i,2]
    rej = "reject***" if tr>cv1 else ("reject**" if tr>cv5 else "no")
    print(f"  r<={i}:  trace={tr:8.2f}   5%CV={cv5:6.2f}   1%CV={cv1:6.2f}   {rej}")
    if tr>cv5 and arb_rank==i: arb_rank = i+1
print(f"Johansen rank = {arb_rank}  (>=1 confirms Engle-Granger)")

# Residual figure
fig, ax = plt.subplots(figsize=(11,3.6))
ax.plot(arb["ecm_resid"].index, arb["ecm_resid"].values, color=ARB_PURPLE, lw=0.8)
ax.axhline(0, color="black", lw=0.6, ls="--")
for dum,c,lab in [("D_Ukraine",ARB_UKR,"Ukraine"),("D_RedSea",ARB_RED,"Red Sea"),("D_Iran",ARB_IRN,"Iran")]:
    span = arb[arb[dum]==1].index
    if len(span): ax.axvspan(span.min(), span.max(), alpha=0.10, color=c, label=lab)
ax.legend(frameon=False, ncol=3, loc="upper left", fontsize=9)
arb_style(ax, "Cointegrating residual  (JKM - theta.[TTF, Brent])",
          "Mean-reverting around zero; crisis windows shaded. LSEG panel.")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/taskA_coint_residuals.png", dpi=300, bbox_inches="tight"); plt.show()

#Cointegration between Hubs

In [ ]:
# %% Task A.1c - Pairwise ECM adjustment speeds (economic strength of arbitrage)
# Replaces the unreliable cointegration t-stat heatmap. Lambda = speed the price gap
# closes per day; faster (more negative) lambda = tighter arbitrage. Half-life in days.
from statsmodels.api import OLS, add_constant
from itertools import combinations
import numpy as np, pandas as pd, matplotlib.pyplot as plt

_base = p if "p" in dir() else panel.loc["2016-01-01":]
HUB_COLS = {"TTF":"ttf_c1_usd","THE":"the_c1_usd","NBP":"nbp_c1_usd",
            "JKM":"jkm_c1_usd","HH":"hh_c1_usd"}
hub_px = pd.DataFrame({k:_base[v] for k,v in HUB_COLS.items() if v in _base.columns}).dropna()
hubs = list(hub_px.columns)

def pairwise_ecm(y, x):
    """Cointegrate y~x, then ECM: dy = a + lam*resid_{t-1} + b*dx. Return lam, half-life, p, R2coint."""
    coint = OLS(y, add_constant(x)).fit()
    resid = coint.resid
    reg = pd.DataFrame({"dy":y.diff(), "resid_lag":resid.shift(1), "dx":x.diff()}).dropna()
    m = OLS(reg["dy"], add_constant(reg[["resid_lag","dx"]])).fit()
    lam = m.params["resid_lag"]; pval = m.pvalues["resid_lag"]
    hl = np.log(2)/(-np.log(1+lam)) if -1 < lam < 0 else np.nan
    return lam, hl, pval, coint.rsquared

# Estimate for every pair. Cointegration is directional, so estimate BOTH ways and
# report the stronger (more negative lambda) - the direction where arbitrage binds.
rows = []
for a, b in combinations(hubs, 2):
    lam_ab, hl_ab, p_ab, _ = pairwise_ecm(hub_px[a], hub_px[b])
    lam_ba, hl_ba, p_ba, _ = pairwise_ecm(hub_px[b], hub_px[a])
    # keep the stronger direction
    if (lam_ab if not np.isnan(lam_ab) else 0) <= (lam_ba if not np.isnan(lam_ba) else 0):
        lam, hl, pv = lam_ab, hl_ab, p_ab
    else:
        lam, hl, pv = lam_ba, hl_ba, p_ba
    rows.append({"pair":f"{a}-{b}", "hub_a":a, "hub_b":b,
                 "lambda":lam, "pct_per_day":-lam*100, "half_life_days":hl, "p":pv})

ecm = pd.DataFrame(rows).sort_values("lambda").reset_index(drop=True)
ecm.to_csv(f"{OUT_DIR}/taskA_pairwise_ecm.csv", index=False)
print("Pairwise ECM adjustment speeds (faster = tighter arbitrage):\n")
print(ecm[["pair","pct_per_day","half_life_days","p"]].round(3).to_string(index=False))

# ---- Per-hub average speed: which hub is the slowest to re-integrate? ----
hub_speed = {}
for h in hubs:
    sp = ecm[(ecm.hub_a==h)|(ecm.hub_b==h)]["pct_per_day"]
    hub_speed[h] = sp.mean()
print("\nMean adjustment speed by hub (lower = more isolated):")
for h,v in sorted(hub_speed.items(), key=lambda x:x[1]):
    print(f"  {h}: {v:.2f}%/day")

# %% Task A.1c-viz - Pairwise arbitrage half-life (lollipop, clearer than heatmap)
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ecm = pd.read_csv(f"{OUT_DIR}/taskA_pairwise_ecm.csv")   # from the ECM cell
df = ecm[["pair","half_life_days"]].dropna().sort_values("half_life_days").reset_index(drop=True)

PURPLE="#500778"; HHCOL="#c0392b"
fig, ax = plt.subplots(figsize=(9, 5.5))
for yi, r in df.iterrows():
    c = HHCOL if "HH" in r["pair"] else PURPLE
    ax.plot([0, r["half_life_days"]], [yi, yi], color=c, lw=1.5, alpha=0.4, zorder=1)
    ax.scatter(r["half_life_days"], yi, s=130, color=c, zorder=3, edgecolor="white", linewidth=1.2)
    lab = f"{r['half_life_days']:.1f}d" if r["half_life_days"] < 10 else f"{r['half_life_days']:.0f}d"
    ax.text(r["half_life_days"] + 0.8, yi, lab, va="center", fontsize=10, fontweight="bold", color="#1a1a1a")

ax.set_yticks(range(len(df))); ax.set_yticklabels(df["pair"], fontsize=11)
ax.set_xlabel("Arbitrage half-life (days) — shorter = tighter integration")
ax.set_xlim(0, df["half_life_days"].max() * 1.15)
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", lw=0.8); ax.set_axisbelow(True); ax.tick_params(left=False)
ax.text(0, len(df)+0.3, "Henry Hub pairs (red) adjust ~10x slower — short-run isolated",
        fontsize=10, color=HHCOL, fontweight="bold")
ax.text(0, len(df)+1.0, "Speed of arbitrage between gas hubs",
        fontsize=14, fontweight="bold", color="#1a1a1a")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/taskA_ecm_lollipop.png", dpi=300, bbox_inches="tight")
plt.show()


# %% Per-hub mean arbitrage speed (summary table)
import pandas as pd, numpy as np

ecm = pd.read_csv(f"{OUT_DIR}/taskA_pairwise_ecm.csv")   # from the pairwise ECM cell

hubs = ["TTF", "THE", "NBP", "JKM", "HH"]
rows = []
for h in hubs:
    m = ecm[(ecm.hub_a == h) | (ecm.hub_b == h)]
    rows.append({
        "Hub": h,
        "Mean %/day": m["pct_per_day"].mean(),
        "Mean half-life (d)": m["half_life_days"].mean(),
        "Fastest pair": m.loc[m["half_life_days"].idxmin(), "pair"],
        "Slowest pair": m.loc[m["half_life_days"].idxmax(), "pair"],
        "N pairs": len(m),
    })
hub_summary = pd.DataFrame(rows).sort_values("Mean %/day").reset_index(drop=True)

print("Per-hub arbitrage speed (lower %/day = more isolated):\n")
print(hub_summary.round(2).to_string(index=False))
hub_summary.to_csv(f"{OUT_DIR}/taskA_hub_speed_summary.csv", index=False)

In [ ]:
# %% Task A.1d - Directional influence: which hub leads, which follows?
# ECM weak-exogeneity: for each pair, estimate how much EACH hub adjusts to the
# disequilibrium. The hub that adjusts LESS (small lambda) is the LEADER (price-setter);
# the hub that adjusts MORE is the FOLLOWER. Cointegration is symmetric; this breaks the tie.
from statsmodels.api import OLS, add_constant
from itertools import combinations
import numpy as np, pandas as pd

_base = p if "p" in dir() else panel.loc["2016-01-01":]
HUB_COLS = {"TTF":"ttf_c1_usd","THE":"the_c1_usd","NBP":"nbp_c1_usd",
            "JKM":"jkm_c1_usd","HH":"hh_c1_usd"}
hub_px = pd.DataFrame({k:_base[v] for k,v in HUB_COLS.items() if v in _base.columns}).dropna()
hubs = list(hub_px.columns)

def adjustment_lambda(y, x):
    """How much does Y adjust to restore equilibrium with X? Returns (lambda, p-value)."""
    resid = OLS(y, add_constant(x)).fit().resid
    reg = pd.DataFrame({"dy":y.diff(), "resid_lag":resid.shift(1), "dx":x.diff()}).dropna()
    m = OLS(reg["dy"], add_constant(reg[["resid_lag","dx"]])).fit()
    return m.params["resid_lag"], m.pvalues["resid_lag"]

rows = []
for a, b in combinations(hubs, 2):
    lam_a, p_a = adjustment_lambda(hub_px[a], hub_px[b])   # how much A adjusts
    lam_b, p_b = adjustment_lambda(hub_px[b], hub_px[a])   # how much B adjusts
    # The hub that adjusts MORE (larger |lambda|) is the follower; the other leads.
    if abs(lam_a) > abs(lam_b):
        leader, follower = b, a
    else:
        leader, follower = a, b
    rows.append({"pair":f"{a}-{b}", "leader":leader, "follower":follower,
                 f"|lambda|_{a}":abs(lam_a), f"p_{a}":p_a,
                 f"|lambda|_{b}":abs(lam_b), f"p_{b}":p_b})

direction = pd.DataFrame(rows)
print("Directional influence (leader = price-setter, doesn't adjust; follower = adjusts):\n")
for _, r in direction.iterrows():
    print(f"  {r['pair']:10s} ->  {r['leader']} LEADS, {r['follower']} follows")

# Score: how many pairs each hub LEADS (net price-setter ranking)
lead_count = pd.Series([r["leader"] for _, r in direction.iterrows()]).value_counts()
follow_count = pd.Series([r["follower"] for _, r in direction.iterrows()]).value_counts()
score = pd.DataFrame({"leads": lead_count, "follows": follow_count}).fillna(0).astype(int)
score["net_leadership"] = score["leads"] - score["follows"]
score = score.sort_values("net_leadership", ascending=False)
print("\nPrice-setter ranking (net leadership = pairs led minus pairs followed):")
print(score.to_string())
direction.to_csv(f"{OUT_DIR}/taskA_hub_leadership.csv", index=False)
score.to_csv(f"{OUT_DIR}/taskA_hub_leadership_score.csv")

In [ ]:
# %% Task A.2 - Baseline error-correction model (Newey-West HAC)
arb["dJKM"]=arb["JKM"].diff(); arb["dTTF"]=arb["TTF"].diff(); arb["dBrent"]=arb["Brent"].diff()
arb["GPR_c"]=arb["GPR"]-arb["GPR"].mean()
for L in [1,2,3]:
    arb[f"dJKM_L{L}"]=arb["dJKM"].shift(L); arb[f"dTTF_L{L}"]=arb["dTTF"].shift(L)
arb["ecm_lag"]=arb["ecm_resid"].shift(1)

arb_b = arb[["dJKM","dTTF","dBrent","ecm_lag"]].dropna()
arb_base = OLS(arb_b["dJKM"], add_constant(arb_b[["dTTF","dBrent","ecm_lag"]])).fit(
    cov_type="HAC", cov_kwds={"maxlags":7})
arb_lam_base = arb_base.params["ecm_lag"]
arb_half = lambda x: np.log(0.5)/np.log(1+x) if x<0 else np.nan
print(f"lambda = {arb_lam_base:.5f}  (p = {arb_base.pvalues['ecm_lag']:.4g})")
print(f"{abs(arb_lam_base)*100:.2f}% of the JKM-TTF/Brent gap corrects per day; "
      f"half-life = {arb_half(arb_lam_base):.1f} days")

In [ ]:
# %% Task A.3 - Regime-interacted ECM (panel evt_* dummies)
arb["ecm_x_GPR_c"]=arb["ecm_lag"]*arb["GPR_c"]
for dum in ["Ukraine","RedSea","Iran"]:
    arb[f"ecm_x_{dum}"]=arb["ecm_lag"]*arb[f"D_{dum}"]
arb_cols=["dTTF","dBrent","ecm_lag","ecm_x_GPR_c","ecm_x_Ukraine","ecm_x_RedSea","ecm_x_Iran",
          "D_Ukraine","D_RedSea","D_Iran","GPR_c","dJKM_L1","dJKM_L2","dJKM_L3","dTTF_L1","dTTF_L2","dTTF_L3"]
arb_e = arb[["dJKM"]+arb_cols].dropna()
arb_reg = OLS(arb_e["dJKM"], add_constant(arb_e[arb_cols])).fit(cov_type="HAC", cov_kwds={"maxlags":7})
arb_lam = arb_reg.params["ecm_lag"]

arb_rows=[]
for name,var,shock in [("Normal",None,"No shock"),("Ukraine","ecm_x_Ukraine","Supply destruction"),
                       ("Red Sea","ecm_x_RedSea","Transit disruption"),("Iran","ecm_x_Iran","Chokepoint closure")]:
    eff=arb_lam+(arb_reg.params[var] if var else 0)
    pv =arb_reg.pvalues[var] if var else arb_reg.pvalues["ecm_lag"]
    arb_dcol = var.replace("ecm_x_","D_") if var else None
    ndays=int(arb_e[arb_dcol].sum()) if arb_dcol else int((arb_e[["D_Ukraine","D_RedSea","D_Iran"]].sum(axis=1)==0).sum())
    arb_rows.append({"Regime":name,"Shock type":shock,"N days":ndays,"Eff. lambda":round(eff,4),
                     "%/day":round(abs(eff)*100,1),"Half-life (d)":round(arb_half(eff),1),
                     "psi p":round(pv,3),"Sig":"***" if pv<.01 else("**" if pv<.05 else("*" if pv<.10 else "n.s."))})
arb_reg_tbl=pd.DataFrame(arb_rows)
try: display(arb_reg_tbl)
except NameError: print(arb_reg_tbl.to_string(index=False))

fig,ax=plt.subplots(figsize=(8,4.4))
arb_bar_c=[ARB_GREY,ARB_UKR,ARB_RED,ARB_IRN]
bars=ax.bar(arb_reg_tbl["Regime"], arb_reg_tbl["%/day"], color=arb_bar_c, width=0.62)
for bar,r in zip(bars,arb_rows):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.25,
            f"{r['%/day']:.1f}%\n{r['Sig']}\n(n={r['N days']})", ha="center", va="bottom", fontsize=8.5)
ax.set_ylabel("Correction speed (% of gap/day)"); ax.set_ylim(0, max(arb_reg_tbl["%/day"])*1.3)
arb_style(ax, "Arbitrage correction speed by geopolitical regime",
          "Ukraine (supply destruction) corrects fastest & is significant; Red Sea (transit) is not.")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/taskA_regime_speeds.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# %% Task A.4 - Arbitrage threshold (continuous + binning)
arb_t = arb[["dJKM","dTTF","dBrent","ecm_lag"]].copy()
arb_t["abs_spread"]=(arb["JKM"]-arb["TTF"]).abs().shift(1)
arb_t["ecm_x_spread"]=arb_t["ecm_lag"]*arb_t["abs_spread"]
arb_t=arb_t.dropna()
arb_thr = OLS(arb_t["dJKM"], add_constant(arb_t[["dTTF","dBrent","ecm_lag","ecm_x_spread"]])).fit(
    cov_type="HAC", cov_kwds={"maxlags":7})
arb_lam0, arb_delta = arb_thr.params["ecm_lag"], arb_thr.params["ecm_x_spread"]
arb_tau = -arb_lam0/arb_delta
print(f"lambda0 = {arb_lam0:.5f} (p={arb_thr.pvalues['ecm_lag']:.4g})")
print(f"delta   = {arb_delta:.5f} (p={arb_thr.pvalues['ecm_x_spread']:.4g})")
print(f"Arbitrage threshold tau = -lambda0/delta = ${arb_tau:.2f}/MMBtu   (coursework provisional: $3.28)")

# Binning corroboration
arb_bins=[0,1,3,5,10,20,100]; arb_labels=["<1","1-3","3-5","5-10","10-20",">20"]
arb_t["bin"]=pd.cut(arb_t["abs_spread"],bins=arb_bins,labels=arb_labels)
arb_brow=[]
for lab in arb_labels:
    sub=arb_t[arb_t["bin"]==lab]
    if len(sub)>30:
        mm=OLS(sub["dJKM"],add_constant(sub[["dTTF","dBrent","ecm_lag"]])).fit()
        arb_brow.append({"|Spread| bin":lab,"n":len(sub),"lambda":round(mm.params["ecm_lag"],4),
                         "p":round(mm.pvalues["ecm_lag"],3),"Sig":mm.pvalues["ecm_lag"]<0.05})
arb_bin_tbl=pd.DataFrame(arb_brow)
try: display(arb_bin_tbl)
except NameError: print(arb_bin_tbl.to_string(index=False))

fig,axes=plt.subplots(1,2,figsize=(13,4))
axb=axes[0]
axb.bar(arb_bin_tbl["|Spread| bin"], arb_bin_tbl["lambda"],
        color=[ARB_PURPLE if s else "#c9c2dc" for s in arb_bin_tbl["Sig"]], width=0.65)
axb.axhline(0,color="black",lw=0.6)
axb.set_ylabel("lambda (per-bin correction)"); axb.set_xlabel("|JKM - TTF| spread ($/MMBtu)")
arb_style(axb,"Correction strengthens with spread size",
          f"Insignificant near zero; switches on ~1-3. Continuous tau = {arb_tau:.2f}.")
axs=axes[1]
arb_spread=(arb["JKM"]-arb["TTF"])
axs.plot(arb_spread.index, arb_spread.values, color=ARB_GREY, lw=0.6)
axs.axhspan(-arb_tau,arb_tau,color=ARB_LILAC,alpha=0.30,label=f"+/- {arb_tau:.2f} band")
axs.axhline(0,color="black",lw=0.5,ls="--")
for dum,c in [("D_Ukraine",ARB_UKR),("D_RedSea",ARB_RED),("D_Iran",ARB_IRN)]:
    span=arb[arb[dum]==1].index
    if len(span): axs.axvspan(span.min(),span.max(),alpha=0.08,color=c)
axs.legend(frameon=False,fontsize=9,loc="lower left")
arb_style(axs,"JKM - TTF spread vs arbitrage band",
          "Ukraine breaches the band (fast correction); Red Sea sits inside it (dormant).")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/taskA_threshold.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# %% FIGURE — Arbitrage threshold: JKM-TTF spread vs the ±tau band, regimes labeled
import matplotlib.pyplot as plt, matplotlib.dates as mdates
import pandas as pd, numpy as np

# (assumes arb_tau computed above; and arb has D_Ukraine/D_RedSea/D_Iran dummies)
arb_spread = (arb["JKM"] - arb["TTF"])

fig, ax = plt.subplots(figsize=(12, 5.0))

# --- shaded regime windows with HORIZONTAL labels ---
REGIMES = [("D_Ukraine", ARB_UKR, "Ukraine"),
           ("D_RedSea",  ARB_RED, "Red Sea"),
           ("D_Iran",    ARB_IRN, "Hormuz")]
ymax = arb_spread.max(); ymin = arb_spread.min()
yrange = ymax - ymin
for dum, c, lab in REGIMES:
    span = arb[arb[dum] == 1].index
    if len(span):
        s, e = span.min(), span.max()
        ax.axvspan(s, e, alpha=0.10, color=c, zorder=0)
        mid = s + (e - s) / 2
        # horizontal label near the top of the plot area (fixed offset, not scaled by value)
        ax.text(mid, ymax - 0.04 * yrange, lab, ha="center", va="top", rotation=0,
                fontsize=12, fontweight="bold", color=c, zorder=5)

# --- the arbitrage no-trade band ---
ax.axhspan(-arb_tau, arb_tau, color=ARB_LILAC, alpha=0.30, zorder=1,
           label=f"no-arbitrage band  (±${arb_tau:.2f}/MMBtu)")
ax.axhline(0, color="black", lw=0.8, ls="--", zorder=2)

# bold black labels marking the band edges directly on the plot
ax.axhline( arb_tau, color="black", lw=0.8, zorder=2)
ax.axhline(-arb_tau, color="black", lw=0.8, zorder=2)
xr = arb_spread.index.max()   # right edge for label placement
ax.text(xr, arb_tau,  f" +${arb_tau:.2f}", va="center", ha="left",
        fontsize=10, fontweight="bold", color="black", zorder=6)
ax.text(xr, -arb_tau, f" −${arb_tau:.2f}", va="center", ha="left",
        fontsize=10, fontweight="bold", color="black", zorder=6)

# --- the spread line ---
ax.plot(arb_spread.index, arb_spread.values, color=ARB_PURPLE, lw=0.9, zorder=3)

# annotate the Ukraine extreme by LARGEST ABSOLUTE spread (the true peak is TTF>>JKM, i.e. most negative)
pk_d = arb_spread.abs().idxmax()          # date of largest-magnitude deviation
pk_v = arb_spread.loc[pk_d]               # signed value there (~ -27)
# place the text on the same side as the extreme, offset inward from the edge
y_text = pk_v + (0.06 * yrange if pk_v < 0 else -0.06 * yrange)
ax.annotate(f"peak ${abs(pk_v):.0f}/MMBtu (TTF premium)", xy=(pk_d, pk_v),
            xytext=(pk_d + pd.Timedelta(days=180), y_text),
            fontsize=9.5, fontweight="bold", color=ARB_PURPLE,
            arrowprops=dict(arrowstyle="-|>", color=ARB_PURPLE, lw=1.2,
                            connectionstyle="arc3,rad=-0.15"))

ax.set_ylabel("JKM − TTF spread (USD/MMBtu)")
ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(frameon=False, fontsize=9.5, loc="lower left")

ax.set_title("JKM − TTF spread against the arbitrage band", fontsize=13,
             fontweight="bold", color=ARB_PURPLE, pad=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/taskA_threshold.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/taskA_threshold.pdf", bbox_inches="tight")
plt.show()
print("Saved taskA_threshold.png")

In [ ]:
# %% Task A.5 - Save arbitrage outputs to OUT_DIR (Drive)
pd.DataFrame([{
    "theta0":round(arb_th0,4),"theta1_TTF":round(arb_th1,4),"theta2_Brent":round(arb_th2,4),
    "R2":round(arb_coint.rsquared,4),"EG_ADF":round(arb_eg[0],4),"EG_p":round(arb_eg[1],4),
    "Johansen_trace_r0":round(arb_jo.lr1[0],2),"Johansen_rank":arb_rank,
    "baseline_lambda":round(arb_lam_base,5),"baseline_halflife_d":round(arb_half(arb_lam_base),1)}]
).to_csv(f"{OUT_DIR}/taskA_cointegration.csv", index=False)
arb_reg_tbl.to_csv(f"{OUT_DIR}/taskA_ecm_regime.csv", index=False)
pd.DataFrame([{"lambda0":round(arb_lam0,5),"delta":round(arb_delta,5),
               "threshold_usd":round(arb_tau,2),"provisional_3_28":3.28}]
).to_csv(f"{OUT_DIR}/taskA_threshold.csv", index=False)
arb_bin_tbl.to_csv(f"{OUT_DIR}/taskA_threshold_bins.csv", index=False)
print("Saved to OUT_DIR: taskA_cointegration.csv, taskA_ecm_regime.csv, taskA_threshold.csv, taskA_threshold_bins.csv")
print("Figures: taskA_coint_residuals.png, taskA_regime_speeds.png, taskA_threshold.png")
print(f"\nHEADLINE (LSEG): cointegration confirmed; "
      f"Ukraine {arb_reg_tbl.loc[1,'%/day']}%/day ({arb_reg_tbl.loc[1,'Sig']}), "
      f"Red Sea {arb_reg_tbl.loc[2,'%/day']}%/day ({arb_reg_tbl.loc[2,'Sig']}); "
      f"arbitrage threshold tau = ${arb_tau:.2f}/MMBtu.")

## Structural breaks (A5)
Chow (known crisis dates) tests on the JKM–TTF spread. Produces `sq2_chow.csv` and the breaks figure.

In [ ]:
# %% Cell A5b: Chow structural breaks across ALL hub pairs (global vs link-specific?)
# Tests whether geopolitical breaks appear in every hub relationship or only certain links.
import statsmodels.api as sm
from scipy.stats import f as fdist
from itertools import combinations
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def chow_test(y, X, split_idx):
    X = sm.add_constant(X)
    ssr = sm.OLS(y, X).fit().ssr
    s1 = sm.OLS(y.iloc[:split_idx], X.iloc[:split_idx]).fit().ssr
    s2 = sm.OLS(y.iloc[split_idx:], X.iloc[split_idx:]).fit().ssr
    k, nn = X.shape[1], len(y)
    F = ((ssr - (s1 + s2)) / k) / ((s1 + s2) / (nn - 2*k))
    return F, 1 - fdist.cdf(F, k, nn - 2*k)

HUB_COLS = {"TTF":"ttf_c1_usd","THE":"the_c1_usd","NBP":"nbp_c1_usd",
            "JKM":"jkm_c1_usd","HH":"hh_c1_usd"}
hub_px = pd.DataFrame({k:p[v] for k,v in HUB_COLS.items() if v in p.columns}).dropna()
hub_list = list(hub_px.columns)
pairs = list(combinations(hub_list, 2))
events = list(EVENTS.keys())

# build F-stat and significance matrices
Fmat = pd.DataFrame(index=[f"{a}-{b}" for a,b in pairs], columns=events, dtype=float)
Pmat = pd.DataFrame(index=[f"{a}-{b}" for a,b in pairs], columns=events, dtype=float)
for a, b in pairs:
    y, X = hub_px[a], hub_px[[b]]
    for ev, (s_, _) in EVENTS.items():
        t = pd.Timestamp(s_)
        if t <= y.index.min() or t >= y.index.max():
            continue
        split = y.index.searchsorted(t)
        F, pv = chow_test(y, X, split)
        Fmat.loc[f"{a}-{b}", ev] = F
        Pmat.loc[f"{a}-{b}", ev] = pv

print("Chow F-statistics across all hub pairs (critical ~4.6 at 1%):\n")
print(Fmat.round(1).to_string())
Fmat.to_csv(f"{OUT_DIR}/sq2_chow_allpairs_F.csv")
Pmat.to_csv(f"{OUT_DIR}/sq2_chow_allpairs_p.csv")

# how many pairs break significantly per event?
print("\nPairs with significant break (p<0.05) per event:")
for ev in events:
    n_sig = (Pmat[ev] < 0.05).sum()
    print(f"  {ev}: {n_sig}/{len(pairs)} pairs break")

# ---- Heatmap: pairs (rows) x events (cols), coloured by log F ----
ev_label = {"ukraine":"Ukraine\n2022","redsea":"Red Sea\n2023","iran":"Hormuz\n2026"}
fig, ax = plt.subplots(figsize=(7, 8))
plot_vals = np.log10(Fmat.values.astype(float))   # log because F ranges hugely
im = ax.imshow(plot_vals, cmap="Purples", aspect="auto")
ax.set_xticks(range(len(events))); ax.set_xticklabels([ev_label.get(e,e) for e in events], fontsize=11)
ax.set_yticks(range(len(pairs)));  ax.set_yticklabels(Fmat.index, fontsize=10)
CRIT = 4.61
for i in range(len(pairs)):
    for j in range(len(events)):
        f = Fmat.iloc[i,j]
        if pd.isna(f): continue
        star = "***" if Pmat.iloc[i,j]<0.01 else ("**" if Pmat.iloc[i,j]<0.05 else "")
        txt = f"{f:.0f}{star}" if f>=CRIT else f"{f:.0f}\nn.s."
        colr = "white" if np.log10(f) > np.nanmax(plot_vals)*0.6 else "#1a1a1a"
        ax.text(j, i, txt, ha="center", va="center", fontsize=8.5, color=colr, fontweight="bold")
ax.spines[:].set_visible(False); ax.tick_params(length=0)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("log10(Chow F)  |  darker = sharper break", fontsize=9)
ax.text(0, -0.9, "Are the breaks global or link-specific?", fontsize=14, fontweight="bold", color="#1a1a1a")
ax.text(0, -0.4, "Chow F per hub pair per event. *** p<0.01, ** p<0.05. Critical F \u2248 4.6.",
        fontsize=9.5, color="#6b6b6b")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_chow_allpairs_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## Volatility — GARCH(1,1)/GARCH-MIDAS (A6)
Volatility persistence per hub and the GPR→realised-volatility link. Produces `sq2_garch.csv`.

In [ ]:
# %% Cell A6: SECTION 5.4 - Volatility: GARCH(1,1) + GARCH-MIDAS(GPR)
from arch import arch_model
from arch.univariate import ConstantMean, GARCH, StudentsT

gar_rows = []
for h in HUBS:
    r = (p[f"{h}_ret"].dropna() * 100)                     # scale for stability
    res = arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
    a, b = res.params["alpha[1]"], res.params["beta[1]"]
    gar_rows.append({"Hub": h.upper(), "alpha": round(a, 3), "beta": round(b, 3),
                     "persistence a+b": round(a + b, 3)})
garch_df = pd.DataFrame(gar_rows); print("--- GARCH(1,1) persistence ---")
print(garch_df.to_string(index=False)); garch_df.to_csv(f"{OUT_DIR}/sq2_garch.csv", index=False)

# GARCH-MIDAS: does monthly GPR drive the long-run vol component?
# Practical proxy using arch's exogenous long-run via ARX not built-in; we use the
# 'arch' GARCH-MIDAS through the 'mgarch'-style approximation: regress realized vol
# on lagged GPR (MIDAS-lite) to evidence GPR -> volatility, then report.
if "gpr" in p.columns:
    rv = p["ttf_ret"].rolling(21).std().dropna() * np.sqrt(252)
    gpr_m = p["gpr"].reindex(rv.index).ffill()
    Xg = sm.add_constant(gpr_m.shift(1).fillna(method="bfill"))
    midas = sm.OLS(rv, Xg).fit()
    print("\nGPR -> realized-vol (MIDAS-lite):",
          f"coef={midas.params['gpr']:.4f}, p={midas.pvalues['gpr']:.4f}",
          "(positive & significant supports GARCH-MIDAS story)")
    # NOTE: for the full GARCH-MIDAS estimator use the 'mfGARCH' R package or the
    # 'arch' dev build; the MIDAS-lite regression is a defensible in-notebook proxy.



## Cross-market connectedness (A7)
Rolling correlations and Diebold-Yilmaz connectedness — identifies TTF as price-setter and Henry Hub as near-isolated. Produces `sq2_connectedness.csv`.

In [ ]:
# %% Cell A7: SECTION 5.5 - Cross-market: rolling correlation + connectedness
# Rolling 60-day correlations show the TTF-JKM coupling / HH decoupling.
roll = pd.DataFrame(index=p.index)
roll["corr_jkm_ttf"] = p["jkm_ret"].rolling(60).corr(p["ttf_ret"])
roll["corr_ttf_hh"]  = p["ttf_ret"].rolling(60).corr(p["hh_ret"])
fig, ax = plt.subplots(figsize=(15, 4))
roll.plot(ax=ax, lw=1)
for nm,(s_,e_) in EVENTS.items():
    e_ = e_ or str(p.index.max().date())
    ax.axvspan(pd.Timestamp(s_), pd.Timestamp(e_), alpha=0.10)
ax.set_title("Rolling 60d return correlations (coupling tightens in crises)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/sq2_rolling_corr.png", dpi=150); plt.show()

# Diebold-Yilmaz connectedness via VAR + FEVD (robust to axis ordering)
from statsmodels.tsa.api import VAR
ret_cols = [f"{h}_ret" for h in HUBS]
vd = p[ret_cols].dropna()
vres = VAR(vd).fit(2)

H = 10
fevd_obj = vres.fevd(H)
decomp = np.asarray(fevd_obj.decomp)          # find the (n,n) matrix at horizon H
n = len(ret_cols)

# decomp can be (H, n, n) or (n, H, n) depending on version; select the n x n slice
if decomp.shape == (H, n, n):
    mat = decomp[-1]
elif decomp.shape == (n, H, n):
    mat = decomp[:, -1, :]
else:
    # fallback: reshape by taking the last horizon along whichever axis == H
    ax = list(decomp.shape).index(H)
    mat = np.take(decomp, -1, axis=ax)
mat = np.asarray(mat).reshape(n, n)

conn = pd.DataFrame(mat, index=[c.upper() for c in ret_cols],
                    columns=[c.upper() for c in ret_cols])
total_conn = 100 * (conn.values.sum() - np.trace(conn.values)) / conn.values.sum()
print(f"Diebold-Yilmaz total connectedness index: {total_conn:.1f}%")
print(conn.round(3).to_string())
conn.round(3).to_csv(f"{OUT_DIR}/sq2_connectedness.csv")

#Research Question #2

## SQ2 forecasting — walk-forward engine (A8)
Leakage-safe design matrix and the expanding-window `TimeSeriesSplit(10, test_size=60)` engine. All forecasting models below reuse `Xall`/`yall`/`tscv` and the `walk_forward` helper.

In [ ]:
# %% Cell A8: SECTION 4 - SQ1 walk-forward engine (returns target)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# Feature matrix: leakage-safe engineered features (all already lagged/trailing)
feature_cols = [c for c in p.columns if any(c.endswith(s) or k in c for s, k in [
    ("_lag1",""),("_lag2",""),("_lag7",""),("_slope",""),("_prompt_spread",""),
    ("_curvature",""),("_vol21",""),("","""spread_"""),("","storage_"),
    ("","chk_"),("","gpr"),("","doy_"),("","heating")]) ]
feature_cols = sorted(set(c for c in feature_cols
                          if not c.endswith("_ret") and not c.endswith("_c1_usd")
                          and not c.startswith("evt_")))
targets = [f"{h}_ret" for h in HUBS]

data = p[feature_cols + targets].copy()
data[feature_cols] = data[feature_cols].ffill(limit=5)
data = data.dropna(subset=targets)                     # need targets; features imputed
Xall = data[feature_cols].fillna(0.0)                  # residual NaN in features -> 0 AFTER fill
yall = data[targets]
print("SQ1 design matrix:", Xall.shape, "features |", len(targets), "targets")

tscv = TimeSeriesSplit(n_splits=10, test_size=60)      # expanding window, 60d blocks

def walk_forward(predict_fn, X, y, target):
    errs, preds_all, actual_all = [], [], []
    for tr, te in tscv.split(X):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y[target].iloc[tr], y[target].iloc[te]
        yp = predict_fn(Xtr, ytr, Xte)
        errs.append(np.sqrt(np.mean((yte.values - yp)**2)))
        preds_all.append(pd.Series(yp, index=yte.index)); actual_all.append(yte)
    return np.array(errs), pd.concat(preds_all), pd.concat(actual_all)


# Trimming the features

In [ ]:
# %% Cell A8b (SAFE TO RE-RUN): curate from the ORIGINAL feature matrix
# Rebuild the full matrix from `data` (unchanged from A8) so re-running is safe.
Xall_full = data[feature_cols].fillna(0.0)      # the original 123-feature matrix

def curate_features(all_cols):
    keep = []
    keep += [c for c in all_cols if c.startswith("spread_")]
    keep += [c for c in all_cols if c.endswith("_slope") or c.endswith("_curvature")]
    keep += [c for c in all_cols if c.endswith("_lag1") or c.endswith("_vol21")]
    keep += [c for c in all_cols if c in ("storage_anom", "storage_chg")]
    keep += [c for c in all_cols if c.startswith("chk_") and c.endswith("_ma7")]
    keep += [c for c in all_cols if c in ("brent_c1_lag1", "wti_c1_lag1", "eua_c1_lag1")]
    keep += [c for c in all_cols if c in ("gpr", "gpr_act", "gpr_threat")]
    keep += [c for c in all_cols if c.startswith("evt_") and c != "evt_any"]   # 3 regime dummies, not evt_any
    keep += [c for c in all_cols if c in ("heating", "doy_sin", "doy_cos")]
    seen, out = set(), []
    for c in keep:
        if c in all_cols and c not in seen:
            out.append(c); seen.add(c)
    return out

curated = curate_features(list(Xall_full.columns))
Xall = Xall_full[curated].copy()

print(f"Feature reduction: {Xall_full.shape[1]} -> {Xall.shape[1]} features")
print(f"Obs per feature: {len(Xall)/Xall.shape[1]:.0f}")
print("evt_ in Xall:", [c for c in Xall.columns if c.startswith("evt_")])
print("Xall:", Xall.shape)

### Adding EIA calendar features to the modelling set (Task 1a)
The A8b selector matches column-name suffixes, so the `eia_*` columns are excluded unless added explicitly. Here they are appended onto the **curated** design matrix `Xall` (pulled from `p`, not from `feature_cols`, since they were never in it). The subsequent walk-forward cells then see them. Expected effect on point forecasts is small — reported honestly (a null is fine).

In [ ]:
# %% Task 1a: append EIA calendar features to the curated modelling matrix Xall
EIA_FEATURES = ["eia_gas_release", "eia_oil_release", "eia_any_release", "days_to_gas_release"]
_add = [c for c in EIA_FEATURES if c in p.columns and c not in Xall.columns]
if _add:
    _eia = p[_add].reindex(Xall.index).fillna(0.0)
    Xall = Xall.join(_eia)
    print(f"Added {len(_add)} EIA calendar features {_add} -> Xall now {Xall.shape}")
else:
    print("No new eia_* columns to add (either absent from p, or already present).")
# NOTE: run the walk-forward model cells (A9, B*) AFTER this so the models use the calendar features.


In [ ]:
# %% List the 46 model features grouped by family (for the methodology table)
import pandas as pd, re

feats = list(Xall.columns)   # your 46-feature matrix
print(f"Total features: {len(feats)}\n")

# classify each feature into a family by name pattern — adjust patterns to your naming
def family(f):
    f = f.lower()
    if 'spread' in f or ('_' in f and any(h in f for h in ['jkm','ttf','nbp','the','hh']) and 'minus' in f): return "Cross-hub spread"
    if 'slope' in f or 'curve' in f or 'm1' in f or 'm2' in f or 'm3' in f or 'c1c2' in f: return "Curve shape"
    if 'lag' in f or 'roll' in f or 'vol' in f or 'std' in f or 'ret' in f: return "Price lag / volatility"
    if 'stor' in f: return "Storage"
    if any(s in f for s in ['hormuz','suez','bab','cape','transit','chk','strait']): return "Chokepoint"
    if 'brent' in f or 'wti' in f or 'oil' in f or 'coal' in f or 'eua' in f or 'carbon' in f: return "Oil / carbon / coal"
    if 'gpr' in f: return "Geopolitical (GPR)"
    if any(s in f for s in ['ukraine','redsea','red_sea','iran','hormuz_d','regime','dummy','_d_']): return "Regime dummy"
    if any(s in f for s in ['sin','cos','month','week','weekend','holiday','season','doy','dow']): return "Seasonal / calendar"
    if 'eur' in f or 'gbp' in f or 'usd' in f or 'fx' in f: return "FX"
    if 'eia' in f: return "EIA calendar"
    return "OTHER (unclassified)"

rows = [(family(f), f) for f in feats]
df = pd.DataFrame(rows, columns=["Family","Feature"]).sort_values(["Family","Feature"])

# grouped view
for fam, g in df.groupby("Family"):
    print(f"\n{fam}  ({len(g)})")
    for f in g["Feature"]:
        print(f"   {f}")

# summary counts
print("\n--- family counts ---")
print(df.groupby("Family").size().to_string())

# flag anything unclassified so you can fix the patterns
unc = df[df["Family"].str.startswith("OTHER")]
if len(unc):
    print("\n⚠ UNCLASSIFIED — adjust patterns:")
    print(unc["Feature"].to_string(index=False))

In [ ]:
# %% Cell A9: SQ1 baselines (random walk, ARIMA) + models (RF, XGB, ensemble)
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

def rw_predict(Xtr, ytr, Xte):        # random walk on returns = predict 0
    return np.zeros(len(Xte))

def arima_predict(Xtr, ytr, Xte):
    try:
        fit = ARIMA(ytr, order=(1, 0, 1)).fit()
        return np.repeat(fit.forecast(1).iloc[0], len(Xte))
    except Exception:
        return np.zeros(len(Xte))

def xgb_predict(Xtr, ytr, Xte):
    m = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                     subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0)
    m.fit(Xtr, ytr); return m.predict(Xte)

def rf_predict(Xtr, ytr, Xte):
    m = RandomForestRegressor(n_estimators=300, max_depth=8, n_jobs=-1)
    m.fit(Xtr, ytr); return m.predict(Xte)

TARGET = "ttf_ret"                                     # headline hub; loop others as needed
results = {}
for name, fn in [("RandomWalk", rw_predict), ("ARIMA", arima_predict),
                 ("RandomForest", rf_predict), ("XGBoost", xgb_predict)]:
    errs, preds, actual = walk_forward(fn, Xall, yall, TARGET)
    results[name] = {"rmse": errs.mean(), "preds": preds, "actual": actual}
    print(f"{name:14s} mean fold RMSE (returns): {errs.mean():.5f}")

# Price-space reconstruction for reporting (Section 4.5 - both spaces)
lvl = p["ttf_c1_usd"].reindex(results["XGBoost"]["actual"].index)
print("\nReporting note: multiply return-RMSE by price level (~USD 12-40) for "
      "approximate price-space error; full reconstruction done in the results table.")


In [ ]:
# %% Cell A10: SQ1 significance - Diebold-Mariano (each model vs random walk)
from scipy.stats import norm
def dm_test(e_a, e_b):
    d = e_a**2 - e_b**2
    dbar = d.mean(); v = np.var(d, ddof=1) / len(d)
    stat = dbar / np.sqrt(v); pval = 2 * (1 - norm.cdf(abs(stat)))
    return stat, pval

base = results["RandomWalk"]
dm_rows = []
for name in ["ARIMA", "RandomForest", "XGBoost"]:
    idx = results[name]["actual"].index.intersection(base["actual"].index)
    e_model = (results[name]["actual"].loc[idx] - results[name]["preds"].loc[idx]).values
    e_rw    = (base["actual"].loc[idx] - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_model, e_rw)
    dm_rows.append({"Model": name, "DM stat": round(stat, 3), "p-value": round(pv, 4),
                    "Beats RW?": "Yes" if (pv < 0.05 and stat < 0) else "No/tie"})
dm_df = pd.DataFrame(dm_rows); print("--- Diebold-Mariano vs Random Walk ---")
print(dm_df.to_string(index=False)); dm_df.to_csv(f"{OUT_DIR}/sq1_dm.csv", index=False)


### Statistical tests — what they are (Task S1a)
For the examiner's "did you run a significance test?" question:
- **Model comparison** uses the **Diebold-Mariano test** (above) — the time-series-appropriate paired test of equal predictive accuracy on the forecast errors, correct for serially-dependent data (a plain paired t-test on daily errors would be invalid).
- **Dataset characterisation** uses descriptives + ADF/KPSS + excess kurtosis (EDA section).
- A **regime-error ANOVA** (Task S1b, below) adds a formal test that forecast errors differ across market regimes.

# SECTION 5.1-5.2 - SQ2

In [ ]:
# %% Cell A11: SECTION 5.1-5.2 - SQ2 attribution (ablation + SHAP)
import shap
# Fit one XGB on a fixed train/test split for attribution (SHAP needs a fitted model)
split = int(len(Xall) * 0.8)
Xtr, Xte = Xall.iloc[:split], Xall.iloc[split:]
ytr, yte = yall[TARGET].iloc[:split], yall[TARGET].iloc[split:]
m_full = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0)
m_full.fit(Xtr, ytr)
rmse_full = np.sqrt(np.mean((yte.values - m_full.predict(Xte))**2))

# (a) Ablation: drop the geopolitical block, re-fit, measure the rise in error
geo_block = [c for c in Xall.columns if "gpr" in c or "chk_" in c]
Xtr_abl, Xte_abl = Xtr.drop(columns=geo_block), Xte.drop(columns=geo_block)
m_abl = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                     subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0).fit(Xtr_abl, ytr)
rmse_abl = np.sqrt(np.mean((yte.values - m_abl.predict(Xte_abl))**2))
print(f"Ablation (drop geopolitical block): RMSE {rmse_full:.5f} -> {rmse_abl:.5f}  "
      f"delta {rmse_abl - rmse_full:+.5f}  ({100*(rmse_abl-rmse_full)/rmse_full:+.1f}%)")

# (b) SHAP global + calm-vs-crisis split
explainer = shap.TreeExplainer(m_full)
shap_vals = explainer.shap_values(Xte)
shap.summary_plot(shap_vals, Xte, show=False, max_display=15)
plt.title("SHAP global importance (test period)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_shap_global.png", dpi=150, bbox_inches="tight"); plt.show()

crisis_mask = p.loc[Xte.index, "evt_any"].fillna(0).astype(bool).values \
    if "evt_any" in p.columns else np.zeros(len(Xte), bool)
mean_abs = pd.DataFrame({
    "calm":   np.abs(shap_vals[~crisis_mask]).mean(0),
    "crisis": np.abs(shap_vals[crisis_mask]).mean(0) if crisis_mask.any() else np.nan
}, index=Xte.columns).sort_values("crisis", ascending=False)
print("\nTop drivers, calm vs crisis (mean |SHAP|):")
print(mean_abs.head(12).round(5).to_string())
mean_abs.to_csv(f"{OUT_DIR}/sq2_shap_calm_vs_crisis.csv")

## Does GPR lead or lag? — Granger causality (Task E)
Prior evidence (ablation/SHAP/reverse test) shows GPR is a *weak predictor*, not that it *lags*. This tests the lead-lag direction directly: does `spread_jkm_ttf` Granger-cause `gpr`, or the reverse? Series are first-differenced (both are highly persistent). Saves `gpr_granger.csv`. **Read the result honestly — it may not support the "GPR lags" framing.**

In [ ]:
# %% Task E: Granger causality + cross-correlation, spread vs GPR
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np, pandas as pd

_sub = p[["spread_jkm_ttf", "gpr"]].dropna()
_d = _sub.diff().dropna()                       # difference to stationarity
MAXLAG = 10
_g_gpr_to_spread = grangercausalitytests(_d[["spread_jkm_ttf", "gpr"]], maxlag=MAXLAG, verbose=False)
_g_spread_to_gpr = grangercausalitytests(_d[["gpr", "spread_jkm_ttf"]], maxlag=MAXLAG, verbose=False)

def _lagp(g):   # p-value of the ssr F-test at each lag
    return {l: g[l][0]["ssr_ftest"][1] for l in g}
pA, pB = _lagp(_g_gpr_to_spread), _lagp(_g_spread_to_gpr)
gr = pd.DataFrame({"lag": list(pA), "p(GPR->spread)": list(pA.values()),
                   "p(spread->GPR)": list(pB.values())}).round(4)
try: display(gr)
except NameError: print(gr.to_string(index=False))
gr.to_csv(f"{OUT_DIR}/gpr_granger.csv", index=False)

minA, minB = min(pA.values()), min(pB.values())
print(f"\nmin p  GPR->spread = {minA:.4f} | spread->GPR = {minB:.4f}")
if minB < 0.05 and minA >= 0.05:
    print("Interpretation: spread Granger-causes GPR (GPR LAGS the market) -> supports the lag claim.")
elif minA < 0.05 and minB >= 0.05:
    print("Interpretation: GPR Granger-causes the spread (GPR LEADS statistically). This does NOT support the "
          "'GPR lags' framing. Reconcile with ablation/SHAP (GPR is still a weak predictor); report honestly.")
else:
    print("Interpretation: mixed/bidirectional or neither — 'GPR lags' remains an interpretation, not a proven result.")

# Cross-correlation: TTF return vs GPR at +/- lags (negative lag = GPR leads)
_ret = np.log(p["ttf_c1_usd"]).diff()
_cc = pd.concat([_ret.rename("ret"), p["gpr"].rename("gpr")], axis=1).dropna()
_lags = range(-10, 11)
_ccf = {L: _cc["ret"].corr(_cc["gpr"].shift(L)) for L in _lags}
print("Cross-corr TTF-return vs GPR (lag<0 = GPR leads):",
      {L: round(v, 3) for L, v in _ccf.items() if abs(L) <= 3})


In [ ]:
# %% Cell A12: SECTION 5.2 - Shock premium (crisis error the geo-blind model misses)
blind_err  = np.abs(yte.values - m_abl.predict(Xte_abl))
informed_err = np.abs(yte.values - m_full.predict(Xte))
prem = pd.DataFrame({"blind": blind_err, "informed": informed_err}, index=Xte.index)
prem["crisis"] = p.loc[prem.index, "evt_any"].fillna(0).astype(int) if "evt_any" in p.columns else 0
sp_crisis = prem.loc[prem["crisis"] == 1, ["blind", "informed"]].mean()
print("Shock premium (mean abs error in crisis windows):")
print(f"  geo-blind model : {sp_crisis['blind']:.5f}")
print(f"  geo-informed    : {sp_crisis['informed']:.5f}")
print(f"  premium closed  : {sp_crisis['blind'] - sp_crisis['informed']:+.5f}")


In [ ]:
# %% Cell A13: SECTION 4.6 - TFT probabilistic tier (PyTorch, quantile forecasts)
# The distinction lever: quantile forecasts + coverage, not point RMSE.
# Kept as a compact, runnable block; expand horizons/epochs as compute allows.
try:
    import torch
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import QuantileLoss
    import lightning.pytorch as pl

    # Long format for a single hub target
    tft_df = pd.DataFrame({
        "time_idx": np.arange(len(data)),
        "target":   data[TARGET].values,
        "group":    "ttf",
    })
    for c in ["gpr", "storage_anom", "spread_jkm_ttf", "ttf_slope"]:
        if c in p.columns:
            tft_df[c] = p[c].reindex(data.index).ffill().fillna(0).values

    max_enc, max_pred = 60, 5
    cutoff = tft_df["time_idx"].max() - 60
    known = [c for c in ["gpr","storage_anom","spread_jkm_ttf","ttf_slope"] if c in tft_df]
    training = TimeSeriesDataSet(
        tft_df[tft_df.time_idx <= cutoff], time_idx="time_idx", target="target",
        group_ids=["group"], max_encoder_length=max_enc, max_prediction_length=max_pred,
        time_varying_unknown_reals=["target"], time_varying_known_reals=known,
        target_normalizer=GroupNormalizer(groups=["group"]))
    val = TimeSeriesDataSet.from_dataset(training, tft_df, min_prediction_idx=cutoff+1)
    tl = training.to_dataloader(train=True, batch_size=64)
    vl = val.to_dataloader(train=False, batch_size=64)

    tft = TemporalFusionTransformer.from_dataset(
        training, learning_rate=0.03, hidden_size=16, attention_head_size=2,
        dropout=0.1, loss=QuantileLoss(), output_size=7)
    trainer = pl.Trainer(max_epochs=15, enable_progress_bar=False, logger=False,
                         enable_checkpointing=False)
    trainer.fit(tft, tl, vl)
    preds = tft.predict(vl, mode="quantiles")
    print("TFT trained. Quantile output shape:", tuple(preds.shape),
          "-> evaluate pinball loss & interval coverage next.")
except Exception as e:
    print("TFT block skipped/incomplete:", repr(e),
          "\n(Run on a GPU runtime; this is the distinction lever, not required for SQ1 point results.)")


In [ ]:
# %% Cell A14: Results table - assemble SQ1 comparison for the dissertation
summary = pd.DataFrame({
    "Model": list(results.keys()),
    "MeanRMSE_returns": [round(results[k]["rmse"], 5) for k in results],
})
summary = summary.merge(dm_df.rename(columns={"Model":"Model"}), on="Model", how="left")
print("=== SQ1 model comparison ===")
print(summary.to_string(index=False))
summary.to_csv(f"{OUT_DIR}/sq1_summary.csv", index=False)
print("\nAll outputs saved to", OUT_DIR)

In [ ]:
# %% Cell A13b: TFT probabilistic evaluation - pinball loss, coverage, crisis widening
# The distinction lever: does the TFT produce calibrated intervals that WIDEN in crises?
import numpy as np

# Get quantile predictions and the matching actuals from the validation set
raw = tft.predict(vl, mode="raw", return_x=True)
q_levels = tft.loss.quantiles                      # e.g. [0.02,0.1,0.25,0.5,0.75,0.9,0.98]
preds_q = raw.output.prediction.cpu().numpy()      # (samples, horizon, n_quantiles)

# Align actuals (decoder targets)
actuals = raw.x["decoder_target"].cpu().numpy()    # (samples, horizon)

# Flatten across horizon for aggregate metrics
P = preds_q.reshape(-1, preds_q.shape[-1])
A = actuals.reshape(-1)

def pinball_loss(y, qp, quantiles):
    out = []
    for i, q in enumerate(quantiles):
        d = y - qp[:, i]
        out.append(np.mean(np.maximum(q * d, (q - 1) * d)))
    return np.mean(out)

def coverage(y, lo, hi):
    return np.mean((y >= lo) & (y <= hi))

# locate quantile indices for 10/50/90 (nearest available)
def qidx(target): return int(np.argmin(np.abs(np.array(q_levels) - target)))
i10, i50, i90 = qidx(0.1), qidx(0.5), qidx(0.9)

pl = pinball_loss(A, P, q_levels)
cov80 = coverage(A, P[:, i10], P[:, i90])
print(f"TFT pinball loss (avg): {pl:.5f}")
print(f"80% interval coverage : {cov80:.3f}  (well-calibrated ~ 0.80)")
print(f"Median interval width : {np.mean(P[:, i90] - P[:, i10]):.5f}")

# --- The thesis-critical test: do intervals WIDEN in crisis windows? ---
# Map each prediction back to its date to tag crisis vs calm.
dec_time = raw.x["decoder_time_idx"].cpu().numpy().reshape(-1)
# tft_df has time_idx -> real date; build the lookup
idx_to_date = dict(zip(tft_df["time_idx"], data.index))
dates = pd.to_datetime([idx_to_date.get(t, pd.NaT) for t in dec_time])
crisis = np.zeros(len(dates), bool)
for nm,(s_,e_) in EVENTS.items():
    e_ = pd.Timestamp(e_) if e_ else data.index.max()
    crisis |= (dates >= pd.Timestamp(s_)) & (dates <= e_)

width = P[:, i90] - P[:, i10]
print(f"\nMean 80% interval width - calm  : {width[~crisis].mean():.5f}")
print(f"Mean 80% interval width - crisis: {width[crisis].mean():.5f}")
print(f"Crisis widening factor          : {width[crisis].mean()/max(width[~crisis].mean(),1e-9):.2f}x")
# EXPECT: crisis width > calm width. If so, the TFT expresses elevated tail risk
# during geopolitical shocks - the probabilistic contribution GPR point-forecasts miss.

In [ ]:
for nm in ["ukraine","redsea","iran","any"]:
    if f"evt_{nm}" in p.columns:
        Xall[f"evt_{nm}"] = p[f"evt_{nm}"].reindex(Xall.index).fillna(0)
Xtr, Xte = Xall.iloc[:split], Xall.iloc[split:]   # rebuild splits
# then re-run the A14a block-ablation loop

In [ ]:
# %% Cell A14a: Block ablation - which feature GROUPS actually matter
# Ablation = remove a group of features, retrain, measure the rise in error.
# A big rise = that group was doing real work. A tiny rise = it wasn't.
from xgboost import XGBRegressor
import numpy as np

def make_xgb():
    return XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0)

# Define feature blocks by name pattern (uses the corrected notebook's Xall)
blocks = {
    "geopolitical_raw": [c for c in Xall.columns if "gpr" in c],
    "regime_dummies":   [c for c in Xall.columns if c.startswith("evt_")],
    "chokepoint":       [c for c in Xall.columns if "chk_" in c],
    "cross_hub_spread": [c for c in Xall.columns if c.startswith("spread_")],
    "curve_shape":      [c for c in Xall.columns if any(s in c for s in
                          ["_slope","_prompt_spread","_curvature"])],
    "storage":          [c for c in Xall.columns if "storage" in c],
    "price_lags":       [c for c in Xall.columns if "_lag" in c],
}
# NOTE: evt_ dummies may not be in Xall yet - add them (see A14c) then re-run.

split = int(len(Xall) * 0.8)
Xtr, Xte = Xall.iloc[:split], Xall.iloc[split:]
ytr, yte = yall[TARGET].iloc[:split], yall[TARGET].iloc[split:]

m_full = make_xgb().fit(Xtr, ytr)
rmse_full = np.sqrt(np.mean((yte.values - m_full.predict(Xte))**2))
print(f"FULL model RMSE: {rmse_full:.5f}\n")

abl_rows = []
for name, cols in blocks.items():
    cols = [c for c in cols if c in Xall.columns]
    if not cols:
        print(f"  [skip] {name}: no columns present"); continue
    Xtr_a, Xte_a = Xtr.drop(columns=cols), Xte.drop(columns=cols)
    m = make_xgb().fit(Xtr_a, ytr)
    rmse_a = np.sqrt(np.mean((yte.values - m.predict(Xte_a))**2))
    abl_rows.append({"block": name, "n_features": len(cols),
                     "RMSE_without": round(rmse_a, 5),
                     "delta_vs_full": round(rmse_a - rmse_full, 5),
                     "pct_change": round(100*(rmse_a - rmse_full)/rmse_full, 1)})
abl = pd.DataFrame(abl_rows).sort_values("delta_vs_full", ascending=False)
print("--- Block ablation (bigger delta = more important) ---")
print(abl.to_string(index=False))
abl.to_csv(f"{OUT_DIR}/sq2_block_ablation.csv", index=False)
# INTERPRETATION: expect cross_hub_spread / curve_shape to have the largest delta
# (structure carries the signal), geopolitical_raw small (GPR adds little directly),
# regime_dummies/chokepoint hopefully meaningful (your sharper geopolitical markers).

In [ ]:
# %% Cell A14b: Reverse ablation - prove the transmission runs THROUGH structure
# If dropping spreads craters accuracy but dropping GPR doesn't, that IS the
# "geopolitics transmits through market structure" claim, shown directly.
keep_only_geo = [c for c in Xall.columns if ("gpr" in c or c.startswith("evt_")
                 or "chk_" in c)]
struct_cols   = [c for c in Xall.columns if c.startswith("spread_")
                 or any(s in c for s in ["_slope","_prompt_spread","_curvature"])]

def rmse_on(cols):
    cols = [c for c in cols if c in Xall.columns]
    if not cols: return np.nan
    m = make_xgb().fit(Xtr[cols], ytr)
    return np.sqrt(np.mean((yte.values - m.predict(Xte[cols]))**2))

print("Structure-only vs geopolitics-only (predictive content test):")
print(f"  ALL features            : {rmse_full:.5f}")
print(f"  STRUCTURE features only  : {rmse_on(struct_cols):.5f}")
print(f"  GEOPOLITICAL only        : {rmse_on(keep_only_geo):.5f}")
print(f"  Random-walk (predict 0)  : {np.sqrt(np.mean(yte.values**2)):.5f}")
# INTERPRETATION: if 'structure only' ~ 'all features' while 'geopolitical only'
# ~ random walk, the predictive signal lives in the price structure, and
# geopolitics reaches price THROUGH that structure. That is your thesis, evidenced.

#Additional Analysis

In [ ]:
# %% Cell C1: PER-CRISIS COMPARATIVE TABLE - the taxonomy, made rigorous
# Runs the same battery separately for each crisis so their SIGNATURES contrast.
# This is the core exhibit: supply destruction vs transit disruption vs chokepoint.
import numpy as np, pandas as pd

def crisis_mask(nm):
    s, e = EVENTS[nm]; e = pd.Timestamp(e) if e else p.index.max()
    return (p.index >= pd.Timestamp(s)) & (p.index <= e)

calm_mask = ~np.any([crisis_mask(k) for k in EVENTS], axis=0)
calm_spread = p.loc[calm_mask, "spread_jkm_ttf"].mean()

rows = []
for nm in EVENTS:
    m = crisis_mask(nm); sub = p[m]
    rows.append({
        "Crisis": nm,
        "Type": {"ukraine":"supply destruction","redsea":"transit disruption",
                 "iran":"chokepoint closure"}[nm],
        "N days": int(m.sum()),
        "Spread mean": round(sub["spread_jkm_ttf"].mean(), 2),
        "Spread vs calm": round(sub["spread_jkm_ttf"].mean() - calm_spread, 2),
        "Spread max": round(sub["spread_jkm_ttf"].abs().max(), 2),
        "TTF vol (ann)": round(sub["ttf_ret"].std() * np.sqrt(252), 3),
        "JKM vol (ann)": round(sub["jkm_ret"].std() * np.sqrt(252), 3),
        "HH vol (ann)":  round(sub["hh_ret"].std() * np.sqrt(252), 3),
        "corr JKM-TTF": round(sub["jkm_ret"].corr(sub["ttf_ret"]), 3),
        "corr TTF-HH":  round(sub["ttf_ret"].corr(sub["hh_ret"]), 3),
    })
comp = pd.DataFrame(rows)
print("=== PER-CRISIS COMPARATIVE SIGNATURES ===")
print(comp.to_string(index=False))
comp.to_csv(f"{OUT_DIR}/crisis_comparison.csv", index=False)
# INTERPRET (taxonomy predictions):
#  Ukraine  -> spread WIDENS (supply destruction hits Europe specifically)
#  Red Sea  -> spread ~ unchanged (null / transit disruption below arbitrage cost)
#  Hormuz   -> spread COMPRESSES (symmetric closure lifts both basins together)
#  Watch also: does corr JKM-TTF rise in Hormuz (co-movement) vs Ukraine?
# CAUTION: Hormuz N is small (~80 days); report as descriptive, not precision estimate.

In [ ]:
# %% Cell C2: ROLLING CONNECTEDNESS - watch the network tighten at each event
# Static connectedness was one number; this shows it move through time.
from statsmodels.tsa.api import VAR

cols = [f"{h}_ret" for h in HUBS]
rr = p[cols].dropna()
W, STEP, H = 150, 1, 10     # STEP=1 -> daily estimates, ~79 Iran points

n_hub = len(cols)

def total_connectedness(window_df):
    try:
        fe = np.asarray(VAR(window_df).fit(1).fevd(H).decomp)
        if fe.shape == (H, n_hub, n_hub):      mat = fe[-1]
        elif fe.shape == (n_hub, H, n_hub):    mat = fe[:, -1, :]
        else:                                  mat = np.take(fe, -1, axis=list(fe.shape).index(H))
        mat = np.asarray(mat).reshape(n_hub, n_hub)
        return 100 * (mat.sum() - np.trace(mat)) / mat.sum()
    except Exception:
        return np.nan

pts = {}
for i in range(W, len(rr), STEP):
    pts[rr.index[i-1]] = total_connectedness(rr.iloc[i-W:i])
tc = pd.Series(pts).dropna()

fig, ax = plt.subplots(figsize=(15, 4.5))
tc.plot(ax=ax, lw=1.2, color="tab:blue")
colors = {"ukraine":"tab:red","redsea":"tab:orange","iran":"tab:purple"}
for nm,(s_,e_) in EVENTS.items():
    e_ = e_ or str(p.index.max().date())
    ax.axvspan(pd.Timestamp(s_), pd.Timestamp(e_), alpha=0.12, color=colors[nm], label=nm)
ax.set_ylabel("Total connectedness (%)"); ax.legend(ncol=4)
ax.set_title("Rolling Diebold-Yilmaz total connectedness (150d) - network tightens in crises")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/rolling_connectedness.png", dpi=150); plt.show()

# Mean connectedness by regime (numeric companion to the plot) - FIXED
reg = pd.Series("calm", index=tc.index)
for nm in EVENTS:
    s_, e_ = EVENTS[nm]
    e_ = pd.Timestamp(e_) if e_ else p.index.max()
    m = (tc.index >= pd.Timestamp(s_)) & (tc.index <= e_)   # boolean array on tc.index
    reg[m] = nm
print("Mean connectedness by regime:")
print(tc.groupby(reg).agg(["mean", "max", "count"]).round(1).to_string())
# INTERPRET: expect higher connectedness in Ukraine/Hormuz than calm or Red Sea.

In [ ]:
# %% Cell C3: EVENT STUDY - directional response (which hub leads each shock)
# Cumulative return per hub around each event start. This exposes the
# asymmetric (Ukraine, TTF-led) vs symmetric (Hormuz, TTF+JKM together) pattern.
def event_car(nm, hubs, pre=5, post=20):
    s, _ = EVENTS[nm]; d = pd.Timestamp(s)
    loc = p.index.searchsorted(d)
    if loc - pre < 0 or loc + post > len(p):
        pre, post = min(pre, loc), min(post, len(p)-loc)
    out = {}
    for h in hubs:
        r = p[f"{h}_ret"].iloc[loc-pre:loc+post]
        out[h] = r.cumsum()
    car = pd.DataFrame(out)
    car.index = range(-pre, -pre + len(car))     # event time (day 0 = shock)
    return car

fig, axes = plt.subplots(1, len(EVENTS), figsize=(16, 4.5), sharey=False)
car_summary = []
for ax, nm in zip(axes, EVENTS):
    car = event_car(nm, HUBS)
    (car * 100).plot(ax=ax)
    ax.axvline(0, color="k", ls="--", lw=0.8); ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(f"{nm}\n(day 0 = event start)"); ax.set_xlabel("event day")
    ax.set_ylabel("cumulative return %")
    final = (car.iloc[-1] * 100).round(1).to_dict()
    car_summary.append({"crisis": nm, **{f"CAR_{h}": final[h] for h in HUBS}})
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/event_study_car.png", dpi=150); plt.show()
print("=== Cumulative abnormal return (%) by hub, +20 days ===")
print(pd.DataFrame(car_summary).to_string(index=False))
# INTERPRET: Ukraine -> TTF strongly +, HH flat/negative (Europe-specific, asymmetric).
#            Hormuz  -> TTF AND JKM + together (symmetric), HH decoupled.
#            Red Sea -> all muted (null case).

In [ ]:
# %% Cell C5: EQUAL ACUTE-WINDOW COMPARISON - removes the regime-duration confound
# Your insight: a 439-day Ukraine regime blends the acute shock with months of
# re-pricing and adjustment, while Hormuz (79d) is mostly acute. Averaging over
# unequal regimes is not a fair comparison. This compares the FIRST N days after
# each shock onset - equal windows, acute phase only.
import numpy as np, pandas as pd
from statsmodels.tsa.api import VAR

def acute_window(nm, days):
    s, _ = EVENTS[nm]
    loc = p.index.searchsorted(pd.Timestamp(s))
    return p.iloc[loc: loc + days]

def window_connectedness(df):
    cols = [f"{h}_ret" for h in HUBS]
    w = df[cols].dropna()
    if len(w) < 15:                      # too few obs for a stable VAR
        return np.nan
    try:
        fe = np.asarray(VAR(w).fit(1).fevd(10).decomp)
        nn = len(cols)
        if fe.shape == (10, nn, nn):   mat = fe[-1]
        elif fe.shape == (nn, 10, nn): mat = fe[:, -1, :]
        else:                          mat = np.take(fe, -1, axis=list(fe.shape).index(10))
        mat = np.asarray(mat).reshape(nn, nn)
        return 100 * (mat.sum() - np.trace(mat)) / mat.sum()
    except Exception:
        return np.nan

# Compare across two window lengths so you can see robustness to the choice
for D in [20, 30, 45]:
    rows = []
    for nm in EVENTS:
        w = acute_window(nm, D)
        sub = w
        rows.append({
            "Crisis": nm,
            "N": len(w),
            "Connectedness": round(window_connectedness(w), 1),
            "Spread change": round(sub["spread_jkm_ttf"].iloc[-1] - sub["spread_jkm_ttf"].iloc[0], 2)
                              if len(sub) else np.nan,
            "TTF vol": round(sub["ttf_ret"].std() * np.sqrt(252), 3),
            "JKM vol": round(sub["jkm_ret"].std() * np.sqrt(252), 3),
            "corr JKM-TTF": round(sub["jkm_ret"].corr(sub["ttf_ret"]), 3),
        })
    print(f"\n=== ACUTE WINDOW = first {D} trading days after onset ===")
    print(pd.DataFrame(rows).to_string(index=False))

# Also add a matched calm baseline: connectedness in the 45 days BEFORE each shock
print("\n=== Pre-shock baseline (45 days BEFORE each onset) ===")
for nm in EVENTS:
    s, _ = EVENTS[nm]
    loc = p.index.searchsorted(pd.Timestamp(s))
    pre = p.iloc[max(0, loc-45): loc]
    print(f"  {nm:8s} pre-shock connectedness = {window_connectedness(pre):5.1f}")
# INTERPRET: compare each crisis's ACUTE connectedness to ITS OWN pre-shock level.
# A jump from pre to acute = the shock tightened the network. This within-event
# change is far more meaningful than comparing raw levels across crises of
# different length. Expect Hormuz & Ukraine acute > their pre-shock; Red Sea flat.

In [ ]:
# %% Cell B1: Add three model families for systematic coverage (+ shared tuner)
# Rationale: cover the major supervised families, not near-duplicates.
#   Linear/regularised -> ElasticNet | Kernel -> SVR | Neural (shallow) -> MLP
#   (tree ensembles RF/XGBoost and deep sequence models handled separately)
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import numpy as np, pandas as pd

# Scaled models (linear/kernel/neural need standardisation; trees do not)
NEEDS_SCALING = {"ElasticNet", "SVR", "MLP"}

def wf_rmse(build_fn, Xdf, yser, scale=False):
    """Walk-forward mean RMSE for a model builder, leakage-safe scaling per fold."""
    errs = []
    for tr, te in tscv.split(Xdf):
        Xtr, Xte = Xdf.iloc[tr], Xdf.iloc[te]
        ytr, yte = yser.iloc[tr], yser.iloc[te]
        if scale:
            sc = StandardScaler().fit(Xtr)
            Xtr = pd.DataFrame(sc.transform(Xtr), index=Xtr.index, columns=Xtr.columns)
            Xte = pd.DataFrame(sc.transform(Xte), index=Xte.index, columns=Xte.columns)
        m = build_fn(); m.fit(Xtr, ytr)
        errs.append(np.sqrt(np.mean((yte.values - m.predict(Xte))**2)))
    return float(np.mean(errs))

# clean design matrix reused from SQ1 (Xall, yall, TARGET, tscv already defined)
Xd, yd = Xall, yall[TARGET]
print("Tuning on:", Xd.shape, "| target:", TARGET)

In [ ]:
# %% Cell B2 (FAST, WIDENED GRIDS): single-split tuning + LinearSVR
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler

# One expanding split for tuning (last 20% as validation)
cut = int(len(Xd) * 0.8)
Xtr_t, Xva_t = Xd.iloc[:cut], Xd.iloc[cut:]
ytr_t, yva_t = yd.iloc[:cut], yd.iloc[cut:]
sc = StandardScaler().fit(Xtr_t)
Xtr_s = pd.DataFrame(sc.transform(Xtr_t), index=Xtr_t.index, columns=Xtr_t.columns)
Xva_s = pd.DataFrame(sc.transform(Xva_t), index=Xva_t.index, columns=Xva_t.columns)

def score(build_fn, scaled=False):
    Xa, Xb = (Xtr_s, Xva_s) if scaled else (Xtr_t, Xva_t)
    m = build_fn(); m.fit(Xa, ytr_t)
    return float(np.sqrt(np.mean((yva_t.values - m.predict(Xb))**2)))

fig, axes = plt.subplots(2, 3, figsize=(17, 9)); axes = axes.ravel()
best = {}
def curve(ax, xs, ys, xlabel, title, logx=False):
    ax.plot(xs, ys, "o-", lw=1.5); i = int(np.argmin(ys))
    ax.axvline(xs[i], color="red", ls="--", alpha=0.6)
    ax.scatter([xs[i]], [ys[i]], color="red", s=60, zorder=5)
    ax.annotate(f"opt {xs[i]}", (xs[i], ys[i]), textcoords="offset points", xytext=(8,8), color="red")
    if logx: ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("val RMSE"); ax.set_title(title); return xs[i]

depths=[2,3,4,6,8,10,12]
best["xgb_depth"]=curve(axes[0],depths,[score(lambda d=d:XGBRegressor(n_estimators=300,max_depth=d,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=3.0,n_jobs=-1)) for d in depths],"max_depth","XGBoost: depth")

nest=[100,200,400,800,1200,1600]
best["xgb_nest"]=curve(axes[1],nest,[score(lambda k=k:XGBRegressor(n_estimators=k,max_depth=best["xgb_depth"],learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=3.0,n_jobs=-1)) for k in nest],"n_estimators","XGBoost: n trees")

rfd=[5,8,12,20,30,40]
best["rf_depth"]=curve(axes[2],rfd,[score(lambda d=d:RandomForestRegressor(n_estimators=200,max_depth=d,n_jobs=-1)) for d in rfd],"max_depth","Random Forest: depth")

alphas=[1e-6,1e-5,1e-4,1e-3,1e-2,1e-1]
best["en_alpha"]=curve(axes[3],alphas,[score(lambda a=a:ElasticNet(alpha=a,l1_ratio=0.5,max_iter=10000),scaled=True) for a in alphas],"alpha (log)","ElasticNet: alpha",logx=True)

Cs=[0.01,0.1,1,10,100]
best["svr_C"]=curve(axes[4],Cs,[score(lambda c=c:LinearSVR(C=c,max_iter=10000),scaled=True) for c in Cs],"C (log)","LinearSVR: C",logx=True)

mw=[16,32,64,128,256]
best["mlp_width"]=curve(axes[5],mw,[score(lambda w=w:MLPRegressor(hidden_layer_sizes=(w,),max_iter=1500,early_stopping=True,alpha=1e-2,learning_rate_init=1e-3),scaled=True) for w in mw],"hidden units","MLP: width")
fig.suptitle("Hyperparameter Tuning Curves: Validation RMSE by Model",
             fontsize=15, fontweight="bold", color="#500778", y=1.02)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/hyperparam_curves.png", dpi=150, bbox_inches="tight")
plt.show()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/hyperparam_curves.png",dpi=150); plt.show()
print("Chosen hyperparameters:");
for k,v in best.items(): print(f"  {k:12s} = {v}")

#Sweep for XGBoost Hyper

In [ ]:
# %% APPENDIX — XGBoost hyperparameter sweeps (detailed, single model)
import matplotlib.pyplot as plt
import numpy as np

# reuse the same tuning split, score(), and curve() helper from Cell B2
# (cut, Xtr_t/Xva_t, ytr_t/yva_t, sc, Xtr_s/Xva_s all already defined)

def curve_ap(ax, xs, ys, xlabel, title, logx=False):
    ax.plot(xs, ys, "o-", lw=1.6, color="#500778")
    i = int(np.argmin(ys))
    ax.axvline(xs[i], color="#C62828", ls="--", alpha=0.7)
    ax.scatter([xs[i]], [ys[i]], color="#C62828", s=70, zorder=5)
    ax.annotate(f"opt {xs[i]}", (xs[i], ys[i]),
                textcoords="offset points", xytext=(8, 8),
                color="#C62828", fontweight="bold", fontsize=9)
    if logx: ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("validation RMSE")
    ax.set_title(title, fontsize=11, fontweight="bold")
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    return xs[i]

# fix the other params at sensible defaults while sweeping each one
BASE = dict(subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1)

fig, axes = plt.subplots(2, 2, figsize=(11, 8)); axes = axes.ravel()
xgb_best = {}

# 1) max_depth
depths = [2, 3, 4, 6, 8, 10, 12]
xgb_best["max_depth"] = curve_ap(
    axes[0], depths,
    [score(lambda d=d: XGBRegressor(n_estimators=300, max_depth=d,
                                    learning_rate=0.05, **BASE)) for d in depths],
    "max_depth", "Tree depth")

# 2) n_estimators (at the chosen depth)
nest = [100, 200, 400, 800, 1200, 1600]
xgb_best["n_estimators"] = curve_ap(
    axes[1], nest,
    [score(lambda k=k: XGBRegressor(n_estimators=k, max_depth=xgb_best["max_depth"],
                                    learning_rate=0.05, **BASE)) for k in nest],
    "n_estimators", "Number of trees")

# 3) learning_rate
lrs = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]
xgb_best["learning_rate"] = curve_ap(
    axes[2], lrs,
    [score(lambda lr=lr: XGBRegressor(n_estimators=xgb_best["n_estimators"],
                                      max_depth=xgb_best["max_depth"],
                                      learning_rate=lr, **BASE)) for lr in lrs],
    "learning_rate (log)", "Learning rate", logx=True)

# 4) reg_lambda (L2 regularisation)
lambdas = [0, 1, 3, 5, 10, 20]
xgb_best["reg_lambda"] = curve_ap(
    axes[3], lambdas,
    [score(lambda L=L: XGBRegressor(n_estimators=xgb_best["n_estimators"],
                                    max_depth=xgb_best["max_depth"],
                                    learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8,
                                    reg_lambda=L, n_jobs=-1)) for L in lambdas],
    "reg_lambda", "L2 regularisation")

fig.suptitle("XGBoost Hyperparameter Sweeps",
             fontsize=14, fontweight="bold", color="#500778", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/appendix_xgb_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("XGBoost chosen:", xgb_best)

#RF HP

In [ ]:
# %% APPENDIX — Random Forest hyperparameter sweeps
import matplotlib.pyplot as plt
import numpy as np

def curve_ap(ax, xs, ys, xlabel, title, logx=False):
    ax.plot(xs, ys, "o-", lw=1.6, color="#1B9E77")
    i = int(np.argmin(ys))
    ax.axvline(xs[i], color="#C62828", ls="--", alpha=0.7)
    ax.scatter([xs[i]], [ys[i]], color="#C62828", s=70, zorder=5)
    ax.annotate(f"opt {xs[i]}", (xs[i], ys[i]),
                textcoords="offset points", xytext=(8, 8),
                color="#C62828", fontweight="bold", fontsize=9)
    if logx: ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("validation RMSE")
    ax.set_title(title, fontsize=11, fontweight="bold")
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    return xs[i]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5)); axes = axes.ravel()
rf_best = {}

depths = [5, 8, 12, 20, 30, 40]
rf_best["max_depth"] = curve_ap(
    axes[0], depths,
    [score(lambda d=d: RandomForestRegressor(n_estimators=200, max_depth=d, n_jobs=-1)) for d in depths],
    "max_depth", "Tree depth")

nest = [100, 200, 400, 800, 1200]
rf_best["n_estimators"] = curve_ap(
    axes[1], nest,
    [score(lambda k=k: RandomForestRegressor(n_estimators=k, max_depth=rf_best["max_depth"], n_jobs=-1)) for k in nest],
    "n_estimators", "Number of trees")

leaf = [1, 2, 5, 10, 20, 50]
rf_best["min_samples_leaf"] = curve_ap(
    axes[2], leaf,
    [score(lambda L=L: RandomForestRegressor(n_estimators=rf_best["n_estimators"],
           max_depth=rf_best["max_depth"], min_samples_leaf=L, n_jobs=-1)) for L in leaf],
    "min_samples_leaf", "Leaf size")

fig.suptitle("Random Forest Hyperparameter Sweeps", fontsize=14, fontweight="bold", color="#500778", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/appendix_rf_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("RF chosen:", rf_best)

#ElasticNet HP



In [ ]:
# %% APPENDIX — ElasticNet hyperparameter sweeps
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5)); axes = axes.ravel()
en_best = {}

alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
en_best["alpha"] = curve_ap(
    axes[0], alphas,
    [score(lambda a=a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000), scaled=True) for a in alphas],
    "alpha (log)", "Regularisation strength", logx=True)

l1s = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
en_best["l1_ratio"] = curve_ap(
    axes[1], l1s,
    [score(lambda r=r: ElasticNet(alpha=en_best["alpha"], l1_ratio=r, max_iter=10000), scaled=True) for r in l1s],
    "l1_ratio", "L1 vs L2 mix")

fig.suptitle("ElasticNet Hyperparameter Sweeps", fontsize=14, fontweight="bold", color="#500778", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/appendix_en_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("ElasticNet chosen:", en_best)

#LinearSVR HP

In [ ]:
# %% APPENDIX — LinearSVR hyperparameter sweeps
from sklearn.svm import LinearSVR
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5)); axes = axes.ravel()
svr_best = {}

Cs = [0.01, 0.1, 1, 10, 100]
svr_best["C"] = curve_ap(
    axes[0], Cs,
    [score(lambda c=c: LinearSVR(C=c, max_iter=10000), scaled=True) for c in Cs],
    "C (log)", "Regularisation (C)", logx=True)

eps = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]
svr_best["epsilon"] = curve_ap(
    axes[1], eps,
    [score(lambda e=e: LinearSVR(C=svr_best["C"], epsilon=e, max_iter=10000), scaled=True) for e in eps],
    "epsilon", "Insensitivity margin")

fig.suptitle("LinearSVR Hyperparameter Sweeps", fontsize=14, fontweight="bold", color="#500778", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/appendix_svr_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("LinearSVR chosen:", svr_best)

#MLP HP

In [ ]:
# %% APPENDIX — MLP hyperparameter sweeps
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5)); axes = axes.ravel()
mlp_best = {}

widths = [16, 32, 64, 128, 256]
mlp_best["width"] = curve_ap(
    axes[0], widths,
    [score(lambda w=w: MLPRegressor(hidden_layer_sizes=(w,), max_iter=1500,
           early_stopping=True, alpha=1e-2, learning_rate_init=1e-3), scaled=True) for w in widths],
    "hidden units", "Layer width")

alphas_mlp = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
mlp_best["alpha"] = curve_ap(
    axes[1], alphas_mlp,
    [score(lambda a=a: MLPRegressor(hidden_layer_sizes=(mlp_best["width"],), max_iter=1500,
           early_stopping=True, alpha=a, learning_rate_init=1e-3), scaled=True) for a in alphas_mlp],
    "alpha (log)", "L2 regularisation", logx=True)

lrs_mlp = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
mlp_best["learning_rate"] = curve_ap(
    axes[2], lrs_mlp,
    [score(lambda lr=lr: MLPRegressor(hidden_layer_sizes=(mlp_best["width"],), max_iter=1500,
           early_stopping=True, alpha=mlp_best["alpha"], learning_rate_init=lr), scaled=True) for lr in lrs_mlp],
    "learning_rate (log)", "Learning rate", logx=True)

fig.suptitle("MLP Hyperparameter Sweeps", fontsize=14, fontweight="bold", color="#500778", y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/appendix_mlp_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("MLP chosen:", mlp_best)

#Overall Sweaps

In [ ]:
# %% APPENDIX PREP — sweep all hyperparameters, measure stability, flag which to keep
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVR

# ---- stability metrics ------------------------------------------------
def stability_report(name, xs, ys):
    """Judge whether a sweep curve is stable enough to report."""
    ys = np.asarray(ys, float)
    xs = list(xs)
    i_opt = int(np.argmin(ys))

    # 1) roughness: mean absolute second difference, normalised by the curve's range
    #    (low = smooth; high = zig-zaggy)
    if len(ys) >= 3:
        second_diff = np.abs(np.diff(ys, 2))
        rng = ys.max() - ys.min() if ys.max() > ys.min() else 1e-9
        roughness = second_diff.mean() / rng
    else:
        roughness = np.nan

    # 2) optimum prominence: how much better the best point is than the
    #    2nd-best DISTINCT neighbour, relative to the curve's noise level.
    sorted_y = np.sort(ys)
    gap_to_2nd = sorted_y[1] - sorted_y[0]
    noise = np.median(np.abs(np.diff(ys)))  # typical point-to-point wobble
    prominence = gap_to_2nd / noise if noise > 0 else np.inf

    # 3) is the optimum on a boundary? (boundary optima are less trustworthy)
    on_boundary = (i_opt == 0) or (i_opt == len(xs) - 1)

    # verdict
    stable = (roughness < 0.35) and (not on_boundary or prominence > 1.0)

    return dict(
        name=name, opt_x=xs[i_opt], opt_rmse=round(float(ys[i_opt]), 5),
        roughness=round(float(roughness), 3),
        prominence=round(float(prominence), 2),
        on_boundary=on_boundary,
        verdict="STABLE" if stable else "review",
        ys=[round(float(v), 5) for v in ys], xs=xs,
    )

def run_sweep(name, build_fn, values, scaled=False):
    ys = []
    for v in values:
        ys.append(score(lambda vv=v: build_fn(vv), scaled=scaled))
    rep = stability_report(name, values, ys)
    print(f"{name:28s} opt={str(rep['opt_x']):>8} "
          f"rmse={rep['opt_rmse']:.5f}  rough={rep['roughness']:.3f}  "
          f"prom={rep['prominence']:.2f}  bnd={rep['on_boundary']}  -> {rep['verdict']}")
    return rep

# ---- define every candidate sweep -------------------------------------
reports = {}

# XGBoost
reports["xgb_max_depth"] = run_sweep("XGBoost max_depth",
    lambda d: XGBRegressor(n_estimators=300, max_depth=d, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1),
    [2,3,4,6,8,10,12])
_d = reports["xgb_max_depth"]["opt_x"]
reports["xgb_n_estimators"] = run_sweep("XGBoost n_estimators",
    lambda k: XGBRegressor(n_estimators=k, max_depth=_d, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1),
    [100,200,400,800,1200,1600])
reports["xgb_learning_rate"] = run_sweep("XGBoost learning_rate",
    lambda lr: XGBRegressor(n_estimators=400, max_depth=_d, learning_rate=lr,
                            subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1),
    [0.01,0.03,0.05,0.1,0.2,0.3])
reports["xgb_reg_lambda"] = run_sweep("XGBoost reg_lambda",
    lambda L: XGBRegressor(n_estimators=400, max_depth=_d, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, reg_lambda=L, n_jobs=-1),
    [0,1,3,5,10,20])

# Random Forest
reports["rf_max_depth"] = run_sweep("RF max_depth",
    lambda d: RandomForestRegressor(n_estimators=200, max_depth=d, n_jobs=-1),
    [5,8,12,20,30,40])
_rfd = reports["rf_max_depth"]["opt_x"]
reports["rf_n_estimators"] = run_sweep("RF n_estimators",
    lambda k: RandomForestRegressor(n_estimators=k, max_depth=_rfd, n_jobs=-1),
    [100,200,400,800,1200])
reports["rf_min_samples_leaf"] = run_sweep("RF min_samples_leaf",
    lambda L: RandomForestRegressor(n_estimators=200, max_depth=_rfd, min_samples_leaf=L, n_jobs=-1),
    [1,2,5,10,20,50])
reports["rf_max_features"] = run_sweep("RF max_features",
    lambda f: RandomForestRegressor(n_estimators=200, max_depth=_rfd, max_features=f, n_jobs=-1),
    [0.3,0.5,0.7,0.9,1.0])

# ElasticNet
reports["en_alpha"] = run_sweep("ElasticNet alpha",
    lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000),
    [1e-6,1e-5,1e-4,1e-3,1e-2,1e-1], scaled=True)
reports["en_l1_ratio"] = run_sweep("ElasticNet l1_ratio",
    lambda r: ElasticNet(alpha=1e-6, l1_ratio=r, max_iter=10000),
    [0.1,0.3,0.5,0.7,0.9,1.0], scaled=True)

# LinearSVR
reports["svr_C"] = run_sweep("LinearSVR C",
    lambda c: LinearSVR(C=c, max_iter=10000),
    [0.01,0.1,1,10,100], scaled=True)
reports["svr_epsilon"] = run_sweep("LinearSVR epsilon",
    lambda e: LinearSVR(C=1.0, epsilon=e, max_iter=10000),
    [0.0,0.01,0.05,0.1,0.2,0.5], scaled=True)

# MLP
reports["mlp_width"] = run_sweep("MLP width",
    lambda w: MLPRegressor(hidden_layer_sizes=(w,), max_iter=1500, early_stopping=True,
                           alpha=1e-2, learning_rate_init=1e-3),
    [16,32,64,128,256], scaled=True)
reports["mlp_alpha"] = run_sweep("MLP alpha",
    lambda a: MLPRegressor(hidden_layer_sizes=(32,), max_iter=1500, early_stopping=True,
                           alpha=a, learning_rate_init=1e-3),
    [1e-4,1e-3,1e-2,1e-1,1.0], scaled=True)
reports["mlp_learning_rate"] = run_sweep("MLP learning_rate",
    lambda lr: MLPRegressor(hidden_layer_sizes=(32,), max_iter=1500, early_stopping=True,
                            alpha=1e-2, learning_rate_init=lr),
    [1e-4,3e-4,1e-3,3e-3,1e-2], scaled=True)

# ---- summary table ----------------------------------------------------
print("\n" + "="*70)
print("STABILITY SUMMARY (promote only STABLE curves to Table B2 / appendix)")
print("="*70)
summary = pd.DataFrame([
    {k: r[k] for k in ["name","opt_x","opt_rmse","roughness","prominence","on_boundary","verdict"]}
    for r in reports.values()
])
print(summary.to_string(index=False))

stable_keys = [k for k,r in reports.items() if r["verdict"]=="STABLE"]
print(f"\nSTABLE sweeps ({len(stable_keys)}): {stable_keys}")
print("Store `reports` — the plotting cell will use only the STABLE ones.")

In [ ]:
# %% Dump final selected hyperparameters for the spec table
print("=== Classical model selected hyperparameters (leakage-safe tuning) ===")
for k, v in best.items():
    print(f"  {k:14s} = {v}")

print("\n=== Neural selected hyperparameters ===")
print(f"  LSTM hidden   = {best_lstm_h if 'best_lstm_h' in dir() else 'run Task E1'}")
print(f"  TCN channels  = {best_tcn_c if 'best_tcn_c' in dir() else 'run Task E1'}")
print(f"  lookback      = {LOOKBACK}")
print(f"  DNN widths    = (128, 64, 32, 16, 8)")

In [ ]:
# %% Cell B3: Hyperparameter table (documented values + rationale for the paper)
hp_rows = [
    ["Random Walk",  "none",                              "benchmark; predicts zero return"],
    ["ElasticNet",   f"alpha={best['en_alpha']}, l1=0.5", "regularised linear; alpha tuned on elbow"],
    ["SVR",          f"C={best['svr_C']}, rbf, gamma=scale","kernel model; C balances margin vs error"],
    ["MLP",          f"hidden=({best['mlp_width']},), early_stop", "shallow net; width tuned, small to limit overfit"],
    ["Random Forest",f"n=300, max_depth={best['rf_depth']}", "bagged trees; depth capped to control variance"],
    ["XGBoost",      f"n={best['xgb_nest']}, depth={best['xgb_depth']}, lr=0.03, lambda=3",
                     "boosted trees; shallow + regularised for small sample"],
]
hp = pd.DataFrame(hp_rows, columns=["Model", "Key hyperparameters", "Rationale"])
print(hp.to_string(index=False))
hp.to_csv(f"{OUT_DIR}/hyperparameter_table.csv", index=False)

In [ ]:
# %% Cell B4 (FAST): tuned comparison, capped runaway params, LinearSVR
from statsmodels.tsa.arima.model import ARIMA
from sklearn.svm import LinearSVR
from scipy.stats import norm

# Cap edge-pinned params (diminishing returns / overfit control)
XGB_NEST = min(best['xgb_nest'], 600)     # 1600 -> 600
RF_DEPTH = min(best['rf_depth'], 20)      # 30 -> 20

class _ZeroModel:
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X))

def scaled_wf(build_fn):
    preds, actual = [], []
    for tr, te in tscv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]; ytr, yte = yd.iloc[tr], yd.iloc[te]
        sc = StandardScaler().fit(Xtr)
        Xtr2 = pd.DataFrame(sc.transform(Xtr), index=Xtr.index, columns=Xtr.columns)
        Xte2 = pd.DataFrame(sc.transform(Xte), index=Xte.index, columns=Xte.columns)
        m = build_fn(); m.fit(Xtr2, ytr)
        preds.append(pd.Series(m.predict(Xte2), index=yte.index)); actual.append(yte)
    return pd.concat(preds), pd.concat(actual)

def tree_wf(build_fn):
    preds, actual = [], []
    for tr, te in tscv.split(Xd):
        m = build_fn(); m.fit(Xd.iloc[tr], yd.iloc[tr])
        preds.append(pd.Series(m.predict(Xd.iloc[te]), index=yd.iloc[te].index))
        actual.append(yd.iloc[te])
    return pd.concat(preds), pd.concat(actual)

models = {
  "RandomWalk":   ("tree",   lambda: _ZeroModel()),
  "ElasticNet":   ("scaled", lambda: ElasticNet(alpha=best['en_alpha'], l1_ratio=0.5, max_iter=10000)),
  "LinearSVR":    ("scaled", lambda: LinearSVR(C=best['svr_C'], max_iter=10000)),
  "MLP":          ("scaled", lambda: MLPRegressor(hidden_layer_sizes=(best['mlp_width'],),
                              max_iter=1500, early_stopping=True, alpha=1e-2, learning_rate_init=1e-3)),
  "RandomForest": ("tree",   lambda: RandomForestRegressor(n_estimators=300, max_depth=RF_DEPTH, n_jobs=-1)),
  "XGBoost":      ("tree",   lambda: XGBRegressor(n_estimators=XGB_NEST, max_depth=best['xgb_depth'],
                              learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1)),
}

res = {}
for name, (kind, fn) in models.items():
    preds, actual = (tree_wf(fn) if kind == "tree" else scaled_wf(fn))
    res[name] = {"rmse": np.sqrt(np.mean((actual.values - preds.values)**2)),
                 "preds": preds, "actual": actual}
    print(f"  {name} done")

def dm_test(e_a, e_b):
    d = e_a**2 - e_b**2
    stat = d.mean() / np.sqrt(np.var(d, ddof=1) / len(d))
    return stat, 2 * (1 - norm.cdf(abs(stat)))

base = res["RandomWalk"]; rows = []
for name in res:
    if name == "RandomWalk":
        rows.append([name, round(res[name]["rmse"], 5), "-", "-", "benchmark"]); continue
    idx = res[name]["actual"].index.intersection(base["actual"].index)
    e_m = (res[name]["actual"].loc[idx] - res[name]["preds"].loc[idx]).values
    e_b = (base["actual"].loc[idx]     - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_m, e_b)
    verdict = "ties RW" if pv >= 0.05 else ("beats RW" if stat < 0 else "WORSE than RW")
    rows.append([name, round(res[name]["rmse"], 5), round(stat, 3), round(pv, 4), verdict])
comp = pd.DataFrame(rows, columns=["Model","RMSE","DM stat","p-value","vs RW"]).sort_values("RMSE")
print("\n=== TUNED model comparison (walk-forward) ===")
print(comp.to_string(index=False))
comp.to_csv(f"{OUT_DIR}/sq1_tuned_comparison.csv", index=False)

In [ ]:
# %% Standalone: retune MLP only with lbfgs solver, compare to RW
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

def scaled_wf_single(build_fn):
    preds, actual = [], []
    for tr, te in tscv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]; ytr, yte = yd.iloc[tr], yd.iloc[te]
        sc = StandardScaler().fit(Xtr)
        Xtr2 = pd.DataFrame(sc.transform(Xtr), index=Xtr.index, columns=Xtr.columns)
        Xte2 = pd.DataFrame(sc.transform(Xte), index=Xte.index, columns=Xte.columns)
        m = build_fn(); m.fit(Xtr2, ytr)
        preds.append(pd.Series(m.predict(Xte2), index=yte.index)); actual.append(yte)
    return pd.concat(preds), pd.concat(actual)

# lbfgs = better optimiser for small datasets than default adam; stronger reg
mlp_preds, mlp_actual = scaled_wf_single(
    lambda: MLPRegressor(hidden_layer_sizes=(16,), solver="lbfgs",
                         max_iter=2000, alpha=1e-1, random_state=0))

mlp_rmse = np.sqrt(np.mean((mlp_actual.values - mlp_preds.values)**2))
idx = mlp_actual.index.intersection(base["actual"].index)
e_m = (mlp_actual.loc[idx] - mlp_preds.loc[idx]).values
e_b = (base["actual"].loc[idx] - base["preds"].loc[idx]).values
stat, pv = dm_test(e_m, e_b)
verdict = "ties RW" if pv >= 0.05 else ("beats RW" if stat < 0 else "WORSE than RW")
print(f"MLP (lbfgs)  RMSE={mlp_rmse:.5f}  DM={stat:.3f}  p={pv:.4f}  -> {verdict}")
print(f"(Random walk RMSE = {base['rmse']:.5f} for reference)")

# If it now works, update res so the final table includes it:
# res["MLP"] = {"rmse": mlp_rmse, "preds": mlp_preds, "actual": mlp_actual}

#Final Model Evaluation

In [ ]:
res["MLP"] = {"rmse": mlp_rmse, "preds": mlp_preds, "actual": mlp_actual}

# regenerate the final comparison table with the fixed MLP
rows = []
for name in res:
    if name == "RandomWalk":
        rows.append([name, round(res[name]["rmse"], 5), "-", "-", "benchmark"]); continue
    idx = res[name]["actual"].index.intersection(base["actual"].index)
    e_m = (res[name]["actual"].loc[idx] - res[name]["preds"].loc[idx]).values
    e_b = (base["actual"].loc[idx]     - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_m, e_b)
    verdict = "ties RW" if pv >= 0.05 else ("beats RW" if stat < 0 else "WORSE than RW")
    rows.append([name, round(res[name]["rmse"], 5), round(stat, 3), round(pv, 4), verdict])
comp = pd.DataFrame(rows, columns=["Model","RMSE","DM stat","p-value","vs RW"]).sort_values("RMSE")
print(comp.to_string(index=False))
comp.to_csv(f"{OUT_DIR}/sq1_tuned_comparison.csv", index=False)

## Leakage audit — scaler is fold-internal (Task G)
Examiner-critical check: `StandardScaler` for the linear/kernel/neural models must be fit on the **training fold only** and applied to the test fold, inside each walk-forward split (not fit on the whole sample before splitting). This cell programmatically confirms it. Trees (RF/XGB) are unscaled by design. Saves `sq2_leakage_audit.csv`.

In [ ]:
# %% Task G: confirm per-fold (train-only) scaling — no test leakage
import inspect, pandas as pd
checks = []
for fn in [scaled_wf, wf_rmse]:
    src = inspect.getsource(fn)
    loop_at  = src.find("for tr, te")
    fit_at   = src.find("StandardScaler().fit(Xtr")
    refit_test = "fit(Xte" in src or "fit_transform(Xte" in src
    ok = (loop_at != -1 and fit_at != -1 and fit_at > loop_at and not refit_test)
    checks.append({"function": fn.__name__, "fits_inside_fold": fit_at > loop_at,
                   "no_test_refit": not refit_test, "leakage_safe": ok})
audit = pd.DataFrame(checks)
try: display(audit)
except NameError: print(audit.to_string(index=False))
audit.to_csv(f"{OUT_DIR}/sq2_leakage_audit.csv", index=False)
print("VERDICT:", "PASS — scaler fit on train fold only; trees unscaled."
      if audit["leakage_safe"].all() else "REVIEW — a scaler may be fit outside the fold loop.")


## Feature importance on the BEST model — LinearSVR coefficients (Task S2)
The global SHAP earlier ran on XGBoost, but the best model is LinearSVR. For a linear model the standardised coefficient *is* the (exact) SHAP contribution, so we rank |coef| on the scaled features. Confirms the top drivers are spreads/curve and that **no GPR feature ranks** — making the "channel is structure, not the GPR index" finding hold on the best model, not only XGBoost. Saves `sq1_linearsvr_coefs.csv`.

In [ ]:
# %% Task S2: standardised LinearSVR coefficients as feature importance
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

_sc = StandardScaler().fit(Xd)                       # standardise full design (coef interpretability)
_Xs = _sc.transform(Xd)
_svr = LinearSVR(C=best["svr_C"], max_iter=20000).fit(_Xs, yd)
_coef = pd.Series(_svr.coef_, index=Xd.columns)
svr_imp = (pd.DataFrame({"feature": _coef.index, "coef": _coef.values,
                         "abs_coef": _coef.abs().values})
           .sort_values("abs_coef", ascending=False).reset_index(drop=True))
svr_imp["rank"] = svr_imp.index + 1
try: display(svr_imp.head(12))
except NameError: print(svr_imp.head(12).to_string(index=False))
svr_imp.to_csv(f"{OUT_DIR}/sq1_linearsvr_coefs.csv", index=False)

_gpr = svr_imp[svr_imp["feature"].str.contains("gpr", case=False)]
print("\nGPR features and their ranks (of {}):".format(len(svr_imp)))
print(_gpr[["feature", "rank"]].to_string(index=False) if len(_gpr) else "  (no GPR feature in the set)")
print("Top-5 drivers:", list(svr_imp["feature"].head(5)))
print("Conclusion: if the top drivers are spreads/curve and GPR ranks low, the structure-not-GPR channel holds on the BEST model. XGBoost SHAP corroborates.")


## Uncertainty — bootstrap confidence intervals on the error metrics (Task S3)
Answers "where is the uncertainty analysis?". For each model the 10 walk-forward fold-RMSEs are resampled with replacement 5000× to form a 95% CI, so the model table reports ranges, not just point estimates. Saves `sq2_metric_cis.csv`.

In [ ]:
import matplotlib.pyplot as plt

# Sort so the best (lowest) RMSE is at the top of the plot
plot_df = ci_df.sort_values("RMSE", ascending=True).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
metrics = ['RMSE', 'MAE']
baseline_model = 'RandomWalk'  # Change this if your baseline has a different name

for ax, metric in zip(axes, metrics):
    # matplotlib errorbar requires relative errors: [mean - lower_bound, upper_bound - mean]
    err_low = plot_df[metric] - plot_df[f'{metric}_CI_low']
    err_high = plot_df[f'{metric}_CI_high'] - plot_df[metric]

    # Plot the point estimates and CIs
    ax.errorbar(
        x=plot_df[metric],
        y=plot_df['Model'],
        xerr=[err_low, err_high],
        fmt='o',                # Circle marker for the mean
        color='#1f77b4',        # Dot color
        ecolor='#333333',        # Error bar color
        capsize=4,              # Caps on the ends of the error bars
        markersize=7,
        linestyle='None'
    )

    # Highlight the baseline to visually check for overlapping CIs
    if baseline_model in plot_df['Model'].values:
        base_idx = plot_df.index[plot_df['Model'] == baseline_model][0]
        base_val = plot_df.loc[base_idx, metric]
        base_low = plot_df.loc[base_idx, f'{metric}_CI_low']
        base_high = plot_df.loc[base_idx, f'{metric}_CI_high']

        # Draw a dashed line for the baseline mean
        ax.axvline(x=base_val, color='red', linestyle='--', alpha=0.8, label=f'{baseline_model} Mean')
        # Shade the baseline's CI region
        ax.axvspan(base_low, base_high, color='red', alpha=0.15, label=f'{baseline_model} 95% CI')

    ax.set_title(f'{metric} 95% Confidence Intervals')
    ax.set_xlabel(f'{metric} (Lower is Better)')
    ax.grid(axis='x', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Model')
axes[0].legend(loc='lower right')
# Invert y-axis so the first row in the dataframe (lowest error) appears at the top
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# %% Uncertainty visualization - bootstrap CIs on forecast error
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ci = pd.read_csv(f"{OUT_DIR}/sq2_metric_cis.csv")   # from the bootstrap cell
df = ci[["Model","RMSE","RMSE_CI_low","RMSE_CI_high"]].sort_values("RMSE").reset_index(drop=True)
rw = df[df.Model=="RandomWalk"]["RMSE"].values[0]

PURPLE="#500778"; RED="#c0392b"; RWCOL="#6b6b6b"
fig, ax = plt.subplots(figsize=(9, 5.5))
for yi, r in df.iterrows():
    is_rw = r["Model"] == "RandomWalk"
    c = RWCOL if is_rw else (PURPLE if r["RMSE"] < rw else RED)
    ax.plot([r["RMSE_CI_low"], r["RMSE_CI_high"]], [yi, yi], color=c, lw=2.5, alpha=0.5, zorder=2)
    for xend in [r["RMSE_CI_low"], r["RMSE_CI_high"]]:
        ax.plot([xend, xend], [yi-0.12, yi+0.12], color=c, lw=2, zorder=2)
    ax.scatter(r["RMSE"], yi, s=120, color=c, zorder=3, edgecolor="white", linewidth=1.2)

ax.axvline(rw, color=RWCOL, ls="--", lw=1, alpha=0.6, zorder=1)
ax.set_yticks(range(len(df))); ax.set_yticklabels(df["Model"], fontsize=11)
ax.set_xlabel("Walk-forward RMSE with 95% bootstrap CI (10 folds)")
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", lw=0.7); ax.set_axisbelow(True); ax.tick_params(left=False)
ax.text(0, len(df)+0.2, "Bars = 95% CI · dot = point estimate · dashed = random walk",
        fontsize=9.5, color="#6b6b6b")
ax.text(0, len(df)+0.7, "Forecast accuracy with uncertainty",
        fontsize=14, fontweight="bold", color="#1a1a1a")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_uncertainty_ci.png", dpi=300, bbox_inches="tight")
plt.show()

#Sequence models

In [ ]:
# %% Cell D1: Sequence model setup (LSTM + TCN) - PyTorch
# Run in the ANALYSIS notebook, after B4. Uses Xd, yd, tscv, base, dm_test, OUT_DIR
# (all already in memory). Tests whether sequence-aware models beat the flat models.
import torch, torch.nn as nn, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

LOOKBACK = 20          # each prediction uses the prior 20 trading days

def make_sequences(Xarr, yarr, idx, lookback=LOOKBACK):
    """3D [samples, lookback, features]; window ends day BEFORE target (no leakage)."""
    Xs, ys, dates = [], [], []
    for i in range(lookback, len(Xarr)):
        Xs.append(Xarr[i-lookback:i]); ys.append(yarr[i]); dates.append(idx[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), pd.DatetimeIndex(dates)

# Build sequences from the trimmed design matrix + target
Xarr = Xd.values; yarr = yd.values; idx = Xd.index
Xseq, yseq, dseq = make_sequences(Xarr, yarr, idx)
n_feat = Xseq.shape[2]
print(f"Sequences: {Xseq.shape} | targets: {yseq.shape} | features: {n_feat}")

class LSTMReg(nn.Module):
    def __init__(self, n_feat, hidden=32, layers=1, drop=0.1):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, layers, batch_first=True,
                            dropout=drop if layers > 1 else 0.0)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class TCN(nn.Module):
    """2-layer dilated temporal conv net (dilations 1,2 -> receptive field ~ lookback)."""
    def __init__(self, n_feat, ch=32, k=3, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_feat, ch, k, padding=k-1, dilation=1), nn.ReLU(), nn.Dropout(drop),
            nn.Conv1d(ch, ch, k, padding=2*(k-1), dilation=2), nn.ReLU(), nn.Dropout(drop))
        self.fc = nn.Linear(ch, 1)
    def forward(self, x):
        x = x.transpose(1, 2)          # [batch, features, time] for Conv1d
        h = self.net(x)
        return self.fc(h[:, :, -1]).squeeze(-1)

def train_model(model, Xtr, ytr, Xva=None, yva=None,
                epochs=100, lr=1e-3, batch=64, patience=10, min_delta=0.0, verbose=False):
    """Train with optional early stopping. Returns (model, train_losses, val_losses).
    If Xva/yva given: monitors validation MSE, stops after `patience` epochs with no
    improvement, restores best-epoch weights. If not given: trains `epochs` fixed
    (val_losses empty) — preserves the original behaviour for any legacy call."""
    import copy
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.MSELoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(ytr, dtype=torch.float32).to(DEVICE)
    use_val = Xva is not None and yva is not None
    if use_val:
        Xv = torch.tensor(Xva, dtype=torch.float32).to(DEVICE)
        yv = torch.tensor(yva, dtype=torch.float32).to(DEVICE)
    train_losses, val_losses = [], []
    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch):
            b = perm[i:i+batch]
            opt.zero_grad()
            lossf(model(Xt[b]), yt[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            train_losses.append(float(lossf(model(Xt), yt)))
            if use_val:
                v = float(lossf(model(Xv), yv))
                val_losses.append(v)
                if v < best_val - min_delta:
                    best_val, wait = v, 0
                    best_state = copy.deepcopy(model.state_dict())
                else:
                    wait += 1
                    if wait >= patience:
                        if verbose: print(f"  early stop @ epoch {ep+1} (best val {best_val:.5f})")
                        break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses

def predict(model, Xarr):
    """Predict on a (samples, lookback, features) array; returns 1D numpy."""
    model.eval()
    with torch.no_grad():
        X = torch.tensor(Xarr, dtype=torch.float32).to(DEVICE)
        return model(X).cpu().numpy()

## Deep fully-connected DNN baseline (Task C)
A plain feed-forward DNN (5 dense layers) on the flat features, evaluated with the same leakage-safe per-fold scaling and walk-forward as the classical models, added to `res` as `DNN`. Purpose: separate "deep nonlinear modelling fails" from "temporal architectures fail". Expected: the DNN also underperforms the random walk — strengthening the *data-volume* explanation (not an architecture verdict). Feeds the overfitting table (Task D).

In [ ]:
# %% Task C: deep fully-connected DNN baseline (flat features)
import torch, torch.nn as nn, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
torch.manual_seed(0); np.random.seed(0)

class DNN(nn.Module):
    def __init__(self, n_in, widths=(128, 64, 32, 16, 8), drop=0.1):
        super().__init__()
        layers, prev = [], n_in
        for w in widths:
            layers += [nn.Linear(prev, w), nn.ReLU(), nn.Dropout(drop)]; prev = w
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

def dnn_walk_forward(max_epochs=100, lr=1e-3, patience=10, val_frac=0.15):
    """Walk-forward DNN with early stopping. Validation = last val_frac of each
    training fold (inside train, never touches the test block)."""
    preds, actual = [], []
    for tr, te in tscv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]; ytr, yte = yd.iloc[tr], yd.iloc[te]
        vcut = int(len(Xtr) * (1 - val_frac))
        Xtr_in, Xva_in = Xtr.iloc[:vcut], Xtr.iloc[vcut:]
        ytr_in, yva_in = ytr.iloc[:vcut], ytr.iloc[vcut:]
        sc = StandardScaler().fit(Xtr_in)                 # fit on inner-train only
        Xtr_s = sc.transform(Xtr_in).astype("float32")
        Xva_s = sc.transform(Xva_in).astype("float32")
        Xte_s = sc.transform(Xte).astype("float32")
        torch.manual_seed(0)
        m, _, _ = train_model(DNN(Xd.shape[1]),
                              Xtr_s, ytr_in.values.astype("float32"),
                              Xva_s, yva_in.values.astype("float32"),
                              epochs=max_epochs, lr=lr, patience=patience)
        m.eval()
        with torch.no_grad():
            yp = m(torch.tensor(Xte_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        preds.append(pd.Series(yp, index=yte.index)); actual.append(yte)
    return pd.concat(preds), pd.concat(actual)

dnn_pred, dnn_act = dnn_walk_forward()
dnn_rmse = float(np.sqrt(np.mean((dnn_act.values - dnn_pred.values) ** 2)))
res["DNN"] = {"rmse": dnn_rmse, "preds": dnn_pred, "actual": dnn_act}
_idx = dnn_act.index.intersection(base["actual"].index)
_stat, _pv = dm_test((dnn_act.loc[_idx] - dnn_pred.loc[_idx]).values,
                     (base["actual"].loc[_idx] - base["preds"].loc[_idx]).values)
print(f"DNN walk-forward RMSE = {dnn_rmse:.5f}  (RandomWalk = {base['rmse']:.5f})")
print(f"DM vs RW: stat={_stat:.3f}, p={_pv:.4f}")

#Before sweep

# D2 After Sweep

In [ ]:
# %% Cell D2 (REVISED): train + VALIDATION loss vs epochs — the overfitting evidence
# Plots BOTH curves for each neural model. Validation loss reaching its min then
# rising while training loss keeps falling = the overfitting signature.
import matplotlib.pyplot as plt
import numpy as np, torch

PURPLE, ORANGE = "#500778", "#E8890C"

# ---- sequence models (LSTM, TCN) use Xseq ----
split = int(len(Xseq) * 0.8)
sc = StandardScaler().fit(Xseq[:split].reshape(-1, n_feat))
Xseq_s = sc.transform(Xseq.reshape(-1, n_feat)).reshape(Xseq.shape)
Xtr_s, Xva_s = Xseq_s[:split], Xseq_s[split:]
ytr_s, yva_s = yseq[:split], yseq[split:]

# ---- DNN uses flat features Xd ----
splitd = int(len(Xd) * 0.8)
scd = StandardScaler().fit(Xd.iloc[:splitd])
Xd_s  = scd.transform(Xd).astype("float32")
Xtr_d, Xva_d = Xd_s[:splitd], Xd_s[splitd:]
ytr_d, yva_d = yd.values[:splitd].astype("float32"), yd.values[splitd:].astype("float32")

diag = [
    ("LSTM", lambda: LSTMReg(n_feat, hidden=32, layers=1), Xtr_s, ytr_s, Xva_s, yva_s),
    ("TCN",  lambda: TCN(n_feat, ch=32),                   Xtr_s, ytr_s, Xva_s, yva_s),
    ("DNN",  lambda: DNN(Xd.shape[1]),                     Xtr_d, ytr_d, Xva_d, yva_d),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (name, ctor, Xa, ya, Xb, yb) in zip(axes, diag):
    torch.manual_seed(0)
    _, tr, va = train_model(ctor(), Xa, ya, Xb, yb, epochs=100, patience=999)  # patience high: run full to SEE the curve
    ax.plot(tr, lw=1.6, color=PURPLE, label="training loss")
    ax.plot(va, lw=1.6, color=ORANGE, label="validation loss")
    j = int(np.argmin(va))
    ax.axvline(j, color="grey", ls="--", alpha=0.7)
    ax.annotate(f"val min (ep {j})", (j, va[j]), textcoords="offset points",
                xytext=(8, 10), fontsize=9, color="grey")
    ax.set_title(f"{name}: training vs validation loss")
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss"); ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/neural_train_val_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# numeric overfitting signature
for name, ctor, Xa, ya, Xb, yb in diag:
    torch.manual_seed(0)
    _, tr, va = train_model(ctor(), Xa, ya, Xb, yb, epochs=100, patience=999)
    j = int(np.argmin(va))
    print(f"{name:5s}: val-min epoch {j:3d} | final train {tr[-1]:.5f} vs final val {va[-1]:.5f} "
          f"(gap {va[-1]-tr[-1]:+.5f}) -> val rises after ep {j} = overfitting")

In [ ]:
# %% Cell D3: Walk-forward evaluation of LSTM and TCN + DM tests vs RW
# Same expanding-window scheme as the classical models, so results are comparable.
# %% Cell D3 (REVISED): Walk-forward eval of LSTM/TCN with early stopping
def seq_walk_forward(model_builder, max_epochs=100, patience=10, val_frac=0.15):
    preds_all, actual_all, dates_all = [], [], []
    for tr, te in tscv.split(Xseq):
        # carve last val_frac of THIS fold's train as early-stopping validation
        vcut = int(len(tr) * (1 - val_frac))
        tr_in, va_in = tr[:vcut], tr[vcut:]
        scf = StandardScaler().fit(Xseq[tr_in].reshape(-1, n_feat))
        Xtr_s = scf.transform(Xseq[tr_in].reshape(-1, n_feat)).reshape(Xseq[tr_in].shape)
        Xva_s = scf.transform(Xseq[va_in].reshape(-1, n_feat)).reshape(Xseq[va_in].shape)
        Xte_s = scf.transform(Xseq[te].reshape(-1, n_feat)).reshape(Xseq[te].shape)
        model, _, _ = train_model(model_builder(),
                                  Xtr_s, yseq[tr_in],
                                  Xva_s, yseq[va_in],
                                  epochs=max_epochs, patience=patience)
        yp = predict(model, Xte_s)
        preds_all.append(yp); actual_all.append(yseq[te]); dates_all.append(dseq[te])
    preds = np.concatenate(preds_all); actual = np.concatenate(actual_all)
    dates = pd.DatetimeIndex(np.concatenate([d.values for d in dates_all]))
    return pd.Series(preds, index=dates), pd.Series(actual, index=dates)

print("Training LSTM across folds (early stopping)...")
lstm_pred, lstm_act = seq_walk_forward(lambda: LSTMReg(n_feat, hidden=64, layers=1))
print("Training TCN across folds (early stopping)...")
tcn_pred, tcn_act = seq_walk_forward(lambda: TCN(n_feat, ch=16))


def eval_vs_rw(name, pred, act):
    rmse = np.sqrt(np.mean((act.values - pred.values)**2))
    idx = act.index.intersection(base["actual"].index)
    e_m = (act.loc[idx] - pred.loc[idx]).values
    e_b = (base["actual"].loc[idx] - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_m, e_b)
    verdict = "ties RW" if pv >= 0.05 else ("beats RW" if stat < 0 else "WORSE than RW")
    return {"Model": name, "RMSE": round(rmse, 5), "DM stat": round(stat, 3),
            "p-value": round(pv, 4), "vs RW": verdict}

seq_rows = [eval_vs_rw("LSTM", lstm_pred, lstm_act),
            eval_vs_rw("TCN",  tcn_pred,  tcn_act)]
seq_df = pd.DataFrame(seq_rows)
print("\n=== Sequence model results (walk-forward) ===")
print(seq_df.to_string(index=False))
print(f"(Random walk RMSE = {base['rmse']:.5f})")

# fold into the master results dict for the combined final table
res["LSTM"] = {"rmse": seq_rows[0]["RMSE"], "preds": lstm_pred, "actual": lstm_act}
res["TCN"]  = {"rmse": seq_rows[1]["RMSE"], "preds": tcn_pred,  "actual": tcn_act}
seq_df.to_csv(f"{OUT_DIR}/sq1_sequence_models.csv", index=False)

#Validate DNN stability

In [ ]:
# %% DNN stability check — is the "beats RW" result robust across seeds?
import numpy as np, torch

def dnn_walk_forward_seed(seed, max_epochs=100, lr=1e-3, patience=10, val_frac=0.15):
    preds, actual = [], []
    for tr, te in tscv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]; ytr, yte = yd.iloc[tr], yd.iloc[te]
        vcut = int(len(Xtr) * (1 - val_frac))
        Xtr_in, Xva_in = Xtr.iloc[:vcut], Xtr.iloc[vcut:]
        ytr_in, yva_in = ytr.iloc[:vcut], ytr.iloc[vcut:]
        sc = StandardScaler().fit(Xtr_in)
        Xtr_s = sc.transform(Xtr_in).astype("float32")
        Xva_s = sc.transform(Xva_in).astype("float32")
        Xte_s = sc.transform(Xte).astype("float32")
        torch.manual_seed(seed); np.random.seed(seed)
        m, _, _ = train_model(DNN(Xd.shape[1]),
                              Xtr_s, ytr_in.values.astype("float32"),
                              Xva_s, yva_in.values.astype("float32"),
                              epochs=max_epochs, lr=lr, patience=patience)
        m.eval()
        with torch.no_grad():
            yp = m(torch.tensor(Xte_s, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        preds.append(pd.Series(yp, index=yte.index)); actual.append(yte)
    p, a = pd.concat(preds), pd.concat(actual)
    rmse = float(np.sqrt(np.mean((a.values - p.values) ** 2)))
    idx = a.index.intersection(base["actual"].index)
    stat, pv = dm_test((a.loc[idx] - p.loc[idx]).values,
                       (base["actual"].loc[idx] - base["preds"].loc[idx]).values)
    return rmse, stat, pv

print("DNN across seeds (RW RMSE = %.5f):" % base["rmse"])
rmses = []
for s in [0, 1, 42, 123, 2024]:
    r, st, pv = dnn_walk_forward_seed(s)
    verdict = "beats RW" if (pv < 0.05 and st < 0) else ("WORSE" if (pv < 0.05 and st > 0) else "ties RW")
    rmses.append(r)
    print(f"  seed {s:5d}: RMSE {r:.5f}  DM {st:+.3f}  p {pv:.4f}  -> {verdict}")
print(f"\nMean RMSE {np.mean(rmses):.5f} | std {np.std(rmses):.5f} | "
      f"min {min(rmses):.5f} | max {max(rmses):.5f}")

## Overfitting / generalisation — train vs val vs test (Task D)
For the deep models (DNN, LSTM, TCN) a single chronological 60/20/20 split is trained while tracking validation RMSE each epoch. Reports **best epoch** (min val), train/val/test RMSE, and the **generalisation gap** (test − train). A large positive gap = overfitting at this sample size. Saves `sq2_overfitting_table.csv`.

In [ ]:
# %% Task D: train/val/test RMSE + best epoch for the deep models
import torch, torch.nn as nn, numpy as np, pandas as pd, copy
from sklearn.preprocessing import StandardScaler

def _chrono(n, f_tr=0.6, f_va=0.2):
    a, b = int(n * f_tr), int(n * (f_tr + f_va))
    return slice(0, a), slice(a, b), slice(b, n)

def _train_track(model, Xtr, ytr, Xva, yva, Xte, yte, epochs=100, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4); lossf = nn.MSELoss()
    Xtr, ytr = torch.tensor(Xtr), torch.tensor(ytr); Xva, yva = torch.tensor(Xva), torch.tensor(yva)
    Xte, yte = torch.tensor(Xte), torch.tensor(yte)
    best_va, best_ep, best_state = np.inf, 0, None
    for ep in range(epochs):
        model.train(); opt.zero_grad(); lossf(model(Xtr), ytr).backward(); opt.step()
        model.eval()
        with torch.no_grad(): va = float(torch.sqrt(lossf(model(Xva), yva)))
        if va < best_va: best_va, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); model.eval()
    rmse = lambda X, y: float(torch.sqrt(lossf(model(X), y)))
    with torch.no_grad():
        return {"best_epoch": best_ep, "train_RMSE": round(rmse(Xtr, ytr), 5),
                "val_RMSE": round(rmse(Xva, yva), 5), "test_RMSE": round(rmse(Xte, yte), 5)}

rows = []
# DNN (flat features)
_Xf = Xd.values.astype("float32"); _yf = yd.values.astype("float32")
s_tr, s_va, s_te = _chrono(len(_Xf))
_sc = StandardScaler().fit(_Xf[s_tr]); _Xf_s = _sc.transform(_Xf).astype("float32")
torch.manual_seed(0)
r = _train_track(DNN(Xd.shape[1]), _Xf_s[s_tr], _yf[s_tr], _Xf_s[s_va], _yf[s_va], _Xf_s[s_te], _yf[s_te])
r["Model"] = "DNN"; rows.append(r)
# LSTM & TCN (sequences from D1: Xseq/yseq/n_feat)
s_tr, s_va, s_te = _chrono(len(Xseq))
_scf = StandardScaler().fit(Xseq[s_tr].reshape(-1, n_feat))
_Xseq_s = _scf.transform(Xseq.reshape(-1, n_feat)).reshape(Xseq.shape).astype("float32")
_ys = yseq.astype("float32")
for nm, fac in [("LSTM", lambda: LSTMReg(n_feat, hidden=64)), ("TCN", lambda: TCN(n_feat, ch=16))]:
    torch.manual_seed(0)
    r = _train_track(fac(), _Xseq_s[s_tr], _ys[s_tr], _Xseq_s[s_va], _ys[s_va], _Xseq_s[s_te], _ys[s_te])
    r["Model"] = nm; rows.append(r)

overfit = pd.DataFrame(rows)[["Model", "best_epoch", "train_RMSE", "val_RMSE", "test_RMSE"]]
overfit["gen_gap(test-train)"] = (overfit["test_RMSE"] - overfit["train_RMSE"]).round(5)
try: display(overfit)
except NameError: print(overfit.to_string(index=False))
overfit.to_csv(f"{OUT_DIR}/sq2_overfitting_table.csv", index=False)
print("A large positive gen_gap = overfitting at this sample size (a data-volume statement, not an architecture verdict).")


# E1 overfit table:

In [ ]:
# %% Task E1: Fair neural tuning (leakage-safe, early holdout)
# Coarse sweep of neural hyperparameters on the pre-evaluation region, so DL and ML
# are tuned on equal footing. Answers "demonstrate fair tuning across ML/DL models."
import numpy as np, torch

N_TEST_DAYS = 10 * 60
eval_start  = len(Xseq) - N_TEST_DAYS
Xtune, ytune = Xseq[:eval_start], yseq[:eval_start]
vcut = int(len(Xtune) * 0.85)
scT = StandardScaler().fit(Xtune[:vcut].reshape(-1, n_feat))
Xtune_s = scT.transform(Xtune.reshape(-1, n_feat)).reshape(Xtune.shape)
XtA, XvA = Xtune_s[:vcut], Xtune_s[vcut:]
ytA, yvA = ytune[:vcut], ytune[vcut:]

def sweep(name, ctors):
    rows = []
    for label, ctor in ctors:
        torch.manual_seed(0)
        m, _, va = train_model(ctor(), XtA, ytA, XvA, yvA, epochs=100, patience=10)
        rows.append((label, min(va) if va else np.nan))
    best = min(rows, key=lambda r: r[1])
    print(f"{name}: " + " | ".join(f"{l}={v:.5f}" for l, v in rows) + f"  -> best: {best[0]}")
    return best[0]

print("Neural hyperparameter sweep (min val MSE):")
best_lstm_h = sweep("LSTM hidden",  [(h, (lambda h=h: LSTMReg(n_feat, hidden=h, layers=1))) for h in (16, 32, 64)])
best_tcn_c  = sweep("TCN channels", [(c, (lambda c=c: TCN(n_feat, ch=c))) for c in (16, 32, 64)])
print(f"\nSelected: LSTM hidden={best_lstm_h}, TCN channels={best_tcn_c}, lookback={LOOKBACK} (fixed)")

In [ ]:
# %% Task E2 (3-SEED AVERAGED): Sample-size learning curves — data-volume evidence
# Each point averaged over 3 seeds to remove single-run noise. Shaded band = ±1 std.
import numpy as np, torch, matplotlib.pyplot as plt

fractions = [0.25, 0.50, 0.75, 1.0]
SEEDS = [0, 1, 42]

# sequence-model data (LSTM, TCN)
sp = int(len(Xseq) * 0.8)
scL = StandardScaler().fit(Xseq[:sp].reshape(-1, n_feat))
Xseq_L = scL.transform(Xseq.reshape(-1, n_feat)).reshape(Xseq.shape)
Xtr_L, Xte_L = Xseq_L[:sp], Xseq_L[sp:]; ytr_L, yte_L = yseq[:sp], yseq[sp:]

# flat-feature data (DNN, LinearSVR)
spd = int(len(Xd) * 0.8)
scLd = StandardScaler().fit(Xd.iloc[:spd])
Xd_L = scLd.transform(Xd).astype("float32")
Xtr_Ld, Xte_Ld = Xd_L[:spd], Xd_L[spd:]
ytr_Ld = yd.values[:spd].astype("float32"); yte_Ld = yd.values[spd:]

def rmse(yhat, y): return float(np.sqrt(np.mean((y - yhat) ** 2)))

def seq_point(ctor, n, seed):
    Xs, ys = Xtr_L[-n:], ytr_L[-n:]; vc = int(n * 0.85)
    torch.manual_seed(seed); np.random.seed(seed)
    m, _, _ = train_model(ctor(), Xs[:vc], ys[:vc], Xs[vc:], ys[vc:], epochs=100, patience=10)
    m.eval()
    with torch.no_grad():
        yp = m(torch.tensor(Xte_L, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return rmse(yp, yte_L)

def dnn_point(n, seed):
    Xs, ys = Xtr_Ld[-n:], ytr_Ld[-n:]; vc = int(n * 0.85)
    torch.manual_seed(seed); np.random.seed(seed)
    m, _, _ = train_model(DNN(Xd.shape[1]), Xs[:vc], ys[:vc], Xs[vc:], ys[vc:], epochs=100, patience=10)
    m.eval()
    with torch.no_grad():
        yp = m(torch.tensor(Xte_Ld, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return rmse(yp, yte_Ld)

def svr_point(n, seed):   # deterministic; seed ignored
    from sklearn.svm import LinearSVR
    m = LinearSVR(C=0.1, max_iter=10000).fit(Xtr_Ld[-n:], ytr_Ld[-n:])
    return rmse(m.predict(Xte_Ld), yte_Ld)

def averaged_curve(point_fn, sizes_from):
    means, stds = [], []
    for f in fractions:
        n = int(len(sizes_from) * f)
        vals = [point_fn(n, s) for s in SEEDS]
        means.append(np.mean(vals)); stds.append(np.std(vals))
    return np.array(means), np.array(stds)

print("Running 3-seed averaged learning curves (this takes a few minutes)...")
curves = {
    "LSTM": averaged_curve(lambda n, s: seq_point(lambda: LSTMReg(n_feat, hidden=64, layers=1), n, s), Xtr_L),
    "TCN":  averaged_curve(lambda n, s: seq_point(lambda: TCN(n_feat, ch=16), n, s), Xtr_L),
    "DNN":       averaged_curve(dnn_point, Xtr_Ld),
    "LinearSVR": averaged_curve(svr_point, Xtr_Ld),
}
sizes = [int(len(Xtr_L) * f) for f in fractions]

plt.figure(figsize=(8.5, 5.5))
colors = {"LSTM": "#C0392B", "TCN": "#E8890C", "DNN": "#2980B9", "LinearSVR": "#500778"}
for name, (mean, std) in curves.items():
    style = "--o" if name == "LinearSVR" else "-o"
    plt.plot(sizes, mean, style, lw=1.8, color=colors[name], label=name)
    plt.fill_between(sizes, mean - std, mean + std, color=colors[name], alpha=0.15)
plt.axhline(base["rmse"], color="grey", ls=":", lw=1.2, label="Random walk")
plt.xlabel("training-set size (observations)"); plt.ylabel("test RMSE")
plt.title("Sample-size learning curves (mean of 3 seeds, ±1 std)")
plt.legend(frameon=False); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/learning_curves_samplesize.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n(Random walk RMSE = {base['rmse']:.5f})")
for name, (mean, std) in curves.items():
    slope = mean[-1] - mean[-2]          # change over last segment
    trend = "still declining" if slope < -0.0005 else ("flattened" if abs(slope) <= 0.0005 else "rising")
    print(f"{name:10s}: mean RMSE {[round(v,4) for v in mean]}  (±{[round(v,4) for v in std]})  -> {trend}")

# MAE Scores

In [ ]:
# %% Pull RMSE + MAE for all models from current session (for Table 6.6)
import numpy as np, pandas as pd

def rmse_mae(pred, act):
    rmse = float(np.sqrt(np.mean((act.values - pred.values)**2)))
    mae  = float(np.mean(np.abs(act.values - pred.values)))
    return rmse, mae

def get(name):
    for d in ([results, res] if 'results' in dir() else [res]):
        if name in d and "preds" in d[name]:
            return d[name]["preds"], d[name]["actual"]
    return None, None

order = ["LinearSVR","ElasticNet","XGBoost","DNN","RandomForest",
         "RandomWalk","MLP","TCN","LSTM"]
rows = []
for k in order:
    p, a = get(k)
    if p is None:
        print(f"  ⚠ {k} not found"); continue
    r, m = rmse_mae(p, a)
    rows.append({"Model": k, "RMSE": round(r,5), "MAE": round(m,5)})
print(pd.DataFrame(rows).to_string(index=False))

#Multi-horizon

In [ ]:
# %% Multi-horizon skill decay (distinct colors + labeled benchmark)
import os, urllib.request
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
import numpy as np, pandas as pd

FONT_DIR = "/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn,url in {"Lato-Regular.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
               "Lato-Bold.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf"}.items():
    fp=os.path.join(FONT_DIR,fn)
    if not os.path.exists(fp): urllib.request.urlretrieve(url,fp)
    fm.fontManager.addfont(fp)
LATO = fm.FontProperties(fname=os.path.join(FONT_DIR,"Lato-Regular.ttf")).get_name()

mh = pd.read_csv(f"{OUT_DIR}/sq1_multihorizon.csv")
rw = mh[mh.Model=="RandomWalk"].set_index("Horizon")["RMSE"]
mh["ratio"] = mh.apply(lambda r: r["RMSE"]/rw[r["Horizon"]], axis=1)

plt.rcParams.update({"font.family":LATO,"axes.spines.top":False,"axes.spines.right":False})
# Distinct color + distinct marker per model (robust even in grayscale)
styles = {
 "LinearSVR":   {"color":"#500778","marker":"o"},   # deep purple, circle
 "ElasticNet":  {"color":"#E8890C","marker":"s"},   # orange, square
 "XGBoost":     {"color":"#1B9E77","marker":"^"},   # teal-green, triangle
 "RandomForest":{"color":"#C0392B","marker":"D"},   # red, diamond
}
GREEN="#2E7D46"; GREY="#8E8E8E"

fig, ax = plt.subplots(figsize=(10, 6))
xmax = mh["Horizon"].max()
ax.fill_between([0, xmax+2], [1,1], [0.4,0.4], color="#eef6f0", alpha=0.7, zorder=0)  # skill zone
for m, st in styles.items():
    d = mh[mh.Model==m].sort_values("Horizon")
    ax.plot(d["Horizon"], d["ratio"], marker=st["marker"], ms=7, lw=2.3,
            color=st["color"], label=m, zorder=3, markeredgecolor="white", markeredgewidth=0.8)

# Random walk = explicit labeled benchmark (flat at 1.0 because everything is normalized by it)
ax.axhline(1.0, color="#333333", ls="--", lw=1.6, zorder=2)
ax.text(xmax+1.5, 1.0, "Random Walk\n(benchmark)", color="#333", fontsize=9, va="center", ha="left", fontweight="bold")
ax.text(xmax*0.5, 0.55, "models beat random walk", color=GREEN, fontsize=10, ha="center", fontweight="bold")

ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("RMSE relative to random walk")
ax.set_xlim(0, xmax+8); ax.set_ylim(0.4, 3.0)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=2, loc="upper left", fontsize=10.5)
ax.text(0, 3.28, "Forecast skill vanishes within days", fontsize=15, fontweight="bold", color="#1a1a1a")
ax.text(0, 3.12, "Only at h=1 do models beat the benchmark; skill is gone by day 5.", fontsize=10.5, color=GREY)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_horizon_decay.png", dpi=300, bbox_inches="tight")
plt.show()

#Regime split

In [ ]:
# %% Cell F1: Regime-split SQ1 - do the models hold up UNDER geopolitical stress?
# Splits each model's existing walk-forward predictions (h=1) by market regime,
# scoring RMSE and DM-vs-RW separately for calm vs each crisis window.
import numpy as np, pandas as pd
from scipy.stats import norm

def regime_labels(dates):
    reg = pd.Series("calm", index=dates)
    for nm, (s_, e_) in EVENTS.items():
        e_ = pd.Timestamp(e_) if e_ else dates.max()
        reg[(dates >= pd.Timestamp(s_)) & (dates <= e_)] = nm
    return reg

def dm_test(e_a, e_b):
    if len(e_a) < 10: return np.nan, np.nan       # too few obs for a stable test
    d = e_a**2 - e_b**2
    stat = d.mean() / np.sqrt(np.var(d, ddof=1) / len(d))
    return stat, 2 * (1 - norm.cdf(abs(stat)))

base = res["RandomWalk"]
rows = []
for regime in ["calm", "ukraine", "redsea", "iran", "all"]:
    for name in res:
        if name == "RandomWalk": continue
        act = res[name]["actual"]; pred = res[name]["preds"]
        idx = act.index.intersection(base["actual"].index)
        labs = regime_labels(idx)
        mask = (labs == regime).values if regime != "all" else np.ones(len(idx), bool)
        if mask.sum() < 10:
            continue
        sub = idx[mask]
        e_m = (act.loc[sub] - pred.loc[sub]).values
        e_b = (base["actual"].loc[sub] - base["preds"].loc[sub]).values
        rmse_m = np.sqrt(np.mean(e_m**2)); rmse_rw = np.sqrt(np.mean(e_b**2))
        stat, pv = dm_test(e_m, e_b)
        verdict = ("n/a" if np.isnan(pv) else
                   "ties RW" if pv >= 0.05 else ("beats RW" if stat < 0 else "WORSE"))
        rows.append({"Regime": regime, "Model": name, "N days": int(mask.sum()),
                     "RMSE": round(rmse_m, 5), "RW RMSE": round(rmse_rw, 5),
                     "DM": round(stat, 2) if not np.isnan(stat) else "-",
                     "p": round(pv, 4) if not np.isnan(pv) else "-", "vs RW": verdict})

reg_df = pd.DataFrame(rows)
print("=== SQ1 performance by market regime (h=1) ===")
print(reg_df.to_string(index=False))
reg_df.to_csv(f"{OUT_DIR}/sq1_by_regime.csv", index=False)

## Regime-error ANOVA — is skill concentration statistically real? (Task S1b)
One-way ANOVA on the best model's per-day squared forecast errors, grouped by market regime (calm vs Red Sea vs Iran). Confirms statistically (not just visually) that errors — and thus forecast difficulty/skill — differ across regimes. Saves `regime_error_anova.csv`.

In [ ]:
# %% Task S1b: regime-error ANOVA on the best model
from scipy import stats
import numpy as np, pandas as pd

BEST = "LinearSVR" if "LinearSVR" in res else min(res, key=lambda k: res[k]["rmse"])
_act, _pred = res[BEST]["actual"], res[BEST]["preds"]
_err2 = ((_act - _pred) ** 2)
_labs = regime_labels(_err2.index)                     # from Cell F1
_groups = {r: _err2[_labs == r].values for r in ["calm", "ukraine", "redsea", "iran"]
           if (_labs == r).sum() >= 10}
print(f"Best model: {BEST} | regime groups: {[(k, len(v)) for k, v in _groups.items()]}")
F, pv = stats.f_oneway(*_groups.values())
print(f"Regime-error ANOVA: F = {F:.3f}, p = {pv:.3g} -> "
      + ("errors DIFFER across regimes (skill concentration is real)" if pv < 0.05 else "no significant difference"))

out = pd.DataFrame({"regime": list(_groups), "n_days": [len(v) for v in _groups.values()],
                    "mean_sq_err": [round(float(v.mean()), 7) for v in _groups.values()],
                    "rmse": [round(float(np.sqrt(v.mean())), 5) for v in _groups.values()]})
out = pd.concat([out, pd.DataFrame([{"regime": "ANOVA_F", "n_days": int(sum(len(v) for v in _groups.values())),
                                     "mean_sq_err": round(float(F), 4), "rmse": round(float(pv), 6)}])],
                ignore_index=True)
try: display(out)
except NameError: print(out.to_string(index=False))
out.to_csv(f"{OUT_DIR}/regime_error_anova.csv", index=False)


In [ ]:
# %% Cell F2: Regime-split visual - ALL models (capped y-axis, outliers labeled)
# Shows every model for consistency with the main comparison table. The y-axis is
# capped so the competitive models stay readable; the few models exceeding the cap
# are annotated with their value so nothing is hidden.
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, os, urllib.request
import numpy as np, pandas as pd

FONT_DIR="/content/fonts"; os.makedirs(FONT_DIR,exist_ok=True)
for fn,url in {"Lato-Regular.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
               "Lato-Bold.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf"}.items():
    fp=os.path.join(FONT_DIR,fn)
    if not os.path.exists(fp): urllib.request.urlretrieve(url,fp)
    fm.fontManager.addfont(fp)
LATO=fm.FontProperties(fname=os.path.join(FONT_DIR,"Lato-Regular.ttf")).get_name()

regimes = ["calm","redsea","iran"]
reg_labels = {"calm":"Calm","redsea":"Red Sea","iran":"Iran / Hormuz"}
# order models best->worst so the legend/colors read logically
model_order = ["LinearSVR","ElasticNet","XGBoost","RandomForest","DNN","TCN","LSTM","MLP"]

piv = reg_df[reg_df["Regime"].isin(regimes)].copy()
piv["ratio"] = piv["RMSE"] / piv["RW RMSE"]
models = [m for m in model_order if m in piv["Model"].unique()]

plt.rcParams.update({"font.family":LATO,"axes.spines.top":False,"axes.spines.right":False})
colors = {"LinearSVR":"#500778","ElasticNet":"#8E44AD","XGBoost":"#1B9E77","RandomForest":"#66BB6A",
          "DNN":"#B0A0C8","TCN":"#E8890C","LSTM":"#C0392B","MLP":"#8B2020"}
RED="#c0392b"; GREY="#8E8E8E"
YCAP = 2.1

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(regimes)); w = 0.8/len(models)
for i, m in enumerate(models):
    sub = piv[piv["Model"]==m].set_index("Regime").reindex(regimes)
    ratios = sub["ratio"].values
    ax.bar(x + i*w, np.minimum(ratios, YCAP), w, label=m, color=colors.get(m,GREY), zorder=3)
    # annotate any bar that exceeds the cap
    for xi, rt in zip(x + i*w, ratios):
        if rt > YCAP:
            ax.text(xi, YCAP+0.02, f"{rt:.1f}", ha="center", va="bottom", fontsize=7,
                    color=colors.get(m,GREY), rotation=90, fontweight="bold")

ax.axhline(1.0, color=RED, ls="--", lw=1.5, zorder=4)
ax.text(len(regimes)-0.4, 1.03, "random walk", color=RED, fontsize=10, ha="right", fontweight="bold")
ax.set_ylim(0, YCAP)
ax.set_xticks(x + w*(len(models)-1)/2)
ax.set_xticklabels([reg_labels[r] for r in regimes], fontsize=12)
ax.set_ylabel("RMSE relative to random walk")
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=len(models), loc="upper center", bbox_to_anchor=(0.5,-0.09), fontsize=9)
ax.text(0, YCAP*1.10, "Forecast skill by market regime (all models)",
        fontsize=15, fontweight="bold", color="#1a1a1a")
ax.text(0, YCAP*1.045, "Bars below the line beat the random walk. Linear models win in every regime; deep models fail throughout.",
        fontsize=10, color=GREY)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq1_regime_skill.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
print(pd.read_csv(f"{OUT_DIR}/table1_descriptives.csv").to_string())

In [ ]:
# %% FIGURE 6.4 - model comparison (RMSE, colour-coded) - live from results
import matplotlib.pyplot as plt
import numpy as np

RW = base["rmse"]

# check BOTH dicts (classical models in `results`, neural in `res`)
def get_rmse(key):
    for d in ([results, res] if 'results' in dir() else [res]):
        if key in d and "rmse" in d[key]:
            return d[key]["rmse"]
    return None

roster = {"LinearSVR":"LinearSVR","ElasticNet":"ElasticNet","DNN":"DNN",
          "XGBoost":"XGBoost","Random forest":"RandomForest","MLP":"MLP",
          "ARIMA":"ARIMA","Random walk":"RandomWalk","TCN":"TCN","LSTM":"LSTM"}

pairs = []
for disp, key in roster.items():
    r = get_rmse(key)
    if r is not None: pairs.append((disp, r))
    else: print(f"  ⚠ {disp} ({key}) missing from results — skipped")

pairs.sort(key=lambda x: x[1])                 # ascending RMSE
labels = [p[0] for p in pairs][::-1]           # best at top of barh
values = [p[1] for p in pairs][::-1]

def color(name, r):
    if name == "Random walk": return "#555555"
    return "#1E7A34" if r < RW*0.98 else ("#B22222" if r > RW*1.02 else "#F9A825")
colors = [color(n, r) for n, r in zip(labels, values)]

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(labels, values, color=colors, edgecolor="white")
ax.axvline(RW, ls="--", color="k", lw=1)
ax.text(RW, len(labels)-0.3, "  Random Walk", fontsize=9, va="top")
ax.set_xlabel("Walk-forward RMSE (log returns)")
ax.set_title("One-day-ahead forecast accuracy by model\n"
             "(green = beats RW, amber = ties, red = worse)")
for i, v in enumerate(values):
    ax.text(v + max(values)*0.008, i, f"{v:.4f}", va="center", fontsize=8)
ax.set_xlim(0, max(values) * 1.15)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig6_4_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"RW = {RW:.5f}")
for n, r in sorted(pairs, key=lambda x: x[1]):
    print(f"  {n:14s} {r:.5f}  ({'beats' if r<RW*0.995 else 'worse' if r>RW*1.005 else 'ties'})")

In [ ]:
# %% Table 6.6 data dump — all values, computed consistently (4 dp)
import numpy as np, pandas as pd

RW_RMSE = base["rmse"]
RW_MAE  = float(np.mean(np.abs(base["actual"].values - base["preds"].values)))

def metrics_vs_rw(pred, act):
    rmse = float(np.sqrt(np.mean((act.values - pred.values) ** 2)))
    mae  = float(np.mean(np.abs(act.values - pred.values)))
    idx  = act.index.intersection(base["actual"].index)
    e_m  = (act.loc[idx] - pred.loc[idx]).values
    e_b  = (base["actual"].loc[idx] - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_m, e_b)
    impr = 100 * (RW_RMSE - rmse) / RW_RMSE
    return rmse, mae, impr, stat, pv

def _get(k):
    for d in ([results, res] if 'results' in dir() else [res]):
        if k in d and "preds" in d[k]:
            return d[k]["preds"], d[k]["actual"]
    return None, None

model_keys = ["LinearSVR", "ElasticNet", "DNN", "XGBoost", "RandomForest",
              "TCN", "LSTM", "MLP"]

rows = []
for k in model_keys:
    pred, act = _get(k)
    if pred is None:
        print(f"  ⚠ {k} not found — skipping"); continue
    rmse, mae, impr, stat, pv = metrics_vs_rw(pred, act)
    verdict = "beats" if (pv < 0.05 and stat < 0) else ("worse" if (pv < 0.05 and stat > 0) else "ties")
    rows.append({"Model": k, "RMSE": round(rmse,4), "MAE": round(mae,4),
                 "Improve_%": round(impr,1), "DM": round(stat,2),
                 "p": round(pv,4), "verdict": verdict})

rows.append({"Model": "RandomWalk", "RMSE": round(RW_RMSE,4), "MAE": round(RW_MAE,4),
             "Improve_%": 0.0, "DM": np.nan, "p": np.nan, "verdict": "—"})

df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
print("=== TABLE 6.6 VALUES (sorted by RMSE, 4 dp) ===")
print(df.to_string(index=False))

# --- DNN 5-seed summary (for caption) ---
print("\n=== DNN 5-seed summary ===")
dnn_rmses = [dnn_walk_forward_seed(s)[0] for s in [0, 1, 42, 123, 2024]]
print(f"DNN seeds: {[round(r,4) for r in dnn_rmses]}")
print(f"DNN mean RMSE {np.mean(dnn_rmses):.4f} | std {np.std(dnn_rmses):.4f}")

# Figure 6.6 Table

In [ ]:
# %% Table 6.6 data dump — all values for the results table, computed consistently
import numpy as np, pandas as pd

RW_RMSE = base["rmse"]          # random walk RMSE (should be 0.03651)
RW_MAE  = float(np.mean(np.abs(base["actual"].values - base["preds"].values)))

def metrics_vs_rw(pred, act):
    """RMSE, MAE, improvement %, DM stat/p vs random walk — on the shared index."""
    rmse = float(np.sqrt(np.mean((act.values - pred.values) ** 2)))
    mae  = float(np.mean(np.abs(act.values - pred.values)))
    idx  = act.index.intersection(base["actual"].index)
    e_m  = (act.loc[idx] - pred.loc[idx]).values
    e_b  = (base["actual"].loc[idx] - base["preds"].loc[idx]).values
    stat, pv = dm_test(e_m, e_b)
    impr = 100 * (RW_RMSE - rmse) / RW_RMSE
    return rmse, mae, impr, stat, pv

# models to report (skip RW/ARIMA — handled separately)
model_keys = ["LinearSVR", "ElasticNet", "DNN", "XGBoost", "RandomForest",
              "TCN", "LSTM", "MLP"]

rows = []
for k in model_keys:
    if k not in res:
        print(f"  ⚠ {k} not in res — skipping"); continue
    pred, act = res[k]["preds"], res[k]["actual"]
    rmse, mae, impr, stat, pv = metrics_vs_rw(pred, act)
    verdict = "beats" if (pv < 0.05 and stat < 0) else ("worse" if (pv < 0.05 and stat > 0) else "ties")
    rows.append({"Model": k, "RMSE": round(rmse,5), "MAE": round(mae,5),
                 "Improve_%": round(impr,1), "DM": round(stat,2),
                 "p": round(pv,4), "verdict": verdict})

# add the benchmark row
rows.append({"Model": "RandomWalk", "RMSE": round(RW_RMSE,5), "MAE": round(RW_MAE,5),
             "Improve_%": 0.0, "DM": np.nan, "p": np.nan, "verdict": "—"})

df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
print("=== TABLE 6.6 VALUES (sorted by RMSE) ===")
print(df.to_string(index=False))

# --- DNN 5-seed summary (for the caption footnote) ---
print("\n=== DNN 5-seed summary (for caption) ===")
dnn_rmses = []
for s in [0, 1, 42, 123, 2024]:
    r, st, pv = dnn_walk_forward_seed(s)   # from your stability-check cell
    dnn_rmses.append(r)
print(f"DNN seeds: {[round(r,5) for r in dnn_rmses]}")
print(f"DNN mean RMSE {np.mean(dnn_rmses):.5f} | std {np.std(dnn_rmses):.5f} | "
      f"all beat RW: {all(dnn_walk_forward_seed(s)[2] < 0.05 and dnn_walk_forward_seed(s)[1] < 0 for s in [0,1,42])}")

# --- ARIMA (if you have it stored) ---
if "ARIMA" in res:
    a_pred, a_act = res["ARIMA"]["preds"], res["ARIMA"]["actual"]
    ar = metrics_vs_rw(a_pred, a_act)
    print(f"\nARIMA: RMSE {ar[0]:.5f}, MAE {ar[1]:.5f}, DM {ar[3]:.2f}, p {ar[4]:.4f}")
else:
    print("\nARIMA not in res — report its RMSE/p from wherever you computed it separately.")

In [ ]:
# %% Save/verify all figures to Drive + inventory
import os, glob, shutil

# Dedicated figures folder inside your Drive output dir
FIG_DIR = f"{OUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Copy any PNGs sitting in OUT_DIR into the figures subfolder
for png in glob.glob(f"{OUT_DIR}/*.png"):
    dest = os.path.join(FIG_DIR, os.path.basename(png))
    if os.path.abspath(png) != os.path.abspath(dest):
        shutil.copy(png, dest)

# Inventory: what's now on Drive
figs = sorted(glob.glob(f"{FIG_DIR}/*.png"))
print(f"{len(figs)} figures saved to: {FIG_DIR}\n")
for f in figs:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):40s} {size_kb:6.0f} KB")

# Flag the ones the Results chapter expects, so you can see what's still missing
expected = [
    "fig5_1_model_comparison.png", "sq1_multihorizon.png", "sq1_regime_skill.png",
    "fig5_4_breaks.png", "fig5_5_connectedness.png", "sq2_shap_global.png",
    "fig5_7_taxonomy.png", "hyperparam_curves.png", "fig_seq_training.png",
    "prices_events.png", "chokepoints.png",
]
have = {os.path.basename(f) for f in figs}
print("\nChapter figure checklist:")
for e in expected:
    print(f"  {'[OK]' if e in have else '[MISSING]'}  {e}")

#RMSE MAE MSE

In [ ]:
# %% Add MAE and MSE to the model comparison (run after your model res dict exists)
import numpy as np, pandas as pd
from scipy.stats import norm

def dm_test(e_a, e_b):
    d = e_a**2 - e_b**2
    stat = d.mean() / np.sqrt(np.var(d, ddof=1)/len(d))
    return stat, 2*(1-norm.cdf(abs(stat)))

base = res["RandomWalk"]
rows = []
for name in res:
    act, pred = res[name]["actual"], res[name]["preds"]
    err = (act - pred).values
    rmse = np.sqrt(np.mean(err**2)); mae = np.mean(np.abs(err)); mse = np.mean(err**2)
    if name == "RandomWalk":
        rows.append([name, round(rmse,5), round(mae,5), round(mse,6), "-", "-", "benchmark"]); continue
    idx = act.index.intersection(base["actual"].index)
    stat, pv = dm_test((act.loc[idx]-pred.loc[idx]).values,
                       (base["actual"].loc[idx]-base["preds"].loc[idx]).values)
    verdict = "ties RW" if pv>=0.05 else ("beats RW" if stat<0 else "worse")
    rows.append([name, round(rmse,5), round(mae,5), round(mse,6), round(stat,2), round(pv,4), verdict])
comp = pd.DataFrame(rows, columns=["Model","RMSE","MAE","MSE","DM","p","vs RW"]).sort_values("RMSE")
print(comp.to_string(index=False))
comp.to_csv(f"{OUT_DIR}/sq1_metrics_full.csv", index=False)

In [ ]:
# %% Regime figure (Figure 6.6) — grouped bars, regimes on x-axis
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

models = ["LinearSVR","ElasticNet","XGBoost","DNN","RandomForest","MLP"]
regimes = ["calm","redsea","iran"]
regime_label = {"calm":"Calm","redsea":"Red Sea","iran":"Hormuz"}
reg_df["ratio"] = reg_df["RMSE"] / reg_df["RW RMSE"]

def ratio(m, rg):
    r = reg_df[(reg_df.Model==m)&(reg_df.Regime==rg)]["ratio"].values
    return r[0] if len(r) else np.nan

fig, ax = plt.subplots(figsize=(12, 6))
mcols = ["#500778","#7B4BA8","#9B6FC4","#2980B9","#B5A3D4","#6B5B7B"]
x = np.arange(len(regimes)); w = 0.14
for i, m in enumerate(models):
    vals = [ratio(m, rg) for rg in regimes]
    ax.bar(x + i*w, vals, w, label=m, color=mcols[i], edgecolor="white")
ax.axhline(1.0, ls="--", lw=2, color="#C62828", zorder=5)
ax.text(len(regimes)-0.5, 1.02, "Random walk", color="#C62828",
        fontsize=11, fontweight="bold", ha="right")
ax.set_xticks(x + w*(len(models)-1)/2)
ax.set_xticklabels([regime_label[r] for r in regimes], fontsize=12)
ax.set_ylabel("RMSE relative to random walk")
ax.set_title("Forecast skill by regime and model\n(bars below the red line beat the random walk)")
ax.legend(title="Model", ncol=6, fontsize=9)
ax.set_ylim(0, max(reg_df["ratio"])*1.15)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig6_6_regime.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% Cell A8c: Feature inventory table (families mirror the ablation blocks)
import re
import pandas as pd

FEATURE_SOURCE = Xall            # curated set that feeds the models
# FEATURE_SOURCE = Xall_full     # swap in to document the full 123-feature set

HUB_NAMES = {"ttf":"TTF (Dutch)","jkm":"JKM (Asian LNG)","hh":"Henry Hub (US)",
             "nbp":"NBP (UK)","the":"THE (German)"}
OIL_NAMES = {"brent":"Brent crude","wti":"WTI crude","eua":"EUA carbon"}
CHK_NAMES = {"hormuz":"Strait of Hormuz","suez":"Suez Canal",
             "bab":"Bab-el-Mandeb","cape":"Cape of Good Hope"}
EVT_NAMES = {"ukraine":"Russia–Ukraine war","redsea":"Red Sea / Houthi crisis",
             "iran":"Iran / Hormuz tension"}

def _hub(t): return HUB_NAMES.get(t, t.upper())

def classify(col):
    c = col.lower()
    if c.startswith("spread_"):
        toks = [t for t in c.replace("spread_","").split("_") if t in HUB_NAMES]
        d = "Cross-hub price spread: " + " vs ".join(_hub(t) for t in toks) if toks else "Cross-hub price spread"
        return "Cross-hub spread", d, "Levels (arbitrage)"
    if c.endswith("_slope") or c.endswith("_curvature") or "_prompt_spread" in c:
        hub = next((t for t in HUB_NAMES if c.startswith(t)), None)
        shape = "slope" if c.endswith("_slope") else "curvature" if c.endswith("_curvature") else "prompt spread"
        return "Curve shape", f"Forward-curve {shape}" + (f": {_hub(hub)}" if hub else ""), "Levels"
    if c.endswith("_vol21"):
        hub = next((t for t in HUB_NAMES if c.startswith(t)), None)
        return "Price lag / vol", f"21-day realised volatility{': '+_hub(hub) if hub else ''}", "Return-derived"
    if re.search(r"_lag\d+$", c):
        lag = re.search(r"_lag(\d+)$", c).group(1)
        hub = next((t for t in HUB_NAMES if c.startswith(t)), None)
        oil = next((t for t in OIL_NAMES if c.startswith(t)), None)
        if oil: return "Oil / carbon", f"{OIL_NAMES[oil]} return, {lag}-day lag", "Return-derived"
        return "Price lag / vol", f"Lagged return ({lag}-day){': '+_hub(hub) if hub else ''}", "Return-derived"
    if "storage" in c:
        which = "anomaly vs seasonal norm" if "anom" in c else "change" if "chg" in c else "level"
        return "Storage", f"EU gas storage {which}", "Fundamental"
    if c.startswith("chk_"):
        ck = next((k for k in CHK_NAMES if k in c), None)
        return "Chokepoint", f"Shipping-pressure signal (7d MA): {CHK_NAMES.get(ck, ck)}", "Fundamental"
    if "gpr" in c:
        sub = "acts sub-index" if c.endswith("_act") else "threats sub-index" if c.endswith("_threat") else "headline index"
        return "Geopolitical (GPR)", f"Geopolitical Risk Index ({sub})", "Level"
    if c.startswith("evt_"):
        return "Regime dummy", f"Event regime dummy: {EVT_NAMES.get(c.replace('evt_',''), c.replace('evt_',''))}", "Binary"
    if c == "heating":
        return "Seasonal / weather", "Heating-demand proxy", "Fundamental"
    if c.startswith("doy_"):
        return "Seasonal / weather", f"Day-of-year cyclical ({'sine' if c.endswith('sin') else 'cosine'})", "Engineered"
    return "Other", "(uncategorised — check naming)", ""

rows = []
for col in FEATURE_SOURCE.columns:
    fam, desc, form = classify(col)
    rows.append({"Feature": col, "Family": fam, "Description": desc, "Form": form})

order = ["Cross-hub spread","Curve shape","Price lag / vol","Storage","Chokepoint",
         "Oil / carbon","Geopolitical (GPR)","Regime dummy","Seasonal / weather","Other"]
feat_tbl = pd.DataFrame(rows)
feat_tbl["Family"] = pd.Categorical(feat_tbl["Family"], categories=order, ordered=True)
feat_tbl = feat_tbl.sort_values(["Family","Feature"]).reset_index(drop=True)

print(f"Feature inventory — {len(feat_tbl)} features across "
      f"{feat_tbl['Family'].nunique()} families\n")
print(feat_tbl.to_string(index=False))
print("\n--- count per family ---")
print(feat_tbl.groupby("Family", observed=True).size().to_string())

# flag anything the classifier missed, so a renamed feature never slips through
unc = feat_tbl[feat_tbl["Family"] == "Other"]
if len(unc):
    print("\n⚠ Uncategorised — add a rule or fix the name:", list(unc["Feature"]))

# export for the data chapter
feat_tbl.to_csv(f"{OUT_DIR}/table_feature_inventory.csv", index=False)
with open(f"{OUT_DIR}/table_feature_inventory.tex", "w") as f:
    f.write(feat_tbl.to_latex(index=False, escape=True, longtable=True,
            caption="Feature inventory by family.", label="tab:features"))
print(f"\nSaved to {OUT_DIR}/table_feature_inventory.csv (+ .tex)")

In [ ]:
# %% Standalone: LinearSVR feature importance (best-model coefficients)
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd

# needs: Xall (feature matrix), the target y, feature_cols
TARGET = "ttf_ret"                      # adjust to your actual target column
# NEW — use the target you already built
y = yd.reindex(Xall.index)          # yd is your ttf_ret target from earlier cells
mask = y.notna()
X, y = Xall[mask], y[mask]

# scale (linear model needs it) - fit on all data here is fine for IMPORTANCE
# (not forecasting, so no leakage concern for a descriptive coefficient ranking)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

svr = LinearSVR(C=1.0, max_iter=10000, random_state=42).fit(Xs, y)

coefs = pd.DataFrame({
    "feature": X.columns,
    "coef": svr.coef_,
    "abs_coef": np.abs(svr.coef_)
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("Top 15 LinearSVR features (best-model importance):")
print(coefs.head(15).to_string(index=False))
coefs.to_csv(f"{OUT_DIR}/sq1_linearsvr_coefs.csv", index=False)
print(f"\nSaved. Any GPR feature in top 10? ",
      coefs.head(10)["feature"].str.contains("gpr", case=False).any())

# Table 6.8 Across hubs

In [ ]:
# %% Multi-hub robustness: run the h=1 SQ2 benchmark for ALL five hubs
# Reuses walk_forward, Xall, yall, base/RW, dm_test, and the model fns from Cell A9 + tuned models.
# Produces one comparable table per hub, plus a combined summary.
import numpy as np, pandas as pd

HUB_TARGETS = [f"{h}_ret" for h in HUBS]          # ['ttf_ret','jkm_ret','hh_ret','nbp_ret','the_ret']
print("Targets:", HUB_TARGETS)

# The tuned model set (same as your headline 8-model run).
# If you have these builders already from the tuned cell, reuse them; otherwise these match `best`.
from sklearn.linear_model import ElasticNet
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def rw_fn(Xtr, ytr, Xte):        return np.zeros(len(Xte))
def en_fn(Xtr, ytr, Xte):
    m = ElasticNet(alpha=1e-6, l1_ratio=0.5, max_iter=5000); m.fit(Xtr, ytr); return m.predict(Xte)
def svr_fn(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr)
    m = LinearSVR(C=1, max_iter=20000); m.fit(sc.transform(Xtr), ytr)
    return m.predict(sc.transform(Xte))
def rf_fn(Xtr, ytr, Xte):
    m = RandomForestRegressor(n_estimators=300, max_depth=30, n_jobs=-1, random_state=0); m.fit(Xtr, ytr); return m.predict(Xte)
def xgb_fn(Xtr, ytr, Xte):
    m = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=8, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=3.0, random_state=0); m.fit(Xtr, ytr); return m.predict(Xte)

MODELS = [("RandomWalk", rw_fn), ("ElasticNet", en_fn), ("LinearSVR", svr_fn),
          ("RandomForest", rf_fn), ("XGBoost", xgb_fn)]

def dm_vs_rw(err_model, err_rw):
    """Diebold-Mariano: negative stat => model better than RW."""
    d = (err_model**2) - (err_rw**2)
    dbar = d.mean(); n = len(d)
    # HAC (lag-1) variance for h=1
    gamma0 = np.mean((d - dbar)**2)
    var = gamma0 / n
    stat = dbar / np.sqrt(var) if var > 0 else np.nan
    from scipy import stats as st
    p = 2 * (1 - st.norm.cdf(abs(stat))) if np.isfinite(stat) else np.nan
    return stat, p

all_rows = []
for tgt in HUB_TARGETS:
    hub = tgt.replace("_ret", "").upper()
    # RW errors first (benchmark for this hub)
    rw_err, rw_pred, rw_act = walk_forward(rw_fn, Xall, yall, tgt)
    rw_e = (rw_act.values - rw_pred.values)
    rw_rmse = np.sqrt(np.mean(rw_e**2))
    for name, fn in MODELS:
        errs, preds, actual = walk_forward(fn, Xall, yall, tgt)
        e = (actual.values - preds.values)
        rmse = np.sqrt(np.mean(e**2))
        if name == "RandomWalk":
            stat, p, verdict = np.nan, np.nan, "benchmark"
        else:
            # align on common index just in case
            stat, p = dm_vs_rw(e, rw_e)
            verdict = "beats RW" if (p < 0.05 and stat < 0) else ("WORSE" if p < 0.05 else "ties RW")
        all_rows.append({"Hub": hub, "Model": name, "RMSE": round(rmse, 5),
                         "RW RMSE": round(rw_rmse, 5), "DM": round(stat, 3) if np.isfinite(stat) else "-",
                         "p": round(p, 4) if np.isfinite(p) else "-", "vs RW": verdict})
    print(f"done: {hub}")

multi = pd.DataFrame(all_rows)
print("\n=== Multi-hub h=1 comparison ===")
print(multi.to_string(index=False))
multi.to_csv(f"{OUT_DIR}/sq2_multihub_h1.csv", index=False)
print("\nSaved sq2_multihub_h1.csv")

# Compact winner summary: best model per hub
print("\n=== Best model per hub (lowest RMSE, excluding RW) ===")
for hub in multi["Hub"].unique():
    sub = multi[(multi.Hub==hub) & (multi.Model!="RandomWalk")]
    best = sub.loc[sub.RMSE.idxmin()]
    print(f"  {hub}: {best.Model} (RMSE {best.RMSE}, {best['vs RW']})")

In [ ]:
# %% SVR-only re-run (scaled) — patches the broken LinearSVR rows in the multi-hub table
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from scipy import stats as st

HUB_TARGETS = [f"{h}_ret" for h in HUBS]   # ['ttf_ret','jkm_ret','hh_ret','nbp_ret','the_ret']

def svr_fn_scaled(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr)                     # fit on TRAIN only (leakage-safe)
    m = LinearSVR(C=1, max_iter=20000)
    m.fit(sc.transform(Xtr), ytr)
    return m.predict(sc.transform(Xte))

def rw_fn(Xtr, ytr, Xte):
    return np.zeros(len(Xte))

def dm_vs_rw(err_model, err_rw):
    d = (err_model**2) - (err_rw**2)
    dbar = d.mean(); n = len(d)
    var = np.mean((d - dbar)**2) / n
    stat = dbar / np.sqrt(var) if var > 0 else np.nan
    p = 2 * (1 - st.norm.cdf(abs(stat))) if np.isfinite(stat) else np.nan
    return stat, p

svr_rows = []
for tgt in HUB_TARGETS:
    hub = tgt.replace("_ret", "").upper()
    # RW baseline for this hub (recompute so DM is self-contained)
    _, rw_pred, rw_act = walk_forward(rw_fn, Xall, yall, tgt)
    rw_e = (rw_act.values - rw_pred.values)
    rw_rmse = np.sqrt(np.mean(rw_e**2))
    # scaled SVR
    _, preds, actual = walk_forward(svr_fn_scaled, Xall, yall, tgt)
    e = (actual.values - preds.values)
    rmse = np.sqrt(np.mean(e**2))
    stat, p = dm_vs_rw(e, rw_e)
    verdict = "beats RW" if (p < 0.05 and stat < 0) else ("WORSE" if p < 0.05 else "ties RW")
    svr_rows.append({"Hub": hub, "Model": "LinearSVR", "RMSE": round(rmse, 5),
                     "RW RMSE": round(rw_rmse, 5),
                     "DM": round(stat, 3) if np.isfinite(stat) else "-",
                     "p": round(p, 4) if np.isfinite(p) else "-", "vs RW": verdict})
    print(f"SVR done: {hub}  RMSE={rmse:.5f}  {verdict}")

# --- merge: drop old broken LinearSVR rows, insert the corrected ones ---
multi = pd.read_csv(f"{OUT_DIR}/sq2_multihub_h1.csv")
multi = multi[multi["Model"] != "LinearSVR"]                 # remove broken rows
multi = pd.concat([multi, pd.DataFrame(svr_rows)], ignore_index=True)

# tidy ordering: by hub, then a fixed model order
model_order = ["RandomWalk", "ElasticNet", "LinearSVR", "RandomForest", "XGBoost"]
multi["Model"] = pd.Categorical(multi["Model"], categories=model_order, ordered=True)
multi = multi.sort_values(["Hub", "Model"]).reset_index(drop=True)

print("\n=== Corrected multi-hub h=1 comparison ===")
print(multi.to_string(index=False))
multi.to_csv(f"{OUT_DIR}/sq2_multihub_h1.csv", index=False)
print("\nSaved corrected sq2_multihub_h1.csv")

# sanity check: TTF LinearSVR should now be ~0.024 and beat RW
ttf_svr = multi[(multi.Hub=="TTF") & (multi.Model=="LinearSVR")]
print("\nSanity check — TTF LinearSVR:")
print(ttf_svr.to_string(index=False))

In [ ]:
# %% FIGURE 2.1 — Front-month gas prices at the principal hubs (2016–2026)
# Publication quality: Lato font, distinct hub colours, 2022 spike annotated.
import os, urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import numpy as np, pandas as pd

# --- Lato font (matches your other figures) ---
FONT_DIR = "/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn, url in {
    "Lato-Regular.ttf": "https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
    "Lato-Bold.ttf":    "https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf",
}.items():
    fp = os.path.join(FONT_DIR, fn)
    if not os.path.exists(fp):
        try: urllib.request.urlretrieve(url, fp)
        except Exception: pass
    try: fm.fontManager.addfont(fp)
    except Exception: pass
try:
    LATO = fm.FontProperties(fname=f"{FONT_DIR}/Lato-Regular.ttf").get_name()
except Exception:
    LATO = "DejaVu Sans"
plt.rcParams.update({
    "font.family": LATO, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#555555", "axes.linewidth": 0.8,
    "figure.dpi": 120,
})

# --- distinct, colourblind-safe hub colours (UCL purple leads for TTF) ---
HUB_STYLE = {
    "ttf": ("#500778", "TTF (Europe)"),        # UCL purple
    "jkm": ("#E8890C", "JKM (Asia)"),          # amber
    "nbp": ("#1B9E77", "NBP (UK)"),            # teal-green
    "the": ("#C0392B", "THE (Germany)"),       # red
    "hh":  ("#2C7FB8", "Henry Hub (US)"),      # blue
}
EVENTS = {
    "ukraine": ("2022-02-24", "2023-10-31", "#C62828", "Ukraine"),
    "redsea":  ("2023-11-19", "2025-05-06", "#F9A825", "Red Sea"),
    "iran":    ("2026-02-28", None,          "#2c7fb8", "Hormuz"),
}

# --- data: use the modelling frame p (2016+) ---
df = p  # p already starts 2016 and has {hub}_c1_usd columns
fig, ax = plt.subplots(figsize=(12, 5.2))

# shaded event windows (behind the lines)
ymax_guess = max(df[f"{h}_c1_usd"].max() for h in HUB_STYLE if f"{h}_c1_usd" in df.columns)
for k, (s, e, col, lab) in EVENTS.items():
    e = e or str(df.index.max().date())
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=0.07, zorder=0)
    ax.text(pd.Timestamp(s), ymax_guess*0.995, f" {lab}", rotation=90,
            va="top", ha="left", fontsize=8.5, color=col, fontweight="bold", alpha=0.9)

# price lines — TTF drawn last so it sits on top
order = ["hh", "nbp", "the", "jkm", "ttf"]
for h in order:
    col = f"{h}_c1_usd"
    if col in df.columns:
        c, lab = HUB_STYLE[h]
        lw = 1.6 if h == "ttf" else 1.1
        a  = 1.0 if h == "ttf" else 0.9
        ax.plot(df.index, df[col], color=c, lw=lw, label=lab, alpha=a, zorder=3)

# annotate the 2022 TTF peak
ttf = df["ttf_c1_usd"]
pk_date, pk_val = ttf.idxmax(), ttf.max()
ax.annotate(f"TTF peak  ${pk_val:.0f}/MMBtu\n({pk_date.strftime('%b %Y')})",
            xy=(pk_date, pk_val), xytext=(pk_date + pd.Timedelta(days=250), pk_val*0.92),
            fontsize=9.5, color="#500778", fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="#500778", lw=1.3,
                            connectionstyle="arc3,rad=-0.15"))

# long-run average reference for TTF (pre-2021), to show the scale of the spike
pre = ttf.loc[:"2020-12-31"].mean()
ax.axhline(pre, color="#999999", ls=":", lw=1.0, zorder=1)
ax.text(df.index.min(), pre, f" pre-2021 TTF avg ≈ ${pre:.0f}",
        va="bottom", ha="left", fontsize=8, color="#777777", style="italic")

# axes cosmetics
ax.set_ylabel("Front-month price (USD/MMBtu)", fontsize=11)
ax.set_xlabel("")
ax.set_ylim(0, ymax_guess*1.06)
ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%d"))

# legend below, single row, no frame
leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.10), ncol=5,
                frameon=False, fontsize=9.5, handlelength=1.6, columnspacing=1.6)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_1_prices_events.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/fig2_1_prices_events.pdf", bbox_inches="tight")  # vector, for print
plt.show()
print("Saved fig2_1_prices_events.png and .pdf")

In [ ]:
# %% FIGURE 2.1 — Front-month gas prices at the principal hubs (2016–2026)
import os, urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import numpy as np, pandas as pd

# --- locate the price DataFrame (p got clobbered; try known fallbacks) ---
def _find_price_df():
    import pandas as pd
    for name in ["p", "prices", "price_df", "data", "df_prices", "panel", "raw"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and any(c.endswith("_c1_usd") for c in obj.columns):
            print(f"Using price DataFrame: '{name}' {obj.shape}")
            return obj
    raise NameError("No price DataFrame with '*_c1_usd' columns found. "
                    "Re-run your data-loading cell to rebuild it first.")

df = _find_price_df()

# --- Lato font ---
FONT_DIR = "/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn, url in {
    "Lato-Regular.ttf": "https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
    "Lato-Bold.ttf":    "https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf",
}.items():
    fp = os.path.join(FONT_DIR, fn)
    if not os.path.exists(fp):
        try: urllib.request.urlretrieve(url, fp)
        except Exception: pass
    try: fm.fontManager.addfont(fp)
    except Exception: pass
try:
    LATO = fm.FontProperties(fname=f"{FONT_DIR}/Lato-Regular.ttf").get_name()
except Exception:
    LATO = "DejaVu Sans"
plt.rcParams.update({
    "font.family": LATO, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#555555", "axes.linewidth": 0.8, "figure.dpi": 120,
})

HUB_STYLE = {
    "ttf": ("#500778", "TTF (Europe)"),
    "jkm": ("#E8890C", "JKM (Asia)"),
    "nbp": ("#1B9E77", "NBP (UK)"),
    "the": ("#C0392B", "THE (Germany)"),
    "hh":  ("#2C7FB8", "Henry Hub (US)"),
}
EVENTS = {
    "ukraine": ("2022-02-24", "2023-10-31", "#C62828", "Ukraine"),
    "redsea":  ("2023-11-19", "2025-05-06", "#F9A825", "Red Sea"),
    "iran":    ("2026-02-28", None,          "#2c7fb8", "Hormuz"),
}

fig, ax = plt.subplots(figsize=(12, 5.2))
ymax_guess = max(df[f"{h}_c1_usd"].max() for h in HUB_STYLE if f"{h}_c1_usd" in df.columns)

for k, (s, e, col, lab) in EVENTS.items():
    e = e or str(df.index.max().date())
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=0.07, zorder=0)
    ax.text(pd.Timestamp(s), ymax_guess*0.995, f" {lab}", rotation=90,
            va="top", ha="left", fontsize=8.5, color=col, fontweight="bold", alpha=0.9)

order = ["hh", "nbp", "the", "jkm", "ttf"]
for h in order:
    col = f"{h}_c1_usd"
    if col in df.columns:
        c, lab = HUB_STYLE[h]
        ax.plot(df.index, df[col], color=c, lw=1.6 if h=="ttf" else 1.1,
                label=lab, alpha=1.0 if h=="ttf" else 0.9, zorder=3)

ttf = df["ttf_c1_usd"]
pk_date, pk_val = ttf.idxmax(), ttf.max()
ax.annotate(f"TTF peak  ${pk_val:.0f}/MMBtu\n({pk_date.strftime('%b %Y')})",
            xy=(pk_date, pk_val), xytext=(pk_date + pd.Timedelta(days=250), pk_val*0.8),
            fontsize=9.5, color="#500778", fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="#500778", lw=1.3,
                            connectionstyle="arc3,rad=-0.15"))

ax.set_ylabel("Front-month price (USD/MMBtu)", fontsize=11)
ax.set_ylim(0, ymax_guess*1.06); ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%d"))

# --- title ---
ax.set_title("Front-Month Gas Prices at the Principal Hubs, 2016 to 2026",
             fontsize=13, fontweight="bold", color="#500778", pad=12)

# --- legend at the BOTTOM, with thicker lines ---
leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=5,
                frameon=False, fontsize=9.5, handlelength=2.2, columnspacing=1.6)
for line in leg.get_lines():
    line.set_linewidth(3.5)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_1_prices_events.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/fig2_1_prices_events.pdf", bbox_inches="tight")
plt.show()
print("Saved fig2_1_prices_events.png and .pdf")

In [ ]:
# %% FIGURE 2.2 (v2) — cropped to the strait region, clean legend
import subprocess, sys
HAVE_CARTOPY = True
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except Exception:
    try:
        subprocess.run([sys.executable,"-m","pip","install","-q","cartopy"], check=True)
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
    except Exception:
        HAVE_CARTOPY = False

import os, urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
import numpy as np

FONT_DIR="/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn,url in {"Lato-Regular.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
               "Lato-Bold.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf"}.items():
    fp=os.path.join(FONT_DIR,fn)
    if not os.path.exists(fp):
        try: urllib.request.urlretrieve(url,fp)
        except Exception: pass
    try: fm.fontManager.addfont(fp)
    except Exception: pass
try: LATO=fm.FontProperties(fname=f"{FONT_DIR}/Lato-Regular.ttf").get_name()
except Exception: LATO="DejaVu Sans"
plt.rcParams.update({"font.family":LATO})
PURPLE="#500778"; AMBER="#E8890C"

# name, lon, lat, LNG %, in_study, label_offset(dx,dy)
CHOKEPOINTS=[
    ("Strait of Hormuz",    56.5,  26.6, 20.0, True,  (0,-7)),
    ("Strait of Malacca",  100.4,   2.5, 17.0, False, (0,-7)),
    ("Suez Canal",          32.35, 31.2,  8.0, True,  (-11,4)),
    ("Bab el-Mandeb",       43.35, 12.6,  8.0, True,  (9,-6)),
    ("Panama Canal",       -79.7,   9.1,  4.0, False, (0,-7)),
    ("Cape of Good Hope",   19.9, -34.8,  5.0, True,  (0,-7)),
    ("Turkish Straits",     29.0,  41.0,  1.5, False, (-10,5)),
    ("Danish Straits",      11.0,  55.9,  1.0, False, (0,7)),
    ("Str. of Gibraltar",   -5.6,  35.95, 3.0, False, (-11,5)),
]
def bsize(pct): return (pct**0.5)*150

# crop window: cut empty Pacific/Atlantic. Straits span ~80W to ~101E.
LON_MIN, LON_MAX = -95, 118
LAT_MIN, LAT_MAX = -50, 68

if HAVE_CARTOPY:
    fig = plt.figure(figsize=(14, 6.6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#eeecf1", edgecolor="none")
    ax.add_feature(cfeature.OCEAN, facecolor="#fbfcfe")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#b3b3b3")
    ax.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="#dcdcdc")
    tf = ccrs.PlateCarree()

    for name,lon,lat,pct,instudy,(dx,dy) in CHOKEPOINTS:
        col  = PURPLE if instudy else AMBER
        edge = "#2c0a44" if instudy else "#9c5c00"
        ax.scatter(lon, lat, s=bsize(pct), color=col, alpha=0.62,
                   edgecolor=edge, linewidth=1.5 if instudy else 0.8, transform=tf, zorder=6)
        ax.annotate(f"{name}  {pct:.0f}%", xy=(lon,lat), xytext=(lon+dx, lat+dy),
                    transform=tf, ha="center", va="center", fontsize=8.3,
                    fontweight="bold" if instudy else "normal", color="#1a1a1a", zorder=7)

    ax.set_title("Global LNG maritime chokepoints", fontsize=14, fontweight="bold",
                 color=PURPLE, pad=10)


    # --- clean legend: two custom handle groups, white background, outside bottom ---
    size_handles = [Line2D([],[], marker='o', linestyle='None', markersize=np.sqrt(bsize(p))/2.2,
                           markerfacecolor="#b9b9b9", markeredgecolor="#666", alpha=0.6,
                           label=f"{p}% of LNG trade") for p in [20,8,3]]
    cat_handles = [
        Line2D([],[], marker='o', linestyle='None', markersize=10, markerfacecolor=PURPLE,
               markeredgecolor="#2c0a44", alpha=0.7, label="Analysed in this study"),
        Line2D([],[], marker='o', linestyle='None', markersize=10, markerfacecolor=AMBER,
               markeredgecolor="#9c5c00", alpha=0.7, label="Global context"),
    ]
    leg1 = ax.legend(handles=size_handles, title="Throughput", loc="lower left",
                     bbox_to_anchor=(0.005, 0.02), frameon=True, framealpha=0.95,
                     edgecolor="#dddddd", fontsize=8.5, title_fontsize=9,
                     labelspacing=1.3, borderpad=0.9)
    leg1._legend_box.align = "left"
    ax.add_artist(leg1)
    ax.legend(handles=cat_handles, loc="lower left", bbox_to_anchor=(0.2, 0.02),
              frameon=True, framealpha=0.95, edgecolor="#dddddd", fontsize=8.5,
              borderpad=0.9)
else:
    print("cartopy unavailable — run '!pip install cartopy' in its own cell, then re-run")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_2_lng_chokepoints.png", dpi=300, bbox_inches="tight")
try: plt.savefig(f"{OUT_DIR}/fig2_2_lng_chokepoints.pdf", bbox_inches="tight")
except Exception: pass
plt.show()
print("Saved fig2_2_lng_chokepoints.png")

In [ ]:
# %% FIGURE 4.1 — Front-month hub prices, ZOOMED to 2022–mid-2026
import os, urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import numpy as np, pandas as pd

# --- locate the price DataFrame (p got clobbered) ---
def _find_price_df():
    for name in ["p", "prices", "price_df", "data", "df_prices", "panel", "raw"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and any(c.endswith("_c1_usd") for c in obj.columns):
            print(f"Using price DataFrame: '{name}' {obj.shape}")
            return obj
    raise NameError("No price DataFrame with '*_c1_usd' columns found. "
                    "Re-run your data-loading cell to rebuild it.")
price_df = _find_price_df()

FONT_DIR="/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn,url in {"Lato-Regular.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf",
               "Lato-Bold.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Bold.ttf"}.items():
    fp=os.path.join(FONT_DIR,fn)
    if not os.path.exists(fp):
        try: urllib.request.urlretrieve(url,fp)
        except Exception: pass
    try: fm.fontManager.addfont(fp)
    except Exception: pass
try: LATO=fm.FontProperties(fname=f"{FONT_DIR}/Lato-Regular.ttf").get_name()
except Exception: LATO="DejaVu Sans"
plt.rcParams.update({"font.family":LATO,"font.size":11,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.edgecolor":"#555555","axes.linewidth":0.8})

HUB_STYLE = {
    "ttf": ("#500778","TTF (Europe)"),
    "jkm": ("#E8890C","JKM (Asia)"),
    "nbp": ("#1B9E77","NBP (UK)"),
    "the": ("#C0392B","THE (Germany)"),
    "hh":  ("#2C7FB8","Henry Hub (US)"),
}
EVENTS = {
    "ukraine": ("2022-02-24","2023-10-31","#C62828","Ukraine"),
    "redsea":  ("2023-11-19","2025-05-06","#F9A825","Red Sea"),
    "iran":    ("2026-02-28",None,          "#2C7FB8","Hormuz"),
}

START = pd.Timestamp("2022-01-01")
df = price_df.loc[START:].copy()

fig, ax = plt.subplots(figsize=(12, 5.2))
ymax = max(df[f"{h}_c1_usd"].max() for h in HUB_STYLE if f"{h}_c1_usd" in df.columns)

for k,(s,e,col,lab) in EVENTS.items():
    s = max(pd.Timestamp(s), START)
    e = pd.Timestamp(e) if e else df.index.max()
    ax.axvspan(s, e, color=col, alpha=0.09, zorder=0)
    mid = s + (e - s) / 2
    ax.text(mid, ymax*0.965, lab, rotation=0, va="top", ha="center",
            fontsize=13, color=col, fontweight="bold", alpha=0.95)

for h in ["hh","nbp","the","jkm","ttf"]:
    col=f"{h}_c1_usd"
    if col in df.columns:
        c,lab=HUB_STYLE[h]
        ax.plot(df.index, df[col], color=c, lw=1.7 if h=="ttf" else 1.2,
                alpha=1.0 if h=="ttf" else 0.9, label=lab, zorder=3)

ttf=df["ttf_c1_usd"]; pk_d,pk_v=ttf.idxmax(),ttf.max()
ax.annotate(f"TTF peak  ${pk_v:.0f}/MMBtu\n({pk_d.strftime('%b %Y')})",
            xy=(pk_d,pk_v), xytext=(pk_d+pd.Timedelta(days=120), pk_v*0.9),
            fontsize=9.5, color="#500778", fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="#500778", lw=1.3,
                            connectionstyle="arc3,rad=-0.15"))

ax.set_ylabel("Front-month price (USD/MMBtu)")
ax.set_ylim(0, ymax*1.06); ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%d"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

# --- title ---
ax.set_title("Front-Month Hub Prices Through the Three Disruptions, 2022 to 2026",
             fontsize=13, fontweight="bold", color="#500778", pad=12)

# --- legend at the BOTTOM, with thicker lines ---
leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=5,
                frameon=False, fontsize=9.5, handlelength=2.2, columnspacing=1.5)
for line in leg.get_lines():
    line.set_linewidth(3.5)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4_1_prices_events.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/fig4_1_prices_events.pdf", bbox_inches="tight")
plt.show()
print("Saved fig4_1_prices_events.png")

In [ ]:
# %% FIGURE 4.2 — Weekly data coverage grid (self-contained)
import os, urllib.request
import matplotlib.pyplot as plt, matplotlib.dates as mdates, matplotlib.font_manager as fm
import numpy as np, pandas as pd
from matplotlib.colors import ListedColormap

# --- locate the price DataFrame (p/panel may be clobbered) ---
def _find_price_df():
    for name in ["panel", "p", "prices", "price_df", "data", "df_prices", "raw"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and any(c.endswith("_c1_usd") for c in obj.columns):
            print(f"Using price DataFrame: '{name}' {obj.shape}")
            return obj
    raise NameError("No price DataFrame with '*_c1_usd' columns found — re-run your data-loading cell.")
panel = _find_price_df()

# --- font ---
FONT_DIR="/content/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
for fn,url in {"Lato-Regular.ttf":"https://github.com/google/fonts/raw/main/ofl/lato/Lato-Regular.ttf"}.items():
    fp=os.path.join(FONT_DIR,fn)
    if not os.path.exists(fp):
        try: urllib.request.urlretrieve(url,fp)
        except Exception: pass
    try: fm.fontManager.addfont(fp)
    except Exception: pass
try: LATO=fm.FontProperties(fname=f"{FONT_DIR}/Lato-Regular.ttf").get_name()
except Exception: LATO="DejaVu Sans"
plt.rcParams.update({"font.family":LATO,"font.size":11})
PURPLE="#500778"

# --- coal (rebuild only if missing) ---
try:
    coal
except NameError:
    raw = pd.read_excel("/content/drive/MyDrive/Dissertation Analysis/Data/coal price.xlsx", header=None)
    hdr = next(i for i in range(len(raw)) if str(raw.iloc[i,0]).strip()=="Exchange Date")
    coal = raw.iloc[hdr+1:, [0,1]].copy(); coal.columns=["date","coal_usd_t"]
    coal["date"]=pd.to_datetime(coal["date"],errors="coerce")
    coal["coal_usd_t"]=pd.to_numeric(coal["coal_usd_t"],errors="coerce")
    coal=coal.dropna().set_index("date").sort_index()

# --- build SERIES ---
SERIES = {
    "TTF price": panel.get("ttf_c1_usd"), "THE price": panel.get("the_c1_usd"),
    "NBP price": panel.get("nbp_c1_usd"), "JKM price": panel.get("jkm_c1_usd"),
    "Henry Hub price": panel.get("hh_c1_usd"), "Brent crude": panel.get("brent_c1"),
    "Coal (API2)": coal["coal_usd_t"], "Carbon (EUA)": panel.get("eua_c1_lag1"),
    "GPR index": panel.get("gpr"), "EU storage": panel.get("storage_pct"),
    "Hormuz transits": panel.get("chk_hormuz_ma7"), "Suez transits": panel.get("chk_suez_ma7"),
    "Bab el-Mandeb": panel.get("chk_bab_ma7"), "Cape of G.H.": panel.get("chk_cape_ma7"),
}
SERIES = {k:v for k,v in SERIES.items() if v is not None and v.dropna().shape[0] > 0}

# --- weekly presence matrix ---
starts=[v.dropna().index.min() for v in SERIES.values()]
ends  =[v.dropna().index.max() for v in SERIES.values()]
labels=list(SERIES.keys())
order = np.argsort([SERIES[l].dropna().index.min() for l in labels])
labels=[labels[i] for i in order]
cutoff=pd.Timestamp("2016-01-01")
calW = pd.date_range(min(starts), max(ends), freq="W")
MW = np.zeros((len(labels), len(calW)))
for i,lab in enumerate(labels):
    weekly = SERIES[lab].dropna().resample("W").count().reindex(calW).fillna(0)
    MW[i,:] = (weekly>0).astype(float)

fig, ax = plt.subplots(figsize=(12,6))
ax.imshow(MW, aspect="auto", cmap=ListedColormap(["#f7f7f7", PURPLE]),
          interpolation="nearest",
          extent=[mdates.date2num(calW[0]), mdates.date2num(calW[-1]), len(labels)-0.5, -0.5])
ax.axvline(mdates.date2num(cutoff), color="#C0392B", ls="--", lw=1.8, zorder=5)
ax.text(mdates.date2num(cutoff), len(labels)-1.2, " Modelling window starts (2016)",
        color="#C0392B", fontsize=12.5, fontweight="bold", va="top", ha="left")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=10)
ax.xaxis_date(); ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y")); ax.tick_params(axis="x",labelsize=9)
for sp in ["top","right","left"]: ax.spines[sp].set_visible(False)
ax.set_title("Data coverage by series (weekly; filled = data present)",
             fontsize=16, fontweight="bold", color=PURPLE, pad=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4_2_coverage.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/fig4_2_coverage.pdf", bbox_inches="tight")
plt.show()
print("Saved fig4_2_coverage.png")

In [ ]:
# %% FIGURE 4.3 — Chokepoint tanker transits (7-day MA) with event windows
import matplotlib.pyplot as plt, matplotlib.dates as mdates, pandas as pd, numpy as np

def _find_frame_with(prefix):
    for name in ["panel", "p", "data", "prices", "raw"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and any(c.startswith(prefix) for c in obj.columns):
            return obj
    raise NameError(f"No frame with '{prefix}' columns found.")
df = _find_frame_with("chk_")

CHK_STYLE = {
    "chk_hormuz_ma7": ("#500778","Hormuz"),
    "chk_suez_ma7":   ("#E8890C","Suez"),
    "chk_bab_ma7":    ("#C0392B","Bab el-Mandeb"),
    "chk_cape_ma7":   ("#1B9E77","Cape of Good Hope"),
}
EVENTS = {
    "ukraine": ("2022-02-24","2023-10-31","#C62828","Ukraine"),
    "redsea":  ("2023-11-19","2025-05-06","#F9A825","Red Sea"),
    "iran":    ("2026-02-28",None,          "#2C7FB8","Hormuz"),
}

# ============================================================
# ANNOTATION CONFIG — nudge these to reposition the arrows/labels
#   series      : which line the arrow tip sits ON
#   tip_date    : x-position of the drop you're pointing AT
#   text        : the small label
#   text_date   : x-position of the label text (move left/right)
#   text_yfrac  : label height as a FRACTION of ymax (move up/down, 0-1)
#   tip_yoffset : nudges the arrow tip up(+)/down(-) in transit units
#   rad         : arrow curve (+ curves one way, - the other)
# ============================================================
ANNOTATIONS = [
    dict(series="chk_hormuz_ma7", tip_date="2026-02-28",
         text="Hormuz closure", text_date="2025-09-01", text_yfrac=0.9,
         tip_yoffset=0.3, rad=-0.2),
    dict(series="chk_bab_ma7",    tip_date="2023-12-20",
         text="Red Sea rerouting", text_date="2023-04-01", text_yfrac=0.45,
         tip_yoffset=.2, rad=0.2),
]
ARROW_FONTSIZE = 8
ARROW_COLOR    = "#333333"
# ============================================================

present = [c for c in CHK_STYLE if c in df.columns]
print("Plotting:", present)

sub = df[present].dropna(how="all")
print(f"Rows with data: {len(sub)}, span {sub.index.min()} -> {sub.index.max()}")
ymax = float(np.nanmax([sub[c].max() for c in present]))
print(f"ymax = {ymax}")

fig, ax = plt.subplots(figsize=(12,5.2))
for k,(s,e,col,lab) in EVENTS.items():
    s=max(pd.Timestamp(s), sub.index.min()); e=pd.Timestamp(e) if e else sub.index.max()
    ax.axvspan(s,e,color=col,alpha=0.09,zorder=0)
    mid=s+(e-s)/2
    ax.text(mid, ymax*0.99, lab, va="top", ha="center", fontsize=12,
            color=col, fontweight="bold", alpha=0.95)
for c in present:
    col,lab=CHK_STYLE[c]
    ax.plot(sub.index, sub[c], color=col, lw=1.4, label=lab, alpha=0.9, zorder=3)

# --- annotation arrows -------------------------------------------------
def _value_at(series, date):
    """Nearest available value of `series` to `date` (for the arrow tip)."""
    ts = pd.Timestamp(date)
    s = sub[series].dropna()
    if len(s) == 0:
        return np.nan
    idx = s.index.get_indexer([ts], method="nearest")[0]
    return float(s.iloc[idx])

for a in ANNOTATIONS:
    if a["series"] not in sub.columns:
        print(f"skip annotation: {a['series']} not present")
        continue
    tip_x = pd.Timestamp(a["tip_date"])
    tip_y = _value_at(a["series"], a["tip_date"]) + a.get("tip_yoffset", 0)
    txt_x = pd.Timestamp(a["text_date"])
    txt_y = ymax * a["text_yfrac"]
    ax.annotate(
        a["text"],
        xy=(tip_x, tip_y),
        xytext=(txt_x, txt_y),
        fontsize=ARROW_FONTSIZE, color=ARROW_COLOR, fontweight="bold",
        ha="center", va="center", zorder=6,
        arrowprops=dict(arrowstyle="-|>", color=ARROW_COLOR, lw=1.0,
                        connectionstyle=f"arc3,rad={a.get('rad', -0.2)}",
                        shrinkA=2, shrinkB=3),
    )
# ----------------------------------------------------------------------
ax.set_title("Chokepoint Tanker Transits During the Three Disruptions",
             fontsize=13, fontweight="bold", color="#500778", pad=12)
ax.set_ylabel("Tanker transits (7-day moving average)")
ax.set_ylim(0, ymax*1.06); ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# --- legend at the BOTTOM ---------------------------------------------
leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4,
                frameon=False, fontsize=9.5, handlelength=2.6, columnspacing=1.8)
for line in leg.get_lines(): line.set_linewidth(3.5)

plt.savefig(f"{OUT_DIR}/fig4_3_chokepoints.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig4_3_chokepoints.png")

In [ ]:
# %% FIGURE 4.4 — EU gas storage (% of capacity) with event windows
import matplotlib.pyplot as plt, matplotlib.dates as mdates, pandas as pd, numpy as np
PURPLE="#500778"

def _find_price_df():
    for name in ["p", "panel", "prices", "data", "price_df", "raw"]:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and any(c.endswith("_c1_usd") for c in obj.columns):
            return obj
    raise NameError("No price DataFrame found.")
df = _find_price_df().loc["2019-01-01":]

storage_col = next((c for c in ["storage_pct","storage_level","storage"] if c in df.columns), None)
if storage_col is None:
    print("Storage columns:", [c for c in df.columns if "storage" in c.lower()])
    raise KeyError("No storage % column found.")
storage = df[storage_col].dropna()
print(f"storage_col='{storage_col}', rows={len(storage)}, "
      f"span {storage.index.min()} -> {storage.index.max()}, mean={storage.mean():.1f}")

EVENTS = {
    "ukraine": ("2022-02-24","2023-10-31","#C62828","Ukraine"),
    "redsea":  ("2023-11-19","2025-05-06","#F9A825","Red Sea"),
    "iran":    ("2026-02-28",None,          "#2C7FB8","Hormuz"),
}

fig, ax = plt.subplots(figsize=(12,5.0))

for k,(s,e,col,lab) in EVENTS.items():
    s=max(pd.Timestamp(s), storage.index.min()); e=pd.Timestamp(e) if e else storage.index.max()
    ax.axvspan(s,e,color=col,alpha=0.09,zorder=0)
    mid=s+(e-s)/2
    ax.text(mid, 99, lab, va="top", ha="center", fontsize=12,
            color=col, fontweight="bold", alpha=0.95)

ax.plot(storage.index, storage, color=PURPLE, lw=1.7, zorder=3)
ax.fill_between(storage.index, storage, color=PURPLE, alpha=0.10, zorder=2)
mean_s = storage.mean()
ax.axhline(mean_s, color="#888888", ls=":", lw=2.5, zorder=1)
ax.text(storage.index.min(), mean_s, f" mean {mean_s:.0f}%", va="bottom", ha="left",
        fontsize=15, color="black", fontweight="bold")

ax.set_title("EU gas storage (% of capacity)", fontsize=13, fontweight="bold",
             color=PURPLE, pad=10)
ax.set_ylabel("EU gas storage (% of capacity)")
ax.set_ylim(0, 100); ax.margins(x=0.01)
ax.grid(axis="y", color="#ececec", lw=0.7); ax.set_axisbelow(True)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.savefig(f"{OUT_DIR}/fig4_4_storage.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved fig4_4_storage.png")

In [ ]:
# %% Print the Diebold-Yilmaz connectedness table for review
import pandas as pd

# try the saved CSV first
try:
    dy = pd.read_csv(f"{OUT_DIR}/sq2_connectedness.csv", index_col=0)
    print("Loaded from sq2_connectedness.csv\n")
except Exception as e:
    print("CSV not found, checking memory for a connectedness object...")
    # common variable names it might be stored under
    for name in ["connectedness","conn","dy_table","spillover","dy","C"]:
        if name in dir():
            dy = eval(name); print(f"Found in memory as `{name}`\n"); break
    else:
        raise FileNotFoundError("No connectedness table found — tell me the variable name")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.1f}")

print("=== FULL CONNECTEDNESS TABLE ===")
print(dy.to_string())
print("\nShape:", dy.shape)
print("Columns:", list(dy.columns))
print("Index:", list(dy.index))

# if it's the standard D-Y matrix (senders in columns, receivers in rows),
# compute the directional TO / FROM / NET summary
print("\n=== attempting TO / FROM / NET summary ===")
try:
    mat = dy.copy()
    # drop any existing summary rows/cols so we work on the raw NxN block
    for junk in ["FROM","From","from","Directional FROM","Contribution to others","TO","NET"]:
        if junk in mat.columns: mat = mat.drop(columns=junk)
        if junk in mat.index:   mat = mat.drop(index=junk)
    hubs = [h for h in mat.index if h in mat.columns]
    mat = mat.loc[hubs, hubs].astype(float)

    to_others   = mat.sum(axis=0) - pd.Series({h: mat.loc[h,h] for h in hubs})   # column sum minus own
    from_others = mat.sum(axis=1) - pd.Series({h: mat.loc[h,h] for h in hubs})   # row sum minus own
    net = to_others - from_others
    summary = pd.DataFrame({"TO others": to_others, "FROM others": from_others, "NET": net}).round(1)
    print(summary.to_string())
except Exception as e:
    print("Could not auto-compute NET (paste the raw table and I'll do it):", e)

In [ ]:
print("Xall going into SHAP:", Xall.shape)

In [ ]:
# %% Dump final selected hyperparameters for the spec table
print("=== Classical model selected hyperparameters (leakage-safe tuning) ===")
for k, v in best.items():
    print(f"  {k:14s} = {v}")

print("\n=== Neural selected hyperparameters ===")
print(f"  LSTM hidden   = {best_lstm_h if 'best_lstm_h' in dir() else 'run Task E1'}")
print(f"  TCN channels  = {best_tcn_c if 'best_tcn_c' in dir() else 'run Task E1'}")
print(f"  lookback      = {LOOKBACK}")
print(f"  DNN widths    = (128, 64, 32, 16, 8)")

In [ ]:
# %% MLP (leakage-safe, lbfgs solver) — final MLP for Table 6.6
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np, pandas as pd, warnings
from sklearn.exceptions import ConvergenceWarning

def mlp_walk_forward(width=32, alpha=1e-2, solver="lbfgs", max_iter=5000):
    """Walk-forward MLP with per-fold train-only scaling.
    lbfgs is sklearn's recommended solver for small datasets and converges far
    more reliably than Adam here."""
    preds, actual, n_nonconv = [], [], 0
    for tr, te in tscv.split(Xall):
        Xtr, Xte = Xall.iloc[tr], Xall.iloc[te]
        ytr, yte = yall[TARGET].iloc[tr], yall[TARGET].iloc[te]
        sc = StandardScaler().fit(Xtr)                      # train-only
        Xtr_s = sc.transform(Xtr); Xte_s = sc.transform(Xte)
        m = MLPRegressor(hidden_layer_sizes=(width,),
                         alpha=alpha,
                         solver=solver,
                         max_iter=max_iter,
                         random_state=0)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            try:
                m.fit(Xtr_s, ytr)
            except ConvergenceWarning:
                n_nonconv += 1
                warnings.simplefilter("ignore", ConvergenceWarning)
                m.fit(Xtr_s, ytr)
        preds.append(pd.Series(m.predict(Xte_s), index=yte.index))
        actual.append(yte)
    p, a = pd.concat(preds), pd.concat(actual)
    rmse = float(np.sqrt(np.mean((a.values - p.values) ** 2)))
    mae  = float(np.mean(np.abs(a.values - p.values)))
    idx  = a.index.intersection(base["actual"].index)
    stat, pv = dm_test((a.loc[idx] - p.loc[idx]).values,
                       (base["actual"].loc[idx] - base["preds"].loc[idx]).values)
    return rmse, mae, stat, pv, n_nonconv, p, a

rmse, mae, stat, pv, n_nonconv, mlp_p, mlp_a = mlp_walk_forward(width=32)
verdict = "beats RW" if (pv < 0.05 and stat < 0) else ("WORSE than RW" if (pv < 0.05 and stat > 0) else "ties RW")
print(f"MLP (width 32, lbfgs solver):")
print(f"  RMSE = {rmse:.5f} | MAE = {mae:.5f}")
print(f"  DM vs RW: stat={stat:+.3f}, p={pv:.4f} -> {verdict}")
print(f"  RW RMSE = {base['rmse']:.5f}")
print(f"  folds converged: {10 - n_nonconv}/10")

res["MLP"] = {"rmse": rmse, "preds": mlp_p, "actual": mlp_a}

In [ ]:
# %% Threshold confidence interval — moving-block bootstrap on tau = -lambda0/delta
import numpy as np, pandas as pd
from statsmodels.api import OLS, add_constant

# rebuild the SAME design used in Task A.4
_bt = arb[["dJKM","dTTF","dBrent","ecm_lag"]].copy()
_bt["abs_spread"]   = (arb["JKM"] - arb["TTF"]).abs().shift(1)
_bt["ecm_x_spread"] = _bt["ecm_lag"] * _bt["abs_spread"]
_bt = _bt.dropna().reset_index(drop=True)

XCOLS = ["dTTF","dBrent","ecm_lag","ecm_x_spread"]

def _tau(df):
    m = OLS(df["dJKM"], add_constant(df[XCOLS])).fit()   # OLS is fine for point ratio
    return -m.params["ecm_lag"] / m.params["ecm_x_spread"]

point = _tau(_bt)
print(f"Point threshold tau = ${point:.4f}/MMBtu")

# moving-block bootstrap (block ~ 1 trading month, preserves autocorrelation)
rng = np.random.default_rng(0)
n, block, n_boot = len(_bt), 20, 2000
boot = []
for _ in range(n_boot):
    idx = []
    while len(idx) < n:
        s = rng.integers(0, n - block)
        idx.extend(range(s, s + block))
    idx = idx[:n]
    try:
        t = _tau(_bt.iloc[idx])
        if np.isfinite(t) and 0 < t < 20:      # drop degenerate resamples
            boot.append(t)
    except Exception:
        pass

boot = np.array(boot)
lo, hi = np.percentile(boot, [2.5, 97.5])
med = np.median(boot)
print(f"Bootstrap: median ${med:.2f}, 95% CI [${lo:.2f}, ${hi:.2f}]  (n_boot={len(boot)})")
print(f"\nReport: threshold ${point:.2f}/MMBtu (95% bootstrap CI [${lo:.2f}, ${hi:.2f}])")

In [ ]:
# %% Threshold CI via delta method (analytical, uses coefficient covariance)
import numpy as np
# refit to get the covariance matrix
_m = OLS(arb_t["dJKM"], add_constant(arb_t[["dTTF","dBrent","ecm_lag","ecm_x_spread"]])).fit(
    cov_type="HAC", cov_kwds={"maxlags":7})
l0, d = _m.params["ecm_lag"], _m.params["ecm_x_spread"]
tau = -l0/d
# gradient of tau = -l0/d  w.r.t. (l0, d): [d(tau)/d(l0), d(tau)/d(d)] = [-1/d, l0/d^2]
V = _m.cov_params().loc[["ecm_lag","ecm_x_spread"], ["ecm_lag","ecm_x_spread"]].values
g = np.array([-1/d, l0/d**2])
se_tau = np.sqrt(g @ V @ g)
lo, hi = tau - 1.96*se_tau, tau + 1.96*se_tau
print(f"tau = ${tau:.3f}, SE = {se_tau:.3f}")
print(f"Delta-method 95% CI: [${lo:.2f}, ${hi:.2f}]")

In [ ]:
# %% Regenerate bootstrap CIs on RMSE/MAE (current data, all 9 models)
import numpy as np, pandas as pd

def _get(k):
    for d in ([results, res] if 'results' in dir() else [res]):
        if k in d and "preds" in d[k]:
            return d[k]["preds"], d[k]["actual"]
    return None, None

def fold_bootstrap_ci(pred, act, n_boot=2000, seed=0):
    e = (act.values - pred.values)
    ae = np.abs(e); se = e**2
    rng = np.random.default_rng(seed); n = len(e)
    rmse_bs, mae_bs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        rmse_bs.append(np.sqrt(se[idx].mean()))
        mae_bs.append(ae[idx].mean())
    return (np.sqrt(se.mean()), *np.percentile(rmse_bs,[2.5,97.5]),
            ae.mean(), *np.percentile(mae_bs,[2.5,97.5]))

order = ["LinearSVR","ElasticNet","XGBoost","DNN","RandomForest",
         "RandomWalk","MLP","TCN","LSTM"]
rows = []
for k in order:
    p, a = _get(k)
    if p is None:
        print(f"  ⚠ {k} not found in res/results"); continue
    rmse, rlo, rhi, mae, mlo, mhi = fold_bootstrap_ci(p, a)
    rows.append({"Model":k, "RMSE":round(rmse,5), "RMSE_CI_low":round(rlo,5),
                 "RMSE_CI_high":round(rhi,5), "MAE":round(mae,5),
                 "MAE_CI_low":round(mlo,5), "MAE_CI_high":round(mhi,5)})

ci = pd.DataFrame(rows)
ci.to_csv(f"{OUT_DIR}/sq2_metric_cis.csv", index=False)
print(ci.to_string(index=False))

In [ ]:
# %% Regenerate bootstrap CIs — force numpy arrays, positional indexing
import numpy as np, pandas as pd

def _get(k):
    for d in ([results, res] if 'results' in dir() else [res]):
        if k in d and "preds" in d[k]:
            return d[k]["preds"], d[k]["actual"]
    return None, None

def fold_bootstrap_ci(pred, act, n_boot=2000, seed=0):
    idx = pred.index.intersection(act.index)
    e = np.asarray(act.loc[idx]) - np.asarray(pred.loc[idx])   # force numpy, aligned
    e = e[~np.isnan(e)]                                        # drop any NaN
    se = e**2; ae = np.abs(e)
    n = len(e)
    rng = np.random.default_rng(seed)
    rmse_bs = np.empty(n_boot); mae_bs = np.empty(n_boot)
    for b in range(n_boot):
        j = rng.integers(0, n, n)          # positional indices into numpy array
        rmse_bs[b] = np.sqrt(se[j].mean())
        mae_bs[b]  = ae[j].mean()
    rmse = np.sqrt(se.mean()); mae = ae.mean()
    return (rmse, np.percentile(rmse_bs,2.5), np.percentile(rmse_bs,97.5),
            mae,  np.percentile(mae_bs,2.5),  np.percentile(mae_bs,97.5))

order = ["LinearSVR","ElasticNet","XGBoost","DNN","RandomForest",
         "RandomWalk","MLP","TCN","LSTM"]
rows = []
for k in order:
    p, a = _get(k)
    if p is None:
        print(f"  ⚠ {k} not found"); continue
    rmse, rlo, rhi, mae, mlo, mhi = fold_bootstrap_ci(p, a)
    rows.append({"Model":k, "RMSE":round(rmse,5), "RMSE_CI_low":round(rlo,5),
                 "RMSE_CI_high":round(rhi,5), "MAE":round(mae,5),
                 "MAE_CI_low":round(mlo,5), "MAE_CI_high":round(mhi,5)})

ci = pd.DataFrame(rows)
ci.to_csv(f"{OUT_DIR}/sq2_metric_cis.csv", index=False)
print(ci.to_string(index=False))

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:.5f}")
print(ci.to_string(index=False))

In [ ]:
# %% FIGURE — Improvement over random walk with 95% bootstrap CI
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ci = pd.read_csv(f"{OUT_DIR}/sq2_metric_cis.csv")
rw = ci[ci.Model=="RandomWalk"]["RMSE"].values[0]

d = ci[ci.Model!="RandomWalk"].copy()
d["impr"]    = 100*(rw - d["RMSE"])/rw
d["impr_lo"] = 100*(rw - d["RMSE_CI_high"])/rw   # high RMSE -> low improvement
d["impr_hi"] = 100*(rw - d["RMSE_CI_low"])/rw
d = d.sort_values("impr").reset_index(drop=True)

PURPLE="#500778"; AMBER="#B8860B"; RED="#C0392B"
def col(r):
    if r["impr_lo"] > 0: return PURPLE   # CI entirely above 0 -> beats
    if r["impr_hi"] < 0: return RED      # CI entirely below 0 -> worse
    return AMBER                         # CI crosses 0 -> ties

fig, ax = plt.subplots(figsize=(9,5.5))
for yi, r in d.iterrows():
    c = col(r)
    ax.plot([r["impr_lo"], r["impr_hi"]], [yi, yi], color=c, lw=2.5, alpha=0.55, zorder=2)
    for xe in [r["impr_lo"], r["impr_hi"]]:
        ax.plot([xe,xe],[yi-0.12,yi+0.12], color=c, lw=2, zorder=2)
    ax.scatter(r["impr"], yi, s=120, color=c, zorder=3, edgecolor="white", linewidth=1.2)

ax.axvline(0, color="#333", ls="--", lw=1.2, zorder=1)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d["Model"], fontsize=11)
ax.set_xlabel("Improvement over random walk (%), with 95% bootstrap CI")
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", lw=0.7); ax.set_axisbelow(True); ax.tick_params(left=False)
ax.text(ax.get_xlim()[0], len(d)+0.55, "Forecast improvement with uncertainty",
        fontsize=14, fontweight="bold", color="#1a1a1a")
ax.text(ax.get_xlim()[0], len(d)+0.15,
        "Purple = CI beats RW · amber = CI crosses zero (ties) · red = worse",
        fontsize=9.5, color="#6b6b6b")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_uncertainty_improvement.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% FIGURE — RMSE with 95% CI, random-walk band shaded
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ci = pd.read_csv(f"{OUT_DIR}/sq2_metric_cis.csv")
rwrow = ci[ci.Model=="RandomWalk"].iloc[0]
rw, rw_lo, rw_hi = rwrow["RMSE"], rwrow["RMSE_CI_low"], rwrow["RMSE_CI_high"]

d = ci.sort_values("RMSE").reset_index(drop=True)
PURPLE="#500778"; AMBER="#B8860B"; RED="#C0392B"; GREY="#6b6b6b"
def col(r):
    if r["Model"]=="RandomWalk": return GREY
    if r["RMSE_CI_high"] < rw_lo: return PURPLE   # CI clears RW band low -> beats
    if r["RMSE_CI_low"]  > rw_hi: return RED      # CI clears RW band high -> worse
    return AMBER                                   # overlaps RW band -> ties

fig, ax = plt.subplots(figsize=(9,5.5))
ax.axvspan(rw_lo, rw_hi, color=GREY, alpha=0.12, zorder=0)   # RW CI band
ax.axvline(rw, color=GREY, ls="--", lw=1, alpha=0.7, zorder=1)
for yi, r in d.iterrows():
    c = col(r)
    ax.plot([r["RMSE_CI_low"], r["RMSE_CI_high"]], [yi, yi], color=c, lw=2.5, alpha=0.55, zorder=2)
    for xe in [r["RMSE_CI_low"], r["RMSE_CI_high"]]:
        ax.plot([xe,xe],[yi-0.12,yi+0.12], color=c, lw=2, zorder=2)
    ax.scatter(r["RMSE"], yi, s=120, color=c, zorder=3, edgecolor="white", linewidth=1.2)

ax.set_yticks(range(len(d))); ax.set_yticklabels(d["Model"], fontsize=11)
ax.set_xlabel("Walk-forward RMSE with 95% bootstrap CI (n = 600)")
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", lw=0.7); ax.set_axisbelow(True); ax.tick_params(left=False)
ax.text(ax.get_xlim()[0], len(d)+0.55, "Forecast accuracy with uncertainty",
        fontsize=14, fontweight="bold", color="#1a1a1a")
ax.text(ax.get_xlim()[0], len(d)+0.15,
        "Shaded band = random-walk 95% CI · purple beats · amber ties · red worse",
        fontsize=9.5, color=GREY)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sq2_uncertainty_ci.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# %% FIGURE 5.4 — Expanding-window walk-forward validation (10 folds, abbreviated)
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# --- palette (UCL) ---
PURPLE = "#500778"    # training
LILAC  = "#B9A2CC"    # validation
TEAL   = "#1B9E88"    # test
GREY   = "#E6E6E6"    # unused / future

# --- scheme geometry ---
n_rows = 5                              # drawn rows: folds 1,2,3, ellipsis, fold 10
start_year, end_year = 2016, 2026
total = end_year - start_year           # 10 units on the time axis

val_w  = 0.8                            # validation width (years)
test_w = 1.2                            # test width (years)
first_train_end = 3.0                   # fold 1 trains on ~3 years
# step spacing for the first three drawn folds
step = 1.3

fold_labels = ["Fold 1", "Fold 2", "Fold 3", "", "Fold 10"]

row_h = 0.62
gap   = 0.40

fig, ax = plt.subplots(figsize=(11, 4.8))

for i in range(n_rows):
    y = (n_rows - 1 - i) * (row_h + gap)     # top row = fold 1

    # --- ellipsis row ---
    if i == 3:
        ax.text(total/2, y, "\u22ee", ha="center", va="center",
                fontsize=20, color="#999999", fontweight="bold")
        continue

    # --- geometry per row ---
    if i == 4:                                # "Fold 10" — the final fold
        train_end = total - val_w - test_w    # trains on almost the whole span
        val_end   = train_end + val_w
        test_end  = total
    else:                                     # folds 1-3
        train_end = first_train_end + i * step
        val_end   = train_end + val_w
        test_end  = val_end + test_w

    # training (expands each fold)
    ax.barh(y, train_end, left=0, height=row_h,
            color=PURPLE, edgecolor="white", linewidth=1.2)
    # validation
    ax.barh(y, val_w, left=train_end, height=row_h,
            color=LILAC, edgecolor="white", linewidth=1.2)
    # test
    ax.barh(y, test_w, left=val_end, height=row_h,
            color=TEAL, edgecolor="white", linewidth=1.2)
    # unused / future
    if test_end < total:
        ax.barh(y, total - test_end, left=test_end, height=row_h,
                color=GREY, edgecolor="white", linewidth=1.2)

    # fold label on the left
    ax.text(-0.15, y, fold_labels[i], ha="right", va="center",
            fontsize=10, fontweight="bold", color="#333333")

# --- segment labels on the top fold only ---
y_top = (n_rows - 1) * (row_h + gap)
te1 = first_train_end
ax.text(te1/2, y_top, "Training", ha="center", va="center",
        color="white", fontsize=10, fontweight="bold")
ax.text(te1 + val_w/2, y_top, "Validate", ha="center", va="center",
        color="#333333", fontsize=8.5, fontweight="bold")
ax.text(te1 + val_w + test_w/2, y_top, "Test", ha="center", va="center",
        color="white", fontsize=9, fontweight="bold")

# --- ordering arrow under the bottom fold ---
y_bot = 0
ax.annotate("", xy=(6.4, y_bot - 0.75), xytext=(0.2, y_bot - 0.75),
            arrowprops=dict(arrowstyle="->", color="#666666", lw=1.1))
ax.text(3.3, y_bot - 1.12, "train  \u2192  validate  \u2192  test  (time order preserved)",
        ha="center", va="center", fontsize=9, color="#666666", style="italic")

# --- time axis ---
ax.set_xlim(-0.05, total)
ax.set_ylim(y_bot - 1.6, y_top + row_h)
ax.set_xticks(range(0, total + 1, 2))
ax.set_xticklabels([str(start_year + t) for t in range(0, total + 1, 2)])
ax.set_xlabel("Time", fontsize=10)
ax.set_yticks([])
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color("#999999")

# --- legend ---
handles = [
    Patch(facecolor=PURPLE, label="Training (expands each fold)"),
    Patch(facecolor=LILAC,  label="Validation (tuning)"),
    Patch(facecolor=TEAL,   label="Test (out-of-sample)"),
    Patch(facecolor=GREY,   label="Not yet used"),
]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.17),
          ncol=4, frameon=False, fontsize=9)

ax.set_title("Expanding-window walk-forward validation",
             fontsize=13, fontweight="bold", color=PURPLE, pad=12)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/walkforward_scheme.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/walkforward_scheme.pdf", bbox_inches="tight")
plt.show()
print("Saved walkforward_scheme.png")